# Exact verification for double Ore extensions of type (14641)

**Companion to the manuscript by James Gómez and Helbert Venegas.**

Source of the 26 presentations: J. J. Zhang and J. Zhang,
*Double extension regular algebras of type (14641)*, J. Algebra 322 (2009),
373--409, Section 4. [Article](https://doi.org/10.1016/j.jalgebra.2009.03.041) ·
[arXiv](https://arxiv.org/abs/0712.2550).

This notebook is self-contained. Upload it to Google Colab and run all cells;
the package files and the source PDF are not needed at runtime. CPU is sufficient.
All mathematical coefficients are exact. The library cells below reproduce the
readable Python sources supplied in the repository and can be collapsed.

Version **0.3.0** adds an independent checker, generic induction and parity
certificates, claim-level hypotheses, and execution in a real Jupyter kernel.
It includes all 21 closed degree models, their torus/action
matrices, normalizers, rational central witnesses, localization identities,
module bases and general recurrence certificates. A coverage ledger links the
checks to all 108 labels in the supplied manuscript. Finite regressions are
identified separately. The exact degree of C and the remaining primitivity
questions remain open, exactly as in the manuscript.

The mathematical division-ring and ideal-theory arguments remain explicit
dependencies, not proof-assistant formalizations. The PI table helper applies
the stated criterion; the degree calculator instead derives a value from the
torus and its center actions and compares it with the closed formula.


In [1]:
import importlib.util
import importlib.metadata
import subprocess
import sys
if importlib.util.find_spec('sympy') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'sympy==1.14.0'])
import sympy as sp
from IPython.display import display, Math, Markdown
print('Python:', sys.version.split()[0], '| SymPy:', sp.__version__)
if sp.__version__ != '1.14.0':
    print('Reference version: SymPy 1.14.0. Install that version for the recorded environment.')


Python: 3.12.14 | SymPy: 1.14.0


## 1. Defining matrices, parameter restrictions, and conventions

We retain the full twist $h$. Family N has no independent $h$.

$$
\begin{pmatrix}y_1x_1\\y_1x_2\\y_2x_1\\y_2x_2\end{pmatrix}
=\Sigma\begin{pmatrix}x_1y_1\\x_2y_1\\x_1y_2\\x_2y_2\end{pmatrix}.
$$

M is obtained by interchanging positions 2 and 3 on both sides. Its entries
were independently transcribed for cross-checking. The missing overall $h$ in
the printed M for J is explicitly restored.

If $\sigma_{ij}(X)=A_{ij}X$, the matrix of $\det\sigma$ is
$$-p_{11}A_{11}A_{12}+A_{11}A_{22}-p_{12}A_{21}A_{12}.$$
Composition reverses the displayed coefficient matrix order. This object is
different from the ordinary scalar determinant of the $4\times4$ matrix $\Sigma$.


In [2]:
# Defining data
"""Zhang--Zhang Section 4, with the twist parameter retained.

Source: J. Algebra 322 (2009), 373--409, printed pp. 395, 397--400.
The published Sigma matrices are the defining data. M is also transcribed
independently, rather than generated from Sigma, to check the transcription.
"""
from dataclasses import dataclass
from functools import lru_cache
import sympy as sp

p, q, f, g, h, s, t = sp.symbols('p q f g h s t')
I = sp.I
E2 = sp.eye(2)


@dataclass(frozen=True)
class Family:
    name: str
    P: tuple
    Q: tuple
    sigma: sp.ImmutableMatrix
    printed_m: sp.ImmutableMatrix
    expected_detsigma: sp.ImmutableMatrix
    relations: tuple
    nonzero: tuple
    section: str
    page: int
    pi_condition: str

    @lru_cache(maxsize=65536)
    def scalar(self, expression):
        """Exact rational arithmetic modulo the family's parameter equation."""
        expression = sp.sympify(expression)
        if expression.has(sp.Float):
            raise TypeError('Use exact Rational or algebraic coefficients, not floats')
        expression = sp.cancel(expression)
        for polynomial, variable in self.relations:
            numerator, denominator = sp.fraction(expression)
            numerator = sp.rem(numerator, polynomial, variable)
            denominator = sp.rem(denominator, polynomial, variable)
            if denominator == 0:
                raise ZeroDivisionError('Denominator vanishes in parameter field')
            if variable in denominator.free_symbols:
                numerator *= sp.invert(denominator, polynomial, variable)
                numerator = sp.rem(numerator, polynomial, variable)
                denominator = sp.Integer(1)
            expression = sp.cancel(numerator / denominator)
        return sp.expand(expression)

    def matrix(self, matrix):
        return sp.ImmutableMatrix(matrix).applyfunc(self.scalar)

    def specialize(self, substitutions):
        """Reject provably inadmissible exact specializations; retain symbolic ones."""
        substitutions = {sp.sympify(k): sp.sympify(v) for k, v in substitutions.items()}
        if any(value.has(sp.Float) for value in substitutions.values()):
            raise TypeError('Use exact Rational or algebraic parameters, not floats')
        for equation, variable in self.relations:
            value = sp.simplify(equation.subs(substitutions, simultaneous=True))
            if variable in substitutions and value != 0 and value.equals(0) is not True:
                raise ValueError(f'{self.name}: parameter equation {equation}=0 fails')
        for restriction in self.nonzero:
            if sp.simplify(restriction.subs(substitutions, simultaneous=True)) == 0:
                raise ValueError(f'{self.name}: requires {restriction} != 0')
        rel = tuple((sp.expand(eq.subs(substitutions, simultaneous=True)), var)
                    for eq, var in self.relations
                    if var not in substitutions)
        sub = lambda a: sp.sympify(a).subs(substitutions, simultaneous=True)
        mat = lambda a: sp.ImmutableMatrix(a.subs(substitutions, simultaneous=True))
        return Family(self.name, tuple(map(sub, self.P)), tuple(map(sub, self.Q)),
                      mat(self.sigma), mat(self.printed_m), mat(self.expected_detsigma),
                      rel, tuple(map(sub, self.nonzero)), self.section, self.page,
                      self.pi_condition)

    def reshuffled_m(self):
        order = [0, 2, 1, 3]
        return self.sigma.extract(order, order)

    def detsigma(self):
        """Matrix acting on (x1,x2)^T; composition reverses matrix order.

        If sigma_ij(X)=A_ij X, then (sigma_kl o sigma_ij)(X)
        has matrix A_ij A_kl, since each entry is a scalar.
        This is NOT the ordinary determinant of the 4 by 4 Sigma matrix.
        """
        a11 = self.sigma[:2, :2]
        a12 = self.sigma[:2, 2:]
        a21 = self.sigma[2:, :2]
        a22 = self.sigma[2:, 2:]
        p12, p11 = self.P
        return self.matrix(-p11*a11*a12 + a11*a22 - p12*a21*a12)


def _family(name, P, Q, sigma, m, det, section, page, pi, nonzero=(), relation=None):
    twist = 1 if name == 'N' else h
    restrictions = (() if name == 'N' else (h,)) + tuple(nonzero)
    return Family(name, tuple(map(sp.sympify, P)), tuple(map(sp.sympify, Q)),
                  sp.ImmutableMatrix(sigma)*twist, sp.ImmutableMatrix(m)*twist,
                  sp.ImmutableMatrix(det), (() if relation is None else (relation,)),
                  restrictions, section, page, pi)


def registry():
    """All 26 families, in the original coordinates (in particular for G)."""
    result = {}
    def add(*args, **kw):
        fam = _family(*args, **kw)
        result[fam.name] = fam
    add('A', (1,1), (1,0),
        [[1,0,0,0],[0,1,1,0],[0,0,1,0],[0,-2,-1,1]],
        [[1,0,0,0],[0,1,0,0],[0,1,1,0],[0,-1,-2,1]],
        h**2*E2, '4.1.4',395,'never')
    add('B', (p,0), (p,0),
        [[0,0,0,1],[0,0,1,0],[0,-1,0,0],[1,0,0,0]],
        [[0,0,0,1],[0,0,-1,0],[0,1,0,0],[1,0,0,0]],
        p*h**2*sp.diag(1,-1), '4.2.2',397,'h is a root of unity',
        relation=(p**2+1,p))
    add('C', (p,0), (p,0),
        [[-1,p**2,1,-p],[-p,1,1,-p],[-p,-2*p**2,p,-p],[-p,p**2,1,-1]],
        [[-1,1,p**2,-p],[-p,p,-2*p**2,-p],[-p,1,1,-p],[-p,1,p**2,-1]],
        -3*h**2*sp.diag(p,1),'4.2.2',397,'3*h^2 is a root of unity',
        relation=(p**2+p+1,p))
    add('D', (p,0), (-1,0),
        [[-p,0,0,0],[0,-p**2,1,0],[0,0,p,0],[1,0,0,1]],
        [[-p,0,0,0],[0,p,0,0],[0,1,-p**2,0],[1,0,0,1]],
        -p**2*h**2*E2,'4.2.3',397,'p,h roots of unity and p != +/-1',nonzero=(p,))
    add('E', (p,0), (-1,0),
        [[0,0,1,1],[0,0,1,-1],[-1,1,0,0],[1,1,0,0]],
        [[0,1,0,1],[-1,0,1,0],[0,1,0,-1],[1,0,1,0]],
        2*p*h**2*sp.Matrix([[0,1],[-1,0]]),'4.2.3',397,'2*h^2 is a root of unity',
        relation=(p**2+1,p))
    add('F', (p,0), (-1,0),
        [[-1,-p,1,-1],[-p,1,1,1],[-p,p,p,1],[-p,-p,1,-p]],
        [[-1,1,-p,-1],[-p,p,p,1],[-p,1,1,1],[-p,1,-p,-p]],
        -2*p*h**2*E2,'4.2.3',397,'2*h^2 is a root of unity',relation=(p**2+1,p))
    add('G', (p,0), (1,0),
        [[p,0,0,0],[p,p**2,1,0],[0,0,p,0],[f,0,-1,1]],
        [[p,0,0,0],[0,p,0,0],[p,1,p**2,0],[f,-1,0,1]],
        p**2*h**2*E2,'4.2.4',398,'p,h roots of unity',nonzero=(p,p-1,p+1,f))
    add('H', (-1,0), (1,1),
        [[0,0,1,0],[0,0,f,1],[1,0,0,0],[f,1,0,0]],
        [[0,1,0,0],[1,0,0,0],[0,f,0,1],[f,0,1,0]],
        h**2*sp.Matrix([[1,0],[2*f,1]]),'4.3.1',398,'never')
    add('I', (-1,0), (q,0),
        [[-q,-q,1,-q],[1,1,1,-q],[1,q,q,-q],[-1,-q,1,-1]],
        [[-q,1,-q,-q],[1,q,q,-q],[1,1,1,-q],[-1,1,-q,-1]],
        2*h**2*sp.diag(1,-1),'4.3.2',398,'2*h^2 is a root of unity',
        relation=(q**2+1,q))
    # The printed M for J omits its overall h; tensor reshuffling restores it.
    add('J', (-1,0), (q,0),
        [[0,1,0,1],[-1,0,1,0],[0,1,0,-1],[1,0,1,0]],
        [[0,0,1,1],[0,0,1,-1],[-1,1,0,0],[1,1,0,0]],
        2*h**2*E2,'4.3.2',398,'2*h^2 is a root of unity',relation=(q**2+1,q))
    add('K', (-1,0), (q,0),
        [[1,0,0,0],[0,0,0,1],[0,0,1,0],[0,f,0,0]],
        [[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,f,0]],
        h**2*sp.diag(1,f),'4.3.2',398,'q,f,h roots of unity',nonzero=(q,f))
    add('L', (-1,0), (q,0),
        [[0,0,f,0],[0,0,0,1],[f,0,0,0],[0,1,0,0]],
        [[0,f,0,0],[f,0,0,0],[0,0,0,1],[0,0,1,0]],
        h**2*sp.diag(f**2,1),'4.3.2',398,'q,f,h roots of unity',nonzero=(q,f))
    add('M', (-1,0), (-1,0),
        [[0,1,1,0],[f,0,0,-1],[1,0,0,-1],[0,-1,-f,0]],
        [[0,1,1,0],[1,0,0,-1],[f,0,0,-1],[0,-f,-1,0]],
        (1-f)*h**2*E2,'4.3.3',398,'f != 0 and h*(s+/-1) roots of unity',nonzero=(f-1,))
    add('N', (-1,0), (-1,0),
        [[0,-g,0,f],[g,0,f,0],[0,f,0,-g],[f,0,g,0]],
        [[0,0,-g,f],[0,0,f,-g],[g,f,0,0],[f,g,0,0]],
        (f**2-g**2)*E2,'4.3.3',398,'f+g and f-g roots of unity',nonzero=(f**2-g**2,))
    add('O', (-1,0), (-1,0),
        [[1,0,0,f],[0,-1,1,0],[0,f,-1,0],[1,0,0,1]],
        [[1,0,0,f],[0,-1,f,0],[0,1,-1,0],[1,0,0,1]],
        (f-1)*h**2*E2,'4.3.3',398,'f != 0 and h*(1+/-s) roots of unity',nonzero=(f-1,))
    add('P', (-1,0), (-1,0),
        [[0,0,1,f],[0,0,1,1],[1,-f,0,0],[-1,1,0,0]],
        [[0,1,0,f],[1,0,-f,0],[0,1,0,1],[-1,0,1,0]],
        (1-f)*h**2*E2,'4.3.3',399,'f != 0 and h*(1+/-s) roots of unity',nonzero=(f-1,))
    add('Q', (-1,0), (-1,0),
        [[0,0,1,0],[1,1,1,0],[-1,0,0,0],[1,0,-1,1]],
        [[0,1,0,0],[-1,0,0,0],[1,1,1,0],[1,-1,0,1]],
        h**2*sp.diag(-1,1),'4.3.3',399,'h is a root of unity')
    add('R', (-1,0), (-1,0),
        [[1,1,1,0],[0,0,1,0],[0,1,0,0],[0,-1,-1,1]],
        [[1,1,1,0],[0,0,1,0],[0,1,0,0],[0,-1,-1,1]],
        h**2*sp.Matrix([[0,1],[-1,0]]),'4.3.3',399,'h is a root of unity')
    add('S', (-1,0), (-1,0),
        [[-1,1,1,1],[1,-1,1,1],[1,1,-1,1],[1,1,1,-1]],
        [[-1,1,1,1],[1,-1,1,1],[1,1,-1,1],[1,1,1,-1]],
        4*h**2*E2,'4.3.3',399,'2*h is a root of unity')
    add('T', (-1,0), (-1,0),
        [[-1,1,1,1],[1,-1,1,1],[1,1,1,-1],[1,1,-1,1]],
        [[-1,1,1,1],[1,1,1,-1],[1,1,-1,1],[1,-1,1,1]],
        4*h**2*sp.Matrix([[0,1],[1,0]]),'4.3.3',399,'2*h is a root of unity')
    add('U', (-1,0), (-1,0),
        [[-1,1,1,1],[1,1,1,-1],[1,1,-1,1],[1,-1,1,1]],
        [[-1,1,1,1],[1,-1,1,1],[1,1,1,-1],[1,1,-1,1]],
        4*h**2*E2,'4.3.3',399,'2*h is a root of unity')
    add('V', (-1,0), (1,0),
        [[0,1,1,0],[0,1,0,0],[-1,1,0,0],[0,0,0,1]],
        [[0,1,1,0],[-1,0,1,0],[0,0,1,0],[0,0,0,1]],
        h**2*sp.Matrix([[-1,1],[0,1]]),'4.3.4',399,'h is a root of unity')
    add('W', (-1,0), (1,0),
        [[0,f,1,0],[1,0,0,-1],[1,0,0,f],[0,-1,1,0]],
        [[0,1,f,0],[1,0,0,f],[1,0,0,-1],[0,1,-1,0]],
        (f+1)*h**2*E2,'4.3.4',399,'f != 0 and h*(s+/-i) roots of unity',nonzero=(f+1,))
    add('X', (-1,0), (1,0),
        [[0,0,1,0],[0,0,1,1],[1,0,0,0],[1,1,0,0]],
        [[0,1,0,0],[1,0,0,0],[0,1,0,1],[1,0,1,0]],
        h**2*sp.Matrix([[1,0],[2,1]]),'4.3.4',399,'never')
    add('Y', (-1,0), (1,0),
        [[1,0,0,0],[f,-1,1,0],[0,0,1,0],[1,0,f,-1]],
        [[1,0,0,0],[0,1,0,0],[f,1,-1,0],[1,f,0,-1]],
        h**2*E2,'4.3.4',399,'never')
    add('Z', (1,0), (-1,0),
        [[1,0,0,1],[0,1,1,0],[0,f,-1,0],[f,0,0,-1]],
        [[1,0,0,1],[0,-1,f,0],[0,1,1,0],[f,0,0,-1]],
        -(f+1)*h**2*E2,'4.4.4',400,'h*(s+/-i) roots of unity',nonzero=(f,f+1))
    return result


FAMILIES = registry()


In [3]:
# Noncommutative PBW reduction
"""Small exact noncommutative rewriting engine, using tuples of letters.

Letter order: x1 < x2 < y1 < y2. All six rules decrease degree-lex order.
The four cubic overlaps are checked explicitly before using confluence.
No floating point arithmetic, commuting symbol substitution, or degree cutoff.
"""
from functools import lru_cache
from itertools import combinations
import sympy as sp

LETTERS = ('x1', 'x2', 'y1', 'y2')


class Algebra:
    def __init__(self, family):
        self.family = family
        self.scalar = family.scalar
        self.letters = LETTERS
        q12, q11 = family.Q
        p12, p11 = family.P
        self.rules = {
            (1, 0): self.clean({(0, 1): q12, (0, 0): q11}),
            (3, 2): self.clean({(2, 3): p12, (2, 2): p11}),
        }
        lhs = [(2,0),(2,1),(3,0),(3,1)]
        rhs = [(0,2),(1,2),(0,3),(1,3)]
        for row, word in enumerate(lhs):
            self.rules[word] = self.clean({v: family.sigma[row,col]
                                          for col,v in enumerate(rhs)})
        self._initialize()

    def _initialize(self):
        for word, terms in self.rules.items():
            if any((len(v),v) >= (len(word),word) for v in terms):
                raise ValueError('The rewrite rules must decrease degree-lex order')
        # A per-instance cache avoids retaining every past Algebra instance.
        self.reduce_word = lru_cache(maxsize=32768)(self._reduce_word)
        self.generators = tuple(Element(self, {(i,): sp.Integer(1)}) for i in range(len(self.letters)))

    @classmethod
    def from_rules(cls,rules,scalar,letters):
        algebra=object.__new__(cls)
        algebra.scalar=scalar
        algebra.family=None
        algebra.letters=tuple(letters)
        algebra.rules={w:algebra.clean(v) for w,v in rules.items()}
        algebra._initialize()
        return algebra

    def clean(self, terms):
        return {w:c for w,value in terms.items() if (c:=self.scalar(value)) != 0}

    def _reduce_word(self, word):
        for pos in range(len(word)-1):
            pair = word[pos:pos+2]
            if pair in self.rules:
                return tuple(self.first_reduction(word, pos).items())
        return ((word, sp.Integer(1)),)

    def first_reduction(self, word, position):
        """Force one chosen initial step, then reduce deterministically."""
        result = {}
        for replacement, coefficient in self.rules[word[position:position+2]].items():
            new = word[:position]+replacement+word[position+2:]
            for reduced, factor in self.reduce_word(new):
                result[reduced] = result.get(reduced,0)+coefficient*factor
        return self.clean(result)

    def overlaps(self):
        """All critical overlaps; the two starting reductions are independent."""
        residuals = {}
        for ascending in combinations(range(len(self.letters)),3):
            word = tuple(reversed(ascending))
            if word[:2] not in self.rules or word[1:] not in self.rules:
                continue
            left = Element(self,self.first_reduction(word,0))
            right = Element(self,self.first_reduction(word,1))
            residuals[' '.join(self.letters[i] for i in word)] = left-right
        return residuals

    def element(self, value):
        if isinstance(value, Element):
            if value.algebra is not self:
                raise ValueError('Elements belong to different algebras')
            return value
        return Element(self, {(): self.scalar(sp.sympify(value))})


class Element:
    def __init__(self, algebra, terms):
        self.algebra = algebra
        self.terms = algebra.clean(terms)

    def __add__(self, other):
        other = self.algebra.element(other)
        terms = self.terms.copy()
        for word,coefficient in other.terms.items():
            terms[word] = terms.get(word,0)+coefficient
        return Element(self.algebra,terms)

    __radd__ = __add__

    def __neg__(self):
        return Element(self.algebra,{w:-c for w,c in self.terms.items()})

    def __sub__(self,other):
        return self+-self.algebra.element(other)

    def __rsub__(self,other):
        return self.algebra.element(other)+-self

    def __mul__(self,other):
        other = self.algebra.element(other)
        result = {}
        for w,c in self.terms.items():
            for v,d in other.terms.items():
                for word,factor in self.algebra.reduce_word(w+v):
                    result[word] = result.get(word,0)+c*d*factor
        return Element(self.algebra,result)

    def __rmul__(self,other):
        return self.algebra.element(other)*self

    def __pow__(self, exponent):
        if not isinstance(exponent,int) or exponent < 0:
            raise ValueError('Only nonnegative integer powers belong to this engine')
        result = self.algebra.element(1)
        factor = self
        while exponent:
            if exponent % 2:
                result = result*factor
            exponent //= 2
            if exponent:
                factor = factor*factor
        return result

    def commutator(self,other):
        return self*other-other*self

    def __bool__(self):
        return bool(self.terms)

    def certificate(self):
        return {' '.join(self.algebra.letters[i] for i in w) or '1': str(c)
                for w,c in sorted(self.terms.items())}

    def __repr__(self):
        if not self.terms:
            return '0'
        return ' + '.join(f'({c})*'+('*'.join(self.algebra.letters[i] for i in w) or '1')
                          for w,c in sorted(self.terms.items()))


In [4]:
# Fourteen monomial basis changes
"""The 14 basis changes and lambda scalars in the manuscript's Table tab:mono."""
from dataclasses import dataclass, replace
import sympy as sp


@dataclass(frozen=True)
class MonomialDatum:
    name: str
    x_basis: sp.ImmutableMatrix
    y_basis: sp.ImmutableMatrix
    kind: str
    lambdas: tuple
    square_root: bool = False

    @property
    def family(self):
        if self.square_root:
            family = FAMILIES[self.name].specialize({f:s**2})
            return replace(family,nonzero=family.nonzero+(s,))
        return FAMILIES[self.name]

    def transformed_sigma(self):
        change = sp.kronecker_product(self.y_basis,self.x_basis)
        return self.family.matrix(change*self.family.sigma*change.inv())

    def target_sigma(self):
        matrix = sp.zeros(4)
        switch_x = self.kind in ('I','III')
        switch_y = self.kind in ('I','II')
        for a in range(2):
            for b in range(2):
                aa,bb = (1-a if switch_y else a),(1-b if switch_x else b)
                matrix[2*a+b,2*aa+bb] = self.lambdas[2*a+b]
        return sp.ImmutableMatrix(matrix)

    def invariants(self):
        a,b,c,d = self.lambdas
        values = {'0':(a,b,c,d),'I':(a*d,b*c),
                  'II':(a*c,b*d,a/b,c/d),'III':(a*b,c*d,a/c,b/d)}[self.kind]
        return tuple(self.family.scalar(x) for x in values)


def monomial_registry():
    eye = sp.eye(2)
    plus = sp.Matrix([[1,1],[1,-1]])
    complex_plus = sp.Matrix([[1,sp.I],[1,-sp.I]])
    result = {}
    def add(name,X,Y,kind,coefficients,square_root=False):
        twist = 1 if name == 'N' else h
        result[name] = MonomialDatum(name,sp.ImmutableMatrix(X),sp.ImmutableMatrix(Y),
                                    kind,tuple(twist*c for c in coefficients),square_root)
    add('B',eye,eye,'I',(1,1,-1,1))
    add('E',complex_plus,eye,'I',(1+sp.I,1-sp.I,-1+sp.I,-1-sp.I))
    add('J',eye,complex_plus,'I',(1+sp.I,-1+sp.I,1-sp.I,-1-sp.I))
    add('K',eye,[[1,1/s],[1,-1/s]],'0',(1,s,1,-s),True)
    add('L',eye,eye,'II',(f,1,f,1))
    add('M',[[1,sp.I/s],[1,-sp.I/s]],complex_plus,'I',
        (sp.I*(s+1),-sp.I*(s-1),sp.I*(s-1),-sp.I*(s+1)),True)
    add('N',eye,plus,'III',(f-g,f+g,-f-g,g-f))
    add('O',[[1,sp.I*s],[1,-sp.I*s]],complex_plus,'I',(1-s,1+s,1+s,1-s),True)
    add('P',[[1,s],[1,-s]],eye,'II',(1+s,1-s,1-s,1+s),True)
    add('S',plus,plus,'0',(2,-2,-2,-2))
    add('T',plus,complex_plus,'II',(2*sp.I,-2,-2*sp.I,-2))
    add('U',complex_plus,plus,'III',(2*sp.I,-2*sp.I,-2,-2))
    add('W',[[1,sp.I*s],[1,-sp.I*s]],plus,'III',
        (sp.I*(s-sp.I),-sp.I*(s+sp.I),sp.I*(s+sp.I),-sp.I*(s-sp.I)),True)
    add('Z',plus,[[1,sp.I/s],[1,-sp.I/s]],'II',
        (sp.I*(s-sp.I),-sp.I*(s+sp.I),-sp.I*(s+sp.I),sp.I*(s-sp.I)),True)
    return result


MONOMIAL = monomial_registry()


In [5]:
# Smith normal form and Dickson samples
"""Exact Smith normal form certificates for quantum tori at roots of unity.

Convention: v_i v_j = zeta**A[i,j] v_j v_i, with zeta primitive of order N.
This computes the PI degree of THAT TORUS, not automatically its crossed product.
"""
from itertools import combinations, product
from math import gcd, isqrt, lcm, prod
from functools import reduce
import sympy as sp
from sympy.matrices.normalforms import smith_normal_form


def alternating(upper, size=4):
    upper = tuple(upper)
    if len(upper) != size*(size-1)//2:
        raise ValueError('Incorrect number of upper triangular entries')
    matrix = sp.zeros(size)
    for (i,j),value in zip(combinations(range(size),2),upper):
        matrix[i,j],matrix[j,i] = value,-value
    return sp.ImmutableMatrix(matrix)


def torus_certificate(matrix, order):
    if not isinstance(order,int) or order < 1:
        raise ValueError('The root order must be a positive integer')
    matrix = sp.Matrix(matrix)
    if matrix.rows != matrix.cols or matrix+matrix.T != sp.zeros(matrix.rows):
        raise ValueError('The commutation matrix must be alternating')
    if not all(v.is_Integer for v in matrix):
        raise ValueError('Use integer exponents, never approximate logarithms')
    smith = smith_normal_form(matrix,domain=sp.ZZ)
    diagonal = [abs(int(smith[i,i])) for i in range(matrix.rows)]
    image_size = prod(order//gcd(order,d) for d in diagonal)
    degree = isqrt(image_size)
    if degree*degree != image_size:
        raise ArithmeticError('An alternating pairing must have square image size')
    return {'root_order':order,'commutation_matrix':[list(map(int,matrix.row(i)))
                                                  for i in range(matrix.rows)],
            'smith_diagonal':diagonal,'image_size':image_size,
            'radical_size_mod_N':order**matrix.rows//image_size,'pi_degree':degree}


def torus_from_rational_phases(upper):
    values = tuple(sp.sympify(a) for a in upper)
    if any(a.has(sp.Float) for a in values):
        raise TypeError('Use exact rational phases, not decimal approximations')
    phases = tuple(sp.Rational(a) for a in values)
    order = lcm(*(int(a.q) for a in phases))
    return torus_certificate(alternating([int(a*order) for a in phases]),order)


def brute_radical_size(matrix, order):
    """Independent finite enumeration for small regression examples only."""
    matrix = sp.Matrix(matrix)
    rows = [tuple(map(int,matrix.row(i))) for i in range(matrix.rows)]
    return sum(all(sum(a*b for a,b in zip(row,v))%order == 0 for row in rows)
               for v in product(range(order),repeat=matrix.cols))


def dg_matrix(name, order, p_exponent, h_exponent):
    k,kp = p_exponent,h_exponent
    if name not in ('D','G'):
        raise ValueError('Only D and G share this torus model')
    if order < 3 or order//gcd(order,k) <= 2:
        raise ValueError('This birational model requires p != +/-1')
    if name == 'D':
        if order%2 or gcd(order,order//2,k,kp) != 1:
            raise ValueError('For D, zeta must generate <-1,p,h>')
        return alternating((order//2,-order//2-k-kp,-k-kp,-k,k,-k))
    if gcd(order,k,kp) != 1:
        raise ValueError('For G, zeta must generate <p,h>')
    return alternating((0,-k-kp,-k-kp,-k,k,-k))


def dg_closed_formula(order,p_exponent,h_exponent):
    return order**2//gcd(order,2*p_exponent*(p_exponent+h_exponent))


def l_matrix(order,q_exponent,f_exponent,h_exponent):
    if order < 2 or order%2 or gcd(order,order//2,q_exponent,f_exponent,h_exponent) != 1:
        raise ValueError('For L, zeta must generate <-1,q,f,h> and N is even')
    th,ph,eta = q_exponent,f_exponent,h_exponent
    # w*y1=(h^2*f)^(-1)*y1*w; the manuscript's first upper entry has a sign typo.
    return alternating((-2*eta-ph,0,2*th,order//2,2*eta+2*ph,0))


def l_closed_formula(order,q_exponent,f_exponent,h_exponent):
    # N0 = order(<-1,q,f>); the crossed product contributes a factor of 2.
    n0 = order//gcd(order,order//2,q_exponent,f_exponent)
    return 2*order if (order//n0)%2 else order


def dickson_matrix_residual(order, family_name):
    """Check D_n(M(t),alpha)=D_n(t,alpha)I in Q(p,f,t)/(Phi_n(p)).

    Each call is a symbolic certificate for primitive roots of this specified
    order. A finite list of calls does not prove the lemma for every order.
    """
    from dataclasses import replace
    if order < 3 or family_name not in ('D','G'):
        raise ValueError('Use n >= 3 and family D or G')
    fam = replace(FAMILIES[family_name],relations=((sp.cyclotomic_poly(order,p),p),))
    c12,c21 = (-1/p,1/p) if family_name == 'D' else (1/p,f/p)
    matrix = fam.matrix([[p*t,c12],[c21,t/p]])
    alpha = fam.scalar(-c12*c21*p**2/(p**2-1)**2)
    prev2,prev1 = 2*sp.eye(2),matrix
    scalar2,scalar1 = sp.Integer(2),t
    for _ in range(2,order+1):
        nxt = fam.matrix(matrix*prev1-alpha*prev2)
        scalar_next = fam.scalar(t*scalar1-alpha*scalar2)
        prev2,prev1 = prev1,nxt
        scalar2,scalar1 = scalar1,scalar_next
    return fam.matrix(prev1-scalar1*sp.eye(2))


In [6]:
# Exact instantiation of the PI table
"""Exact instantiation of the manuscript's PI criterion (not a new proof)."""
import sympy as sp


def exact_root_order(value):
    """Return n for a root of unity, 0 for certified non-torsion, None if unknown.

    For an exact algebraic number, torsion is detected by its minimal polynomial
    being cyclotomic. No numerical tolerance or arbitrary maximum order is used.
    If phi(n)=d, then n <= 2*d*d, so the subsequent order recovery is exhaustive.
    """
    value = sp.sympify(value)
    if value.has(sp.Float):
        raise TypeError('Floating point cannot certify root-of-unity status')
    if value.free_symbols:
        return None
    if value == 0:
        return 0
    if value.is_algebraic is False:
        return 0
    z = sp.Symbol('z')
    try:
        poly = sp.Poly(sp.minpoly(value,z),z,domain=sp.QQ).monic()
    except (sp.polys.polyerrors.NotAlgebraic,NotImplementedError,ValueError):
        return None
    if not poly.is_cyclotomic:
        return 0
    degree = poly.degree()
    for n in range(1,2*degree*degree+1):
        if sp.totient(n) == degree and poly == sp.Poly(sp.cyclotomic_poly(n,z),z,domain=sp.QQ):
            return n
    raise ArithmeticError('Cyclotomic polynomial has no order in the proved bound')


def pi_decision(name, substitutions):
    family = FAMILIES[name].specialize(substitutions)
    substitutions = {sp.sympify(k):sp.sympify(v) for k,v in substitutions.items()}
    value = lambda x: sp.simplify(sp.sympify(x).subs(substitutions,simultaneous=True))
    result = {'family':name,'method':'Instantiation of manuscript Table tab:main',
              'status':None,'required_scalars':[]}
    if name in ('A','H','X','Y'):
        result['status'] = False
        return result
    if name == 'D' and value(p) in (1,-1):
        result['status'] = False
        return result
    if name in ('M','O','P','W') and value(f) == 0:
        result['status'] = False
        return result
    if name in ('B','Q','R','V'):
        scalars = [h]
    elif name == 'C':
        scalars = [3*h**2]
    elif name in ('E','F','I','J'):
        scalars = [2*h**2]
    elif name in ('D','G'):
        scalars = [p,h]
    elif name in ('K','L'):
        scalars = [q,f,h]
    elif name == 'N':
        scalars = [f+g,f-g]
    elif name in ('S','T','U'):
        scalars = [2*h]
    else:
        root = sp.sqrt(value(f))
        if name == 'M':
            scalars = [value(h)*(root+1),value(h)*(root-1)]
        elif name in ('O','P'):
            scalars = [value(h)*(1+root),value(h)*(1-root)]
        else:
            scalars = [value(h)*(root+sp.I),value(h)*(root-sp.I)]
    orders = []
    for scalar in scalars:
        scalar = value(scalar)
        order = exact_root_order(scalar)
        orders.append(order)
        result['required_scalars'].append({'scalar':str(scalar),'root_order':order})
    result['status'] = (False if 0 in orders else None if None in orders else True)
    return result


In [7]:
# Exact Laurent words and source-derived action matrices
"""Exact Laurent word calculus for the plane bases used in the manuscript.

Each plane is either a coordinate quantum plane or a balanced plane
(v1**2=v2**2). A word is normalized first by the independently checked mixed
relations and then by the plane relations. Inverses are handled algebraically.
The normal form is r**a e**b rho**c z**d, with all exponents integers.
"""
from dataclasses import dataclass
from functools import lru_cache
import sympy as sp

alpha, beta, H, S, F = sp.symbols('alpha beta H S F')


class WordModel:
    def __init__(self, scalar, x_kind, y_kind, x_q, y_q, mixing, name=''):
        self.scalar = scalar
        self.x_kind,self.y_kind = x_kind,y_kind
        self.x_q,self.y_q = sp.sympify(x_q),sp.sympify(y_q)
        # (input y,input x) -> (output x,output y,coefficient)
        self.mixing = mixing
        self.name = name
        self.normalize = lru_cache(maxsize=32768)(self._normalize)
        self.e1,self.e2,self.z1,self.z2 = [Word(self,((plane,index,1),))
                                          for plane,index in [(0,0),(0,1),(1,0),(1,1)]]

    def crossing(self,a,b,sy,sx):
        if sy==1 and sx==1:
            j,i,c=self.mixing[a,b]
            return j,i,c
        matches=[]
        for (i,j),(jj,ii,c) in self.mixing.items():
            if sy==1 and sx==-1 and j==b and ii==a:
                matches.append((jj,i,1/c))
            elif sy==-1 and sx==1 and i==a and jj==b:
                matches.append((j,ii,1/c))
            elif sy==-1 and sx==-1 and jj==b and ii==a:
                matches.append((j,i,c))
        if len(matches)!=1:
            raise ValueError('The mixed inverse relation is not uniquely determined')
        return matches[0]

    def plane_normal(self, letters,kind,parameter):
        # r^a e^b with r=e1^-1 e2. Append individual signed generators.
        a,b,c=0,0,sp.Integer(1)
        for _,index,sign in letters:
            if index==0:
                aa,bb,cc=0,sign,sp.Integer(1)
            elif kind=='balanced':
                aa,bb,cc=-1,sign,sp.Integer(1)
            elif sign==1:
                aa,bb,cc=1,1,1/parameter
            else:
                aa,bb,cc=-1,-1,sp.Integer(1)
            if kind=='balanced':
                a,b,c=a+(-1 if b%2 else 1)*aa,b+bb,c*cc
            else:
                c*=cc*parameter**(-b*aa)
                a,b=a+aa,b+bb
        return int(a),int(b),self.scalar(c)

    def _normalize(self, letters):
        letters=list(letters)
        coefficient=sp.Integer(1)
        while True:
            for j in range(len(letters)-1):
                if letters[j][0]==1 and letters[j+1][0]==0:
                    _,a,sy=letters[j];_,b,sx=letters[j+1]
                    bb,aa,c=self.crossing(a,b,sy,sx)
                    letters[j:j+2]=[(0,bb,sx),(1,aa,sy)]
                    coefficient*=c
                    break
            else:
                break
        xs=[a for a in letters if a[0]==0]
        ys=[a for a in letters if a[0]==1]
        a,b,cx=self.plane_normal(xs,self.x_kind,self.x_q)
        c,d,cy=self.plane_normal(ys,self.y_kind,self.y_q)
        return (a,b,c,d),self.scalar(coefficient*cx*cy)

    def identity(self):
        return Word(self,())

    def canonical_word(self,exponents):
        a,b,c,d=map(int,exponents)
        r=self.e1**-1*self.e2
        rho=self.z1**-1*self.z2
        return r**a*self.e1**b*rho**c*self.z1**d


@dataclass(frozen=True)
class Word:
    model: WordModel
    letters: tuple
    coefficient: object = sp.Integer(1)

    def __mul__(self,other):
        if not isinstance(other,Word):
            return Word(self.model,self.letters,self.coefficient*sp.sympify(other))
        if other.model is not self.model:
            raise ValueError('Words belong to different models')
        return Word(self.model,self.letters+other.letters,self.coefficient*other.coefficient)

    __rmul__=__mul__

    def __pow__(self,n):
        if not isinstance(n,int):
            raise TypeError('Use integer Laurent exponents')
        if n<0:
            inverse=Word(self.model,tuple((a,b,-c) for a,b,c in reversed(self.letters)),
                         1/self.coefficient)
            return inverse**(-n)
        if n==0:
            return self.model.identity()
        return Word(self.model,self.letters*n,self.coefficient**n)

    def normal(self):
        exponents,coefficient=self.model.normalize(self.letters)
        return exponents,self.model.scalar(coefficient*self.coefficient)

    def equals(self,other):
        a,c=self.normal();b,d=other.normal()
        return a==b and self.model.scalar(c-d)==0

    def conjugate(self,other):
        return self*other*self**-1

    def commutation(self,other):
        a,c=(self*other).normal();b,d=(other*self).normal()
        if a!=b:
            raise ValueError('These monomials do not commute up to a scalar')
        return self.model.scalar(c/d)

    def certificate(self):
        a,c=self.normal()
        return {'exponents_r_e_rho_z':list(a),'coefficient':str(c)}


def mixing_from_datum(datum,lambdas=None):
    lambdas=datum.lambdas if lambdas is None else lambdas
    sx=datum.kind in ('I','III');sy=datum.kind in ('I','II')
    return {(a,b):(1-b if sx else b,1-a if sy else a,lambdas[2*a+b])
            for a in range(2) for b in range(2)}


def monomial_model(name,formal=False):
    datum=MONOMIAL[name]
    family=datum.family
    coordinate_x={'B':p,'J':q,'K':q,'L':q,'N':-1,'W':1}
    coordinate_y={'B':p,'E':p,'L':-1,'P':-1,'Z':1}
    substitutions={}
    lambdas=datum.lambdas
    if formal:
        if name=='B':
            lambdas=(H,H,-H,H);substitutions={H:h}
        elif name=='E':
            lambdas=(alpha,-sp.I*alpha,sp.I*alpha,-alpha)
            substitutions={alpha:h*(1+sp.I)}
        elif name=='J':
            lambdas=(alpha,sp.I*alpha,-sp.I*alpha,-alpha)
            substitutions={alpha:h*(1+sp.I)}
        elif name=='K':
            lambdas=(H,H*S,H,-H*S);substitutions={H:h,S:s}
        elif name=='L':
            lambdas=(H*F,H,H*F,H);substitutions={H:h,F:f}
        elif name=='M':
            lambdas=(sp.I*alpha,-sp.I*beta,sp.I*beta,-sp.I*alpha)
            substitutions={alpha:h*(s+1),beta:h*(s-1)}
        elif name=='N':
            lambdas=(alpha,beta,-beta,-alpha)
            substitutions={alpha:f-g,beta:f+g}
        elif name=='O':
            lambdas=(beta,alpha,alpha,beta)
            substitutions={alpha:h*(1+s),beta:h*(1-s)}
        elif name=='P':
            lambdas=(alpha,beta,beta,alpha)
            substitutions={alpha:h*(1+s),beta:h*(1-s)}
        elif name=='S':
            lambdas=(H,-H,-H,-H);substitutions={H:2*h}
        elif name=='T':
            lambdas=(sp.I*H,-H,-sp.I*H,-H);substitutions={H:2*h}
        elif name=='U':
            lambdas=(sp.I*H,-sp.I*H,-H,-H);substitutions={H:2*h}
        elif name=='W':
            lambdas=(sp.I*alpha,-sp.I*beta,sp.I*beta,-sp.I*alpha)
            substitutions={alpha:h*(s-sp.I),beta:h*(s+sp.I)}
        elif name=='Z':
            lambdas=(sp.I*alpha,-sp.I*beta,-sp.I*beta,sp.I*alpha)
            substitutions={alpha:h*(s-sp.I),beta:h*(s+sp.I)}
    model=WordModel(family.scalar,'coordinate' if name in coordinate_x else 'balanced',
                    'coordinate' if name in coordinate_y else 'balanced',
                    coordinate_x.get(name,1),coordinate_y.get(name,1),
                    mixing_from_datum(datum,lambdas),name)
    model.substitutions=substitutions
    model.family=family
    return model


def normalizer_model(name):
    """Models justified by the separately checked Theta/Rnorm/Fnorm/Inorm identities."""
    scalar=FAMILIES[name].scalar
    if name in ('Q','V'):
        lambdas=(-sp.I*H,H,sp.I*H,H) if name=='Q' else (H,-sp.I*H,H,sp.I*H)
        mixing={(a,b):(b,a,lambdas[2*a+b]) for a in range(2) for b in range(2)}
        model=WordModel(scalar,'coordinate','balanced',H**2,1,mixing,name)
        model.phase_variables=(H,)
        model.parameter_map={H:h}
        return model
    if name=='R':
        mixing={(0,0):(0,0,H),(0,1):(1,0,H),
                (1,0):(1,1,H),(1,1):(0,1,-H)}
        model=WordModel(scalar,'coordinate','coordinate',-1,1,mixing,name)
        model.phase_variables=(H,)
        model.parameter_map={H:h}
        return model
    # a is the first eigenvalue of conjugation by Z1; b its swap analogue.
    a=H
    b=-H if name=='F' else H
    mixing={(0,0):(0,0,a),(0,1):(1,0,-a),(1,0):(1,1,b),(1,1):(0,1,b)}
    model=WordModel(scalar,'coordinate','balanced',-1,1,mixing,name)
    model.phase_variables=(H,)
    model.parameter_map={H:-h*(1-p) if name=='F' else 1/(h*(1-q))}
    return model


class TorusWords:
    """Extract commutation and action matrices from exact Laurent words."""
    def __init__(self,model,generators,names):
        self.model=model
        self.generators=tuple(generators)
        self.names=tuple(names)
        columns=[sp.Matrix(w.normal()[0]) for w in generators]
        self.basis=sp.ImmutableMatrix.hstack(*columns)
        if self.basis.det()==0:
            raise ValueError('Torus generators must have independent exponent vectors')

    def coordinates(self,word):
        exponents,coefficient=word.normal()
        vector=self.basis.inv()*sp.Matrix(exponents)
        if not all(v.is_Integer for v in vector):
            raise ValueError('The word is outside the specified torus lattice')
        powers=tuple(map(int,vector))
        base=self.model.identity()
        for gen,power in zip(self.generators,powers):
            base=base*gen**power
        base_exponents,base_coefficient=base.normal()
        if base_exponents!=exponents:
            raise ValueError('The exponent chart is not additive on this subgroup')
        return powers,self.model.scalar(coefficient/base_coefficient)

    def commutations(self):
        result=sp.ones(4)
        for i in range(4):
            for j in range(i+1,4):
                result[i,j]=self.generators[i].commutation(self.generators[j])
                result[j,i]=self.model.scalar(1/result[i,j])
        return sp.ImmutableMatrix(result)

    def action(self,conjugator):
        images=[self.coordinates(conjugator.conjugate(v)) for v in self.generators]
        return {'matrix':sp.ImmutableMatrix.hstack(*(sp.Matrix(v) for v,c in images)),
                'scalars':tuple(c for v,c in images)}


def standard_torus(model):
    e,z=model.e1,model.z1
    r=e**-1*model.e2;rho=z**-1*model.z2
    return TorusWords(model,(r,rho,e**2,z**2),('r','rho','X','Y'))


def normalizer_torus(model):
    e,z=model.e1,model.z1
    r=e**-1*model.e2;rho=z**-1*model.z2
    if model.name in ('Q','V'):
        return TorusWords(model,(e,model.e2,rho,z**2),('n','Theta','rho','Y')),(z,)
    if model.name=='R':
        return TorusWords(model,(r,e**2,z,model.z2**2),('r','X','Z1','W')),(e,model.z2)
    return TorusWords(model,(r,e**2,z**2,rho**2),('r','X','Y','P')),(e,z,rho)


In [8]:
# Normalizer, affine and rational center identities
"""Symbolic polynomial and rational-coordinate identities, with manuscript labels."""
from dataclasses import replace
import sympy as sp


def ratio_of_elements(left,right):
    if not left or not right or set(left.terms)!=set(right.terms):return None
    word=next(iter(left.terms))
    scalar=left.algebra.scalar(left.terms[word]/right.terms[word])
    return scalar if not left-scalar*right else None


def rational_coordinate_matrix(family,normal_index):
    """Derive Y*t=M(t)Y directly from Sigma, t=n^-1*x'."""
    n=normal_index;other=1-n
    sigma=family.sigma
    S=sp.Matrix([[sigma[2*a+n,2*b+n] for b in range(2)] for a in range(2)])
    cross=sp.Matrix([[sigma[2*a+n,2*b+other] for b in range(2)] for a in range(2)])
    if any(cross):raise ValueError('The selected generator is not normal in this chart')
    U=sp.Matrix([[sigma[2*a+other,2*b+n] for b in range(2)] for a in range(2)])
    V=sp.Matrix([[sigma[2*a+other,2*b+other] for b in range(2)] for a in range(2)])
    return family.matrix(S.inv()*(U+t*V)),family.matrix(S)


def ore_algebra(beta1,beta2,scalar=sp.cancel):
    return Algebra.from_rules({
        (1,0):{(0,1):p,(2,):-p*beta1},
        (2,0):{(0,2):1/p,(1,):-beta2/p},
        (2,1):{(1,2):p}},scalar,('t','y1','y2'))


def extended_polynomial_checks(check):
    def polynomial(identifier,family,residual,label,category='additional PBW identities'):
        check(identifier,category,family,not residual,residual.certificate(),labels=(label,))
    A=Algebra(FAMILIES['L']);x1,x2,y1,y2=A.generators
    uu=x1**2;ww=x1*x2
    for j,y in enumerate((y1,y2)):
        polynomial(f'L.torus.u.{j}','L',y*uu-(h*f)**2*uu*y,'eq:Ltorus')
        polynomial(f'L.torus.w.{j}','L',y*ww-h**2*f*ww*y,'eq:Ltorus')
    polynomial('L.torus.wu','L',ww*uu-q**2*uu*ww,'eq:Ltorus')
    polynomial('L.torus.correct_first_sign','L',ww*y1-(1/(h**2*f))*y1*ww,'thm:L')
    wrong=ww*y1-h**2*f*y1*ww
    check('L.manuscript_first_sign_rejected','manuscript discrepancy controls','L',bool(wrong),
          {'expected':'nonzero residual for the manuscript sign','residual':wrong.certificate(),
           'correction':'a12=-(2*eta+phi)'},labels=('thm:L',))
    for name in ('Q','V'):
        A=Algebra(FAMILIES[name]);x1,x2,y1,y2=A.generators
        theta=(x2*(y2**2-y1**2)-x1*(y1**2-2*y1*y2+y2**2) if name=='Q'
               else x1*(y1**2-y2**2)-x2*(y1**2+y1*y2))
        n=x1 if name=='Q' else x2
        tests=([y1*theta-h*theta*y1,y2*theta-h*theta*y2] if name=='Q'
               else [y1*theta-h*theta*y2,y2*theta+h*theta*y1])
        tests.append(theta*n-h**2*n*theta)
        for j,r in enumerate(tests):polynomial(f'{name}.Theta.{j}',name,r,'lem:Theta')
        for sign in (1,-1):
            z=y1+sign*sp.I*y2
            xn=-sign*sp.I*h if name=='Q' else h
            th=h if name=='Q' else -sign*sp.I*h
            polynomial(f'{name}.model.n.{sign}',name,z*n-xn*n*z,'eq:Qmodel' if name=='Q' else 'eq:Vmodel')
            polynomial(f'{name}.model.theta.{sign}',name,z*theta-th*theta*z,'eq:Qmodel' if name=='Q' else 'eq:Vmodel')

    A=Algebra(FAMILIES['R']);x1,x2,y1,y2=A.generators
    zp,zm=y1+y2,y1-y2;ep,em=x1+x2,x1-x2
    Z1=x1*zp+x2*zm;Z2=x1*y2+x2*y1
    tests=[Z1*x1-h*x1*Z1,Z1*x2-h*x2*Z1,Z2*x1-h*x2*Z2,
           Z2*x2+h*x1*Z2,Z2*Z1-Z1*Z2]
    for j,r in enumerate(tests):polynomial(f'R.normalizers.{j}','R',r,'lem:Rnorm')
    tests=[zp*x1-h*(ep*zp+x2*zm),zp*x2+h*x2*zm,
           zm*x1-h*x1*zp,zm*x2-h*(x1*zp-em*zm)]
    for j,r in enumerate(tests):polynomial(f'R.zbasis.{j}','R',r,'eq:Rz')

    A=Algebra(FAMILIES['F']);x1,x2,y1,y2=A.generators
    gg,gb=x1+p*x2,x1-p*x2;c=h*(1-p)
    Z1=gg*y1-gb*y2;Z2=gg*y1+p*gb*y2
    tests=[Z1*x1+c*x1*Z1,Z1*x2-c*x2*Z1,Z2*x1-c*x2*Z2,
           Z2*x2-c*x1*Z2,Z2**2+p*Z1**2]
    for j,r in enumerate(tests):polynomial(f'F.normalizers.{j}','F',r,'lem:Fnorm')
    tests=[y1*gg-h*(1+p)*gg*y2,y1*gb-h*(-2*gg*y1+(1-p)*gb*y2),
           y2*gb+h*(1+p)*gb*y1,y2*gg-h*((1-p)*gg*y1+2*p*gb*y2),gg**2-gb**2]
    for j,r in enumerate(tests):polynomial(f'F.gbasis.{j}','F',r,'eq:Fg')

    A=Algebra(FAMILIES['I']);x1,x2,y1,y2=A.generators
    w1,w2=y2-q*y1,-y2-q*y1;c=h*(1-q)
    Z1=x1*w1+x2*w2;Z2=x1*w1+q*x2*w2
    tests=[y1*Z1-c*Z1*y1,y2*Z1+c*Z1*y2,y1*Z2-c*Z2*y2,
           y2*Z2-c*Z2*y1,Z2**2-q*Z1**2]
    for j,r in enumerate(tests):polynomial(f'I.normalizers.{j}','I',r,'lem:Inorm')
    tests=[w1*x1+h*(1+q)*x2*w1,w1*x2-h*((1-q)*x1*w1+2*x2*w2),
           w2*x2-h*(1+q)*x1*w2,w2*x1-h*(-2*q*x1*w1+(1-q)*x2*w2),w1**2-w2**2]
    for j,r in enumerate(tests):polynomial(f'I.wbasis.{j}','I',r,'eq:Iw')

    for name in ('D','G'):
        A=Algebra(FAMILIES[name]);x1,x2,y1,y2=A.generators
        if name=='G':x2=x2+x1*(1/(p-1))
        b=(1 if name=='D' else f)/(p**2-1)
        c=(1 if name=='D' else -1)/(p**2-1)
        Xi=x2*y1*y2-x1*(c*y2**2+b/p*y1**2)
        qq=h*p
        weights=[qq**2,-qq**2,-1/qq,1/qq] if name=='D' else [qq**2,qq**2,1/qq,1/qq]
        for j,(gen,weight) in enumerate(zip((x1,x2,y1,y2),weights)):
            polynomial(f'{name}.Xi.normal.{j}',name,Xi*gen-weight*gen*Xi,'rem:DGnormal')

    B1,B2=sp.symbols('beta1 beta2')
    A=ore_algebra(B1,B2);tt,y1,y2=A.generators
    b=-B2/(p**2-1);c=p**2*B1/(p**2-1)
    omega=tt*y1*y2-b/p*y1**2-c*y2**2
    for j,gen in enumerate(A.generators):
        polynomial(f'A.Omega.central.{j}','D/G',omega.commutator(gen),'lem:Omega')
    for word,residual in A.overlaps().items():
        polynomial(f'A.PBW.{word}','D/G',residual,'eq:Arel')

    # Normal elements at which the primitive monomial algebras are localized.
    for name in ('M','N','O','P','W','Z'):
        datum=MONOMIAL[name];A=Algebra(datum.family);x1,x2,y1,y2=A.generators
        es=[datum.x_basis[j,0]*x1+datum.x_basis[j,1]*x2 for j in range(2)]
        zs=[datum.y_basis[j,0]*y1+datum.y_basis[j,1]*y2 for j in range(2)]
        nx=(es[0]**2 if name in ('M','O','P','Z') else es[0]*es[1] if name=='W' else x1*x2)
        ny=(y1*y2 if name=='P' else zs[0]*zs[1] if name=='Z' else zs[0]**2)
        for element_name,element in [('Nx',nx),('Ny',ny)]:
            for j,gen in enumerate(es+zs):
                factor=ratio_of_elements(element*gen,gen*element)
                check(f'{name}.localizer.{element_name}.{j}','localization normality',name,
                      factor is not None,{'factor':str(factor)},labels=('lem:loc',))

    # Anti-isomorphisms: evaluate every defining relation with reversed products.
    target=Algebra(FAMILIES['E']);x1,x2,y1,y2=target.generators
    source=replace(FAMILIES['J'],Q=(1/p,sp.Integer(0)),relations=((p**2+1,p),))
    _anti_relations(check,'EJ',Algebra(source),target,(y1,y2,x1,x2),'eq:EJ')
    target=Algebra(FAMILIES['Z'].specialize({f:s**2}));x1,x2,y1,y2=target.generators
    source=Algebra(FAMILIES['W'].specialize({f:1/s**2,h:s*h}))
    _anti_relations(check,'WZ',source,target,(s*y1+y2,s*(s*y1-y2),x1,x2),'rem:monodeg')


def _anti_relations(check,name,source,target,images,label):
    for word,terms in source.rules.items():
        lhs=images[word[1]]*images[word[0]]
        rhs=target.element(0)
        for v,c in terms.items():
            term=target.element(c)
            for j in reversed(v):term=term*images[j]
            rhs=rhs+term
        residual=lhs-rhs
        check(f'duality.{name}.{word}','anti-isomorphism identities',name,not residual,
              residual.certificate(),labels=(label,))


def rational_center_checks(check):
    """Certify fractions by a common normalizing automorphism for numerator/denominator."""
    for name in ('B','C','F','I','R','K','L','S','H','X'):
        family=FAMILIES[name];A=Algebra(family);x1,x2,y1,y2=A.generators
        if name=='B':
            num=(x1**4+x2**4)**2;den=x1**4*x2**4
            images=(x1,x2,h**8*y1,h**8*y2)
            for j,y in enumerate((y1,y2)):
                for k,(left,right) in enumerate(((x1,x2),(x2,x1))):
                    residual=y*left**4-h**4*right**4*y
                    check(f'B.fourth_power_swap.{j}.{k}','central rational certificates','B',
                          not residual,residual.certificate(),labels=('eq:Bswap',))
        elif name in ('F','R'):
            num=(x1*x2)**2;den=(x1**2+x2**2)**2
            factor=(-2*p*h**2)**2 if name=='F' else h**4
            images=(x1,x2,(1/factor)*y1,(1/factor)*y2)
            # Use right normalization num*y = factor^(-1)*y*num.
        elif name=='I':
            num=(y1*y2)**2;den=(y1**2+y2**2)**2
            factor=(-2*q*h**2)**2
            images=(factor*x1,factor*x2,y1,y2)
        elif name=='C':
            a=y1**2*y2;b=y1**3-y2**3
            num=a**3;den=b**3
            factor=(3*(2*p+1)*h**3)**3
            images=(factor*x1,factor*x2,y1,y2)
        elif name=='K':
            num=f**2*y1**4+y2**4;den=y1**2*y2**2
            images=(h**4*x1,h**4*f**2*x2,y1,y2)
        elif name=='L':
            num=y1**4+y2**4;den=y1**2*y2**2
            images=(h**4*f**4*x1,h**4*x2,y1,y2)
        elif name=='S':
            num=2*(x1**4-6*x1**2*x2**2+x2**4);den=(x1**2+x2**2)**2
            images=(x1,x2,(1/(16*h**4))*y1,(1/(16*h**4))*y2)
        else:
            plus=y1**2+y2**2;minus=y1**2-y2**2
            num=minus**2;den=plus**2
            ff=f if name=='H' else sp.Integer(1)
            for j,(gen,image) in enumerate(zip((x1,x2),(x1,2*ff*x1+x2))):
                for sign,element in [(1,plus),(-1,minus)]:
                    residual=element*gen-sign*h**2*image*element
                    check(f'{name}.S_pm.{sign}.{j}','central rational certificates',name,
                          not residual,residual.certificate(),labels=('prop:HXprim',))
            images=(h**4*x1,h**4*(4*ff*x1+x2),y1,y2)
        label=('prop:Bnotprim' if name=='B' else 'prop:HXprim' if name in ('H','X')
               else 'thm:primmono' if name in ('K','L','S') else 'prop:CFIQRV')
        for which,element in [('numerator',num),('denominator',den)]:
            for j,(gen,image) in enumerate(zip(A.generators,images)):
                # B uses left normalization g*N=phi(g)*N; all others N*g=phi(g)*N.
                residual=(gen*element-element*image if name=='B' else element*gen-image*element)
                check(f'{name}.rational_center.{which}.{j}','central rational certificates',name,
                      not residual,residual.certificate(),labels=(label,))
        residual=num.commutator(den)
        check(f'{name}.rational_center.commuting','central rational certificates',name,
              not residual,residual.certificate(),labels=(label,))


def affine_nonpi_checks(check):
    table=[('X',{},0,sp.eye(2),1),('Y',{},0,sp.Matrix([[f,1],[1,f]]),-1),
           ('D',{p:1},0,sp.Matrix([[0,-1],[1,0]]),1),
           ('D',{p:-1},0,sp.Matrix([[0,1],[-1,0]]),-1),
           ('M',{f:0},1,sp.Matrix([[0,1],[-1,0]]),-1),
           ('P',{f:0},0,sp.diag(-1,1),1),
           ('W',{f:0},0,sp.Matrix([[0,1],[1,0]]),-1)]
    z=sp.Symbol('z')
    for index,(name,subs,normal,C,a) in enumerate(table):
        family=FAMILIES[name].specialize(subs)
        matrix,S=rational_coordinate_matrix(family,normal)
        residual=family.matrix(matrix-a*t*sp.eye(2)-C)
        check(f'{name}.affine.{index}','non-PI affine matrices',name,not any(residual),
              {'derived_matrix':str(matrix),'residual':str(residual)},labels=('prop:noPI',))
        characteristic=sp.factor(C.charpoly(z).as_expr())
        valid=(C.trace()!=0 or C.det()!=0) if a==1 else sp.factor(C.trace()**2-4*C.det())!=0
        check(f'{name}.affine.eigenvalues.{index}','non-PI affine matrices',name,valid,
              {'charpoly':str(characteristic),'a':a},labels=('lem:shiftmat','prop:noPI'))
        square=family.matrix(S*S-h**2*sp.eye(2))
        check(f'{name}.central_power_matrix.{index}','central power certificates',name,
              not any(square),{'S_squared':str(family.matrix(S*S)),
                               'general_exponent':'m=lcm(2,ord(h))'},labels=('prop:HXprim',))
    for name in ('D','G'):
        matrix,_=rational_coordinate_matrix(FAMILIES[name],0)
        if name=='G':
            shift=1/(p-1)
            matrix=matrix.subs(t,t-shift)+shift*sp.eye(2)
        expected=sp.Matrix([[p*t,(-1 if name=='D' else 1)/p],
                            [(1 if name=='D' else f)/p,t/p]])
        residual=FAMILIES[name].matrix(matrix-expected)
        check(f'{name}.rational_matrix_from_Sigma','normal-generator matrices',name,
              not any(residual),str(residual),labels=('eq:MDG',))
    for name,normal,expected in [('Q',0,sp.Matrix([[-1,1-t],[1+t,1]])),
                                 ('V',1,sp.Matrix([[1,t],[1-t,0]]))]:
        matrix,_=rational_coordinate_matrix(FAMILIES[name],normal)
        residual=FAMILIES[name].matrix(matrix-expected)
        check(f'{name}.rational_matrix_from_Sigma','normal-generator matrices',name,
              not any(residual),str(residual),labels=('prop:QV',))
    for name in ('A','Y'):
        _,S=rational_coordinate_matrix(FAMILIES[name],0)
        residual=FAMILIES[name].matrix(S-h*sp.eye(2))
        check(f'{name}.central_power_all_orders','central power certificates',name,
              not any(residual) and FAMILIES[name].Q==(1,0),
              {'normalizing_matrix':str(S),'general_exponent':'ord(h)'},labels=('prop:HXprim',))


In [9]:
# All 21 degree models and exact center lattices
"""PI degree certificates for all 21 families with a closed formula.

No root-order cutoff: inputs are exact integer phases in a common cyclic
group. Torus matrices and crossed-product actions are derived from Laurent
words. The center lattice is computed by Hermite normal form and its index
independently by Smith normal form. Theoretical division-ring arguments are
identified in the coverage document, not claimed as machine formalization.
"""
from functools import lru_cache
from itertools import product
from math import gcd,prod,lcm
import sympy as sp
from sympy.matrices.normalforms import hermite_normal_form


def phase(expression,order,assignments):
    expression=sp.factor(sp.cancel(expression))
    if expression==1:return 0
    if expression==-1:
        if order%2:raise ValueError('The phase group must contain -1')
        return order//2
    if expression==sp.I:
        if order%4:raise ValueError('The phase group must contain i')
        return order//4
    if expression in assignments:
        value=sp.sympify(assignments[expression])
        if not value.is_Integer:raise ValueError('Phase exponents must be exact integers')
        return int(value)%order
    if expression.is_Mul:
        return sum(phase(a,order,assignments) for a in expression.args)%order
    if expression.is_Pow and expression.exp.is_Integer:
        return int(expression.exp)*phase(expression.base,order,assignments)%order
    raise ValueError(f'Not a Laurent monomial in the supplied phase variables: {expression}')


def center_lattice(matrix,order):
    """Gamma={a: A*a=0 mod N}, using the dual of Z^d+(A^T/N)Z^d."""
    matrix=sp.Matrix(matrix);size=matrix.rows
    hnf=hermite_normal_form(sp.Matrix.hstack(order*sp.eye(size),matrix.T))
    basis=order*hnf.inv().T
    if not all(a.is_Integer for a in basis):
        raise ArithmeticError('The dual center lattice must be integral')
    if any(int(a)%order for a in matrix*basis):
        raise ArithmeticError('Computed basis does not belong to the modular kernel')
    expected=torus_certificate(matrix,order)['image_size']
    if abs(int(basis.det()))!=expected:
        raise ArithmeticError('Hermite center index disagrees with Smith index')
    return sp.ImmutableMatrix(basis)


def cocycle(matrix,a,b):
    return -sum(int(matrix[i,j])*a[j]*b[i]
                for i in range(len(a)) for j in range(i+1,len(a)))


def monomial_multiply(matrix,order,left,right):
    k,a=left;l,b=right
    return ((k+l+cocycle(matrix,a,b))%order,tuple(x+y for x,y in zip(a,b)))


def monomial_power(matrix,order,monomial,n):
    k,a=monomial
    return ((n*k+n*(n-1)//2*cocycle(matrix,a,a))%order,tuple(n*x for x in a))


def apply_action(matrix,order,action,exponent):
    result=(0,(0,)*matrix.rows)
    for j,n in enumerate(exponent):
        image=(action['phases'][j],tuple(map(int,action['matrix'].col(j))))
        result=monomial_multiply(matrix,order,result,monomial_power(matrix,order,image,int(n)))
    return result


def action_signature(matrix,order,action,center):
    return tuple(apply_action(matrix,order,action,tuple(map(int,center.col(j))))
                 for j in range(center.cols))


@lru_cache(maxsize=None)
def symbolic_model(name):
    if name in ('F','I','Q','R','V'):
        model=normalizer_model(name)
        torus,conjugators=normalizer_torus(model)
    else:
        model=monomial_model(name,formal=True)
        torus=standard_torus(model)
        conjugators=(model.e1,model.z1)
    units=[]
    labels=[]
    for powers in product((0,1),repeat=len(conjugators)):
        word=model.identity()
        for w,k in zip(conjugators,powers):word=word*w**k
        units.append(word)
        labels.append(''.join(map(str,powers)))
    return model,torus,tuple(units),tuple(labels)


def closed_degree(name,order,assignments):
    def value(expr):return phase(expr,order,assignments)
    def root_order(expr):return order//gcd(order,value(expr))
    if name=='B':return 8*root_order(H**4)
    if name in ('E','J'):return 8*root_order(-alpha**4)
    if name in ('S','T','U'):return 8*root_order(H**4)
    if name in ('F','I'):return 8*root_order(H**4)
    if name in ('Q','R','V'):return 4*root_order(H**2)
    if name in ('M','N','O','P','W','Z'):
        u=(-alpha/beta if name=='Z' else alpha/beta)
        v=alpha*beta
        factor=4
        if name in ('O','W','Z') and root_order(u)%2 and root_order(v)%2:
            factor=2
        return factor*root_order(u**2)*root_order(v**2)
    if name in ('K','L'):
        ff=S**2 if name=='K' else F
        denominator=gcd(order,value(-1),value(q),value(ff),value(H))
        n=order//denominator
        if name=='K':return 2*n
        n0=order//gcd(order,value(-1),value(q),value(ff))
        return 2*n if (n//n0)%2 else n
    raise KeyError(name)


def degree_certificate(name,order,assignments,include_action_tables=True):
    """Compute from torus + actions, then compare with the manuscript formula.

    The common order may be any multiple of the actual scalar orders. Including
    i in that common cyclic group is convenient for every monomial family.
    """
    if not isinstance(order,int) or order<1:raise ValueError('Positive integer order required')
    if not all(sp.sympify(v).is_Integer for v in assignments.values()):
        raise ValueError('Phase exponents must be exact integers')
    assignments={sp.sympify(k):int(v)%order for k,v in assignments.items()}
    if name in ('D','G'):
        k,kp=assignments[p],assignments[H]
        if order//gcd(order,k)<=2:raise ValueError('D/G model requires p != +/-1')
        if name=='D' and order%2:raise ValueError('D requires a common group containing -1')
        entries=(order//2,-order//2-k-kp,-k-kp,-k,k,-k) if name=='D' else (0,-k-kp,-k-kp,-k,k,-k)
        matrix=alternating(entries)
        cert=torus_certificate(matrix,order)
        divisor=gcd(order,order//2,k,kp) if name=='D' else gcd(order,k,kp)
        formula=dg_closed_formula(order//divisor,k//divisor,kp//divisor)
        return {'family':name,**cert,'center_lattice':list(map(list,center_lattice(matrix,order).tolist())),
                'center_action_order':1,'algebra_pi_degree':cert['pi_degree'],
                'closed_formula':formula,'agrees':cert['pi_degree']==formula}
    if name in ('B','E'):
        if order%4 or (2*assignments[p])%order!=order//2:
            raise ValueError('p must have square -1')
    if name=='J':
        if order%4 or (2*assignments[q])%order!=order//2:
            raise ValueError('q must have square -1')
    if name in ('M','O','P','W','Z'):
        difference=(assignments[alpha]-assignments[beta])%order
        if difference==0 or (order%2==0 and difference==order//2):
            raise ValueError('These root phases give a singular/degenerate basis or h=0')
    model,torus,units,labels=symbolic_model(name)
    commutation=torus.commutations()
    matrix=alternating([phase(commutation[i,j],order,assignments)
                        for i in range(4) for j in range(i+1,4)])
    certificate=torus_certificate(matrix,order)
    center=center_lattice(matrix,order)
    signatures=[];actions=[]
    for unit,label in zip(units,labels):
        data=torus.action(unit)
        action={'matrix':data['matrix'],
                'phases':tuple(phase(c,order,assignments) for c in data['scalars'])}
        if any(int(v)%order for v in action['matrix'].T*matrix*action['matrix']-matrix):
            raise ArithmeticError('Action does not preserve the torus pairing')
        signatures.append(action_signature(matrix,order,action,center))
        actions.append({'coset':label,'exponent_matrix':data['matrix'].tolist(),
                        'phases':list(action['phases'])})
    image_size=len(set(signatures))
    if len(units)==4:
        if image_size not in (2,4):raise ArithmeticError('Unexpected center action image')
    elif image_size!=len(units):
        raise ArithmeticError('The normalizer model requires distinct center actions')
    computed=image_size*certificate['pi_degree']
    formula=closed_degree(name,order,assignments)
    result={'family':name,**certificate,'center_lattice':center.tolist(),
            'torus_generators':list(torus.names),'coset_count':len(units),
            'center_action_order':image_size,'algebra_pi_degree':computed,
            'closed_formula':formula,'agrees':computed==formula}
    if include_action_tables:result['actions']=actions
    return result


def c_bound_certificate(order,p_exponent,beta_exponent):
    """C remains open in the manuscript: return its certified possible degrees."""
    if order%3 or order//gcd(order,p_exponent)!=3:
        raise ValueError('p must be a primitive cube root')
    k,l=p_exponent,beta_exponent
    matrix=alternating((-k,-l-2*k,-l,-l-2*k,-l,0))
    certificate=torus_certificate(matrix,order)
    n=order//gcd(order,k,l)
    if certificate['pi_degree']!=n:raise ArithmeticError('C subtorus degree disagrees')
    ratios=sorted({ell*b for a in (1,3,9) for b in (1,3,9) for ell in (1,3)
                   if a*b*ell*ell==9})
    return {'family':'C','subtorus':certificate,'module_rank':9,
            'possible_degree_ratios':ratios,'possible_pi_degrees':[n*x for x in ratios],
            'exact_degree_is_open':True}


def phases_of_exact_roots(values):
    """Discrete logs in one exact cyclotomic field, without numerical angles."""
    orders={key:exact_root_order(value) for key,value in values.items()}
    if any(value in (None,0) for value in orders.values()):
        raise ValueError('All phase variables must be assigned exact roots of unity')
    order=lcm(4,*orders.values())
    zeta=sp.exp(2*sp.pi*sp.I/order)
    field=sp.QQ.algebraic_field(zeta)
    root=field.from_sympy(zeta)
    lookup={};power=field.one
    for k in range(order):
        lookup[tuple(power.to_list())]=k
        power*=root
    assignments={key:lookup[tuple(field.from_sympy(sp.simplify(value)).to_list())]
                 for key,value in values.items()}
    return order,assignments


def degree_from_parameters(name,parameters):
    """Exact p,q,f,g,h interface. C returns its three possibilities, not a guess."""
    parameters={sp.sympify(k):sp.sympify(v) for k,v in parameters.items()}
    defaults={}
    if name in ('B','E','F') and p not in parameters:defaults[p]=sp.I
    if name in ('I','J') and q not in parameters:defaults[q]=sp.I
    if name=='C' and p not in parameters:defaults[p]=(-1+sp.sqrt(3)*sp.I)/2
    parameters={**defaults,**parameters}
    decision=pi_decision(name,parameters)
    if decision['status'] is not True:
        raise ValueError('Parameters are non-PI or not sufficiently specified')
    value=lambda x:sp.simplify(sp.sympify(x).subs(parameters,simultaneous=True))
    root=sp.sqrt(value(f))
    if name in ('D','G'):values={p:value(p),H:value(h)}
    elif name=='B':values={p:value(p),H:value(h)}
    elif name in ('E','J'):
        values={alpha:value(h)*(1+sp.I),(p if name=='E' else q):value(p if name=='E' else q)}
    elif name=='K':values={q:value(q),S:root,H:value(h)}
    elif name=='L':values={q:value(q),F:value(f),H:value(h)}
    elif name=='M':values={alpha:value(h)*(root+1),beta:value(h)*(root-1)}
    elif name=='N':values={alpha:value(f-g),beta:value(f+g)}
    elif name in ('O','P'):values={alpha:value(h)*(1+root),beta:value(h)*(1-root)}
    elif name in ('S','T','U'):values={H:2*value(h)}
    elif name in ('W','Z'):values={alpha:value(h)*(root-sp.I),beta:value(h)*(root+sp.I)}
    elif name=='F':values={H:-value(h*(1-p))}
    elif name=='I':values={H:1/value(h*(1-q))}
    elif name in ('Q','R','V'):values={H:value(h)}
    elif name=='C':values={p:value(p),beta:value(3*(2*p+1)*h**3)}
    else:raise KeyError(name)
    order,assignments=phases_of_exact_roots(values)
    result=(c_bound_certificate(order,assignments[p],assignments[beta]) if name=='C'
            else degree_certificate(name,order,assignments))
    result['exact_parameters']={str(k):str(v) for k,v in parameters.items()}
    result['default_parameter_branches']={str(k):str(v) for k,v in defaults.items()}
    result['phase_assignments']={str(k):int(v) for k,v in assignments.items()}
    return result


def all_degree_checks(check):
    names=('B','E','J','K','L','M','N','O','P','S','T','U','W','Z','F','I','Q','R','V','D','G')
    label=lambda name:('prop:DGdeg' if name in ('D','G') else 'thm:L' if name=='L'
                       else 'thm:FIQRV' if name in ('F','I','Q','R','V') else 'thm:monodeg')
    for name in names:
        for order in (8,12,20,3988):
            assignments={p:order//4,q:order//4,H:1,S:1,F:1,alpha:1,beta:2}
            if name in ('D','G'):assignments[p]=1 if order!=3988 else 4
            certificate=degree_certificate(name,order,assignments)
            check(f'{name}.complete_degree_model.{order}','all-family degree certificates',name,
                  certificate['agrees'],certificate,labels=(label(name),'tab:degrees'))
    for name in ('O','W','Z'):
        for a,b in ((2,6),(5,7),(4,8)):
            certificate=degree_certificate(name,12,{alpha:a,beta:b})
            check(f'{name}.parity_branch.{a}.{b}','center-action parity branches',name,
                  certificate['agrees'],certificate,labels=('thm:monodeg',))
    for name in ('B','E','J'):
        for branch in (1,3):
            assignment={p:3*branch,q:3*branch,H:0,alpha:1}
            certificate=degree_certificate(name,12,assignment)
            check(f'{name}.degree_branch.{branch}','fourth-root degree branches',name,
                  certificate['agrees'],certificate,labels=(label(name),))
    # L's two parity cases and K's same-parameter distinction.
    for name in ('K','L'):
        for assignment in ({q:0,H:0,F:0,S:0},{q:0,H:1,F:0,S:0},{q:1,H:0,F:0,S:0}):
            certificate=degree_certificate(name,4,assignment)
            check(f'{name}.degree_special.{assignment}','K/L parity certificates',name,
                  certificate['agrees'],certificate,labels=('thm:L','tab:degrees'))
    for order,k,l in ((12,4,1),(12,8,3),(60,20,7)):
        certificate=c_bound_certificate(order,k,l)
        check(f'C.bound_model.{order}.{k}.{l}','C bound certificates','C',
              certificate['possible_degree_ratios']==[1,3,9],certificate,labels=('prop:bounds',))


In [10]:
# Crossed-product and tensor-splitting certificates
"""Word-level certificates for torus relations, conjugations and crossed products."""
from itertools import product
import sympy as sp


def word_checks(check):
    for name,datum in MONOMIAL.items():
        model=monomial_model(name)
        formal=monomial_model(name,True)
        e,z=model.e1,model.z1
        es=(e,model.e2);zs=(z,model.z2)
        r=e**-1*model.e2;rho=z**-1*model.z2;X=e**2;Y=z**2
        sx=datum.kind in ('I','III');sy=datum.kind in ('I','II')
        sig=lambda b:1-b if sx else b
        tau=lambda a:1-a if sy else a
        lam=lambda a,b:datum.lambdas[2*a+b]
        def verify(identifier,left,right,label):
            check(name+'.'+identifier,'Laurent identities',name,left.equals(right),
                  {'left':left.certificate(),'right':right.certificate()},labels=(label,))
        for a,b in product(range(2),repeat=2):
            verify(f'A1.z_inverse.{a}.{b}',zs[a]*es[b]**-1,
                   (1/lam(tau(a),b))*es[sig(b)]**-1*zs[tau(a)],'lem:A1')
            verify(f'A1.e_inverse.{a}.{b}',es[b]*zs[a]**-1,
                   lam(tau(a),b)*zs[tau(a)]**-1*es[sig(b)],'lem:A1')
            verify(f'A3.square_e.{a}.{b}',zs[a].conjugate(es[b]**2),
                   lam(a,b)*lam(tau(a),b)*es[sig(b)]**2,'eq:A3')
            verify(f'A3.square_z.{a}.{b}',es[b].conjugate(zs[a]**2),
                   (1/(lam(tau(a),b)*lam(tau(a),sig(b))))*zs[tau(a)]**2,'eq:A3')
        for a,b,c in product(range(2),repeat=3):
            verify(f'A2.ratio_e.{a}.{b}.{c}',zs[a].conjugate(es[b]**-1*es[c]),
                   (lam(tau(a),c)/lam(tau(a),b))*es[sig(b)]**-1*es[sig(c)],'eq:A2')
            verify(f'A2.ratio_z.{a}.{b}.{c}',es[b].conjugate(zs[a]**-1*zs[c]),
                   (lam(tau(a),b)/lam(tau(c),b))*zs[tau(a)]**-1*zs[tau(c)],'eq:A2')
        verify('A3.kappa',rho.conjugate(r),
               ((lam(tau(1),1)/lam(tau(1),0))/(lam(tau(0),1)/lam(tau(0),0)))*r,'lem:A3')
        verify('A3.z_r',z.conjugate(r),(lam(tau(0),1)/lam(tau(0),0))*r**(-1 if sx else 1),'lem:A3')
        verify('A3.e_rho',e.conjugate(rho),(lam(tau(0),0)/lam(tau(1),0))*rho**(-1 if sy else 1),'lem:A3')
        # Check the actual transformed plane relation, so the word calculus does
        # not merely assume that a proposed basis is balanced or coordinate.
        A=Algebra(datum.family);x1,x2,y1,y2=A.generators
        xe=[datum.x_basis[j,0]*x1+datum.x_basis[j,1]*x2 for j in range(2)]
        yz=[datum.y_basis[j,0]*y1+datum.y_basis[j,1]*y2 for j in range(2)]
        for side,kind,param,values in [('x',model.x_kind,model.x_q,xe),('y',model.y_kind,model.y_q,yz)]:
            residual=(values[0]**2-values[1]**2 if kind=='balanced'
                      else values[1]*values[0]-param*values[0]*values[1])
            check(f'{name}.plane_chart.{side}','plane charts',name,not residual,
                  residual.certificate(),labels=('lem:A2','tab:mono'))
        actualT,formalT=standard_torus(model),standard_torus(formal)
        am,fm=actualT.commutations(),formalT.commutations()
        residual=datum.family.matrix(am-fm.subs(formal.substitutions,simultaneous=True))
        check(f'{name}.phase_model_matches_source','formal parameter charts',name,not any(residual),
              str(residual),labels=('thm:monodeg',))
        for j,(aw,fw) in enumerate([(model.e1,formal.e1),(model.z1,formal.z1)]):
            aa,fa=actualT.action(aw),formalT.action(fw)
            residual=datum.family.matrix(sp.Matrix(aa['scalars'])-
                       sp.Matrix(fa['scalars']).subs(formal.substitutions,simultaneous=True))
            check(f'{name}.phase_action_matches_source.{j}','formal parameter charts',name,
                  aa['matrix']==fa['matrix'] and not any(residual),str(residual),labels=('thm:monodeg',))

    # Symbolic tensor splitting, including the coordinate-basis corrections.
    for name in ('B','E','J','M','N','O','P','S','T','U','W','Z'):
        model=monomial_model(name,True);base=standard_torus(model)
        r,rho,X,Y=base.generators
        xp=X*rho if name in ('B','J') else X*r if name in ('N','U','W') else X
        yp=Y*r if name in ('B','E') else Y*rho if name in ('P','T','Z') else Y
        split=TorusWords(model,(r,rho,xp,yp),('r','rho','Xprime','Yprime'))
        matrix=split.commutations()
        expected=sp.ones(4)
        first=(-1 if name in ('B','E','J','S','T','U')
               else (alpha/beta)**2 if name in ('M','P','Z') else (beta/alpha)**2)
        second=(H**4 if name in ('B','S','T','U') else -alpha**4 if name in ('E','J')
                else (alpha*beta)**2)
        expected[0,1]=first;expected[1,0]=1/first
        expected[2,3]=1/second;expected[3,2]=second
        residual=sp.Matrix(matrix-expected).applyfunc(model.scalar)
        change=sp.Matrix.hstack(*(sp.Matrix(base.coordinates(w)[0]) for w in split.generators))
        check(f'{name}.tensor_split','symbolic torus splitting',name,
              not any(residual) and abs(change.det())==1,
              {'commutation_matrix':str(matrix),'basis_change':change.tolist(),
               'pair_parameters':[str(first),str(second)],'residual':str(residual)},
              labels=('thm:monodeg','eq:twoparams'))

    for name in ('B','E','J','K','L','M','N','O','P','S','T','U','W','Z','F','I','Q','R','V'):
        model,T,units,labels=symbolic_model(name)
        identity=model.identity()
        inverse_checks=all((w*w**-1).equals(identity) and (w**-1*w).equals(identity)
                           for w in (model.e1,model.e2,model.z1,model.z2))
        check(f'{name}.Laurent_units','Laurent inverse controls',name,inverse_checks,
              labels=('prop:crossed' if name in MONOMIAL else 'lem:cross',))
        valid=True;entries=[]
        for i,j in product(range(len(units)),repeat=2):
            matches=[]
            for k,unit in enumerate(units):
                try:
                    exponent,scalar=T.coordinates(units[i]*units[j]*unit**-1)
                except ValueError:continue
                matches.append((k,exponent,str(scalar)))
            valid &= len(matches)==1
            entries.append({'left':labels[i],'right':labels[j],'result':matches})
        check(f'{name}.crossed_product_closure','crossed-product closure',name,valid,
              {'index_of_torus_lattice':abs(int(T.basis.det())),
               'coset_count':len(units),'multiplication':entries},
              labels=('prop:crossed' if name in MONOMIAL else 'lem:cross',))
        check(f'{name}.coset_index','crossed-product closure',name,
              abs(int(T.basis.det()))==len(units),labels=('prop:crossed' if name in MONOMIAL else 'lem:cross',))
        # Every entry is displayed in the generated matrix catalogue.
        actions=[T.action(w) for w in units]
        unimodular=all(abs(a['matrix'].det())==1 for a in actions)
        check(f'{name}.action_lattice_automorphisms','torus action matrices',name,unimodular,
              {'torus_generators':list(T.names),'commutations':str(T.commutations()),
               'actions':[{'coset':label,'matrix':a['matrix'].tolist(),
                           'scalars':list(map(str,a['scalars']))} for label,a in zip(labels,actions)]},
              labels=('thm:monodeg' if name in MONOMIAL else 'thm:FIQRV',))


In [11]:
# General recurrences, finite modules and weight matrices
"""Algebraic certificates for general recurrences, module bases and weights."""
from dataclasses import replace
from functools import lru_cache
import sympy as sp


def algebraic_checks(check):
    def scalar(identifier,expression,label,family='general',reducer=sp.cancel):
        residual=reducer(expression)
        check(identifier,'general algebraic certificates',family,residual==0,
              {'residual':str(residual)},labels=(label,))
    # The twist exponent is a 2-cocycle for all bidegrees, not just test degrees.
    ka,ja,kb,jb,kc,jc=sp.symbols('ka ja kb jb kc jc',integer=True)
    scalar('twist.cocycle',ja*kb+(ja+jb)*kc-jb*kc-ja*(kb+kc),'thm:twist')
    # Scalar identities that connect the independently checked normal/torus
    # relations to the PI criterion, rather than just evaluating its table.
    fam=FAMILIES['C']
    ca,cb=3*(1-p)*h**3,3*(2*p+1)*h**3
    for identifier,expression in [('alpha_p2beta',ca-p**2*cb),
                                   ('alpha_square',ca**2+p*(3*h**2)**3),
                                   ('beta_square',cb**2+(3*h**2)**3)]:
        scalar('C.criterion.'+identifier,expression,'thm:C','C',fam.scalar)
    scalar('F.criterion.c_square',(h*(1-p))**2+2*p*h**2,'thm:F','F',FAMILIES['F'].scalar)
    scalar('I.criterion.c_square',(h*(1-q))**2+2*q*h**2,'thm:I','I',FAMILIES['I'].scalar)
    scalar('EJ.criterion.alpha_square',(h*(1+sp.I))**2-2*sp.I*h**2,'tab:main','E/J')
    # The only sign correction in the L matrix changes no alternating Smith
    # invariants: it changes the sign of one entry, and a34 is zero.
    N,theta,phi,eta=sp.symbols('N theta phi eta')
    wrong=alternating((2*eta+phi,0,2*theta,N/2,2*eta+2*phi,0))
    correct=alternating((-2*eta-phi,0,2*theta,N/2,2*eta+2*phi,0))
    pf=lambda m:m[0,1]*m[2,3]-m[0,2]*m[1,3]+m[0,3]*m[1,2]
    scalar('L.corrected_pfaffian',pf(correct)-theta*N,'thm:L','L')
    scalar('L.sign_correction_same_pfaffian',pf(wrong)-pf(correct),'thm:L','L')
    k,kp=sp.symbols('k kp')
    for name,upper in [('D',(N/2,-N/2-k-kp,-k-kp,-k,k,-k)),
                       ('G',(0,-k-kp,-k-kp,-k,k,-k))]:
        scalar(name+'.general_pfaffian',pf(alternating(upper))-2*k*(k+kp),
               'prop:DGdeg',name)

    l11,l12,l21,l22,c1,c2,d1,d2,R=sp.symbols('l11 l12 l21 l22 c1 c2 d1 d2 R')
    values=(l11,l12,l21,l22)
    for kind in ('0','I','II','III'):
        sx=kind in ('I','III');sy=kind in ('I','II')
        cs=(c1,c2);ds=(d1,d2)
        changed=tuple(values[2*a+b]*ds[a]*cs[b]/(ds[1-a if sy else a]*cs[1-b if sx else b])
                      for a in range(2) for b in range(2))
        def invariant(v):
            a,b,c,d=v
            return {'0':(a,b,c,d),'I':(a*d,b*c),
                    'II':(a*c,b*d,a/b,c/d),'III':(a*b,c*d,a/c,b/d)}[kind]
        for j,(left,right) in enumerate(zip(invariant(changed),invariant(values))):
            scalar(f'rescale.{kind}.{j}',left-right,'lem:rescale')
    # Normalize type I by r^2*l11*l12=1 and u=(r*l11)^(-1).
    fam=replace(FAMILIES['S'],relations=((l11*l12*R**2-1,R),))
    rr,uu=R,1/(R*l11)
    changed=(rr*uu*l11,rr/uu*l12,uu/rr*l21,l22/(rr*uu))
    for j,(left,right) in enumerate(zip(changed,(1,1,l12*l21,l11*l22))):
        scalar(f'rescale.typeI_normalized.{j}',left-right,'lem:rescale',reducer=fam.scalar)

    # Plane bases: all coefficients are arbitrary, with invertible 2 by 2 change.
    a1,b1,a2,b2=sp.symbols('a1 b1 a2 b2')
    delta=a1*b2-a2*b1
    for sign,name in [(-1,'D'),(1,'G')]:
        A=Algebra(FAMILIES[name]);x1,x2,_,_=A.generators
        e1,e2=a1*x1+b1*x2,a2*x1+b2*x2
        u,v,aa,ap=e1**2,e2**2,e1*e2,e2*e1
        central=(u,v,aa+ap,aa*ap)
        for j,element in enumerate(central):
            for k,gen in enumerate((x1,x2)):
                residual=element.commutator(gen)
                check(f'plane.{sign}.central.{j}.{k}','plane lemma identities','general',
                      not residual,residual.certificate(),labels=('lem:plane',))
        residual=aa*ap-u*v
        check(f'plane.{sign}.product','plane lemma identities','general',not residual,
              residual.certificate(),labels=('lem:plane',))
        if sign==-1:
            for j,residual in enumerate([
                delta**2*x1**2-(b2**2*u+b1**2*v-b1*b2*(aa+ap)),
                delta**2*x2**2-(a2**2*u+a1**2*v-a1*a2*(aa+ap))]):
                check(f'plane.inverse_squared_basis.{j}','plane lemma identities','general',
                      not residual,residual.certificate(),labels=('lem:plane',))
    A0,B0,u,v=sp.symbols('A0 B0 u v')
    scalar('plane.power_sum_recurrence',u**2*A0+v**2*B0-(u+v)*(u*A0+v*B0)+u*v*(A0+B0),'lem:plane')

    # Rank-four module over k[ab,a^2+b^2], including both left and right forms.
    A=Algebra(FAMILIES['D']);a,b,_,_=A.generators
    w1,w2=a*b,a**2+b**2
    identities=[a**3-a*w2-b*w1,a**3-w2*a+w1*b,
                b**3-b*w2+a*w1,b**3-w2*b-w1*a,a**4-w2*a**2-w1**2,
                w1*w2-w2*w1,w1**2+a**2*b**2]
    for j,residual in enumerate(identities):
        check(f'rank4.identity.{j}','finite-module certificates','F/I',not residual,
              residual.certificate(),labels=('lem:rank4',))
    z=sp.Symbol('z')
    scalar('rank4.Hilbert',(1+2*z+z**2)/(1-z**2)**2-1/(1-z)**2,'lem:rank4')

    # Dickson diagonalization is generic in p,c12,c21,w. The final root-of-unity
    # step is represented by P=p^n and W=w^n, so the certificate has no n cutoff.
    c12,c21,w,PP,WW,AA=sp.symbols('c12 c21 w P W A')
    alpha=-c12*c21*p**2/(p**2-1)**2
    tt=w+alpha/w
    l1=p*w+alpha/(p*w);l2=w/p+alpha*p/w
    M=sp.Matrix([[p*tt,c12],[c21,tt/p]])
    V=sp.Matrix([[c12,c12],[l1-p*tt,l2-p*tt]])
    residual=(M*V-V*sp.diag(l1,l2)).applyfunc(sp.cancel)
    check('Dickson.generic_eigenvectors','general Dickson certificate','D/G',not any(residual),
          {'eigenvector_matrix':str(V),'determinant':str(sp.factor(V.det())),
           'residual':str(residual)},labels=('lem:dickson',))
    # The lemma itself permits c12=0; a triangular eigenbasis covers that chart.
    Mt=sp.Matrix([[p*t,0],[c21,t/p]])
    Vt=sp.Matrix([[(p-1/p)*t,0],[c21,1]])
    rt=(Mt*Vt-Vt*sp.diag(p*t,t/p)).applyfunc(sp.cancel)
    check('Dickson.triangular_eigenvectors','general Dickson certificate','general',not any(rt),
          {'c12':0,'eigenvector_matrix':str(Vt),'determinant':str(sp.factor(Vt.det())),
           'residual':str(rt)},labels=('lem:dickson',))
    scalar('Dickson.trace',l1+l2-(p+1/p)*tt,'lem:dickson')
    scalar('Dickson.determinant',l1*l2-tt**2+c12*c21,'lem:dickson')
    scalar('Dickson.all_orders_branch1',(PP*WW+AA/(PP*WW)-WW-AA/WW).subs(PP,1),'lem:dickson')
    scalar('Dickson.all_orders_branch2',(WW/PP+AA*PP/WW-WW-AA/WW).subs(PP,1),'lem:dickson')
    # The q-integer induction only needs the geometric recurrence; Q=q^k.
    qq,Q=sp.symbols('qq Q')
    scalar('geometric.kplus1',(1-qq*Q)/(1-qq)-(1-Q)/(1-qq)-Q,'lem:potY')

    # Exact weight transition identities used in lem:hw and lem:lower.
    B1,B2=sp.symbols('beta1 beta2')
    bb=-B2/(p**2-1);cc=p**2*B1/(p**2-1);aa=bb*cc
    lam=lambda v:v+aa/v
    scalar('weights.aident.1',aa*(p-1/p)**2+B1*B2,'eq:aident')
    scalar('weights.aident.2',aa*(p**2-1)-p**2*B1*bb,'eq:aident')
    scalar('weights.trace',lam(p*w)+lam(w/p)-(p+1/p)*lam(w),'lem:hw')
    scalar('weights.determinant',lam(p*w)*lam(w/p)-lam(w)**2+B1*B2,'lem:hw')
    gamma=p*bb/w
    scalar('weights.hw_gamma',(lam(w/p)-lam(w)/p)/B1-gamma,'lem:hw')
    scalar('weights.hw_Omega',gamma*lam(w/p)-cc*gamma**2-bb,'lem:hw')
    gamma2=bb/w
    scalar('weights.lower_gamma',(p*lam(w)-lam(p*w))/(p*B1)-gamma2,'lem:lower')
    scalar('weights.lower_Omega',lam(w)*gamma2-bb-cc*gamma2**2,'lem:lower')
    scalar('weights.involution',lam(aa/w)-lam(w),'lem:lower')
    P0,Q0=sp.symbols('P0 Q0')
    scalar('weights.distinct_weight_factor',lam(P0*w)-lam(Q0*w)
           -(P0-Q0)*(w-aa/(P0*Q0*w)),'prop:weights')

    # C: explicit module reduction and Hilbert-series certificate for rank nine.
    A=Algebra(FAMILIES['C']);_,_,y1,y2=A.generators
    ca,cb=y1**2*y2,y1**3-y2**3
    identities=[ca*cb-cb*ca,ca**3-y1**6*y2**3,
                ca*y1-p*y1*ca,cb*y1-y1*cb,cb*y2-y2*cb,
                y1**5-cb*y1**2-ca*y2**2]
    for j,residual in enumerate(identities):
        check(f'C.rank9.identity.{j}','C rank-nine module','C',not residual,
              residual.certificate(),labels=('prop:bounds',))
    degrees=[i+j for i,j in C_BASIS]
    scalar('C.rank9.Hilbert',sum(z**d for d in degrees)/(1-z**3)**2-1/(1-z)**2,'prop:bounds','C')
    for total in range(10):
        for i in range(total+1):
            j=total-i;reduction=c_module_reduce(i,j)
            reconstruction=A.element(0)
            for (aexp,bexp,basis),coefficient in reduction.items():
                ii,jj=C_BASIS[basis]
                reconstruction=reconstruction+coefficient*ca**aexp*cb**bexp*y1**ii*y2**jj
            residual=y1**i*y2**j-reconstruction
            check(f'C.rank9.reduce.{i}.{j}','C module reduction regressions','C',not residual,
                  residual.certificate(),labels=('prop:bounds',))


C_BASIS=((0,0),(1,0),(0,1),(2,0),(1,1),(0,2),(3,0),(1,2),(4,0))


@lru_cache(maxsize=None)
def c_module_reduce(i,j):
    """Unbounded exact left k[a,b]-module reduction for a=y1^2*y2,b=y1^3-y2^3."""
    if i<0 or j<0:raise ValueError('Nonnegative exponents required')
    if (i,j) in C_BASIS:return {(0,0,C_BASIS.index((i,j))):sp.Integer(1)}
    if i>=2 and j>=1:
        summands=[(1,0,p**(2-i),c_module_reduce(i-2,j-1))]
    elif j>=3:
        summands=[(0,0,1,c_module_reduce(i+3,j-3)),(0,1,-1,c_module_reduce(i,j-3))]
    elif i>=5:
        summands=[(0,1,1,c_module_reduce(i-3,0)),(1,0,p**(5-i),c_module_reduce(i-5,2))]
    else:raise ArithmeticError('Incomplete C module recurrence')
    result={}
    scalar=FAMILIES['C'].scalar
    for da,db,coefficient,terms in summands:
        for (a,b,k),value in terms.items():
            key=(a+da,b+db,k)
            result[key]=result.get(key,0)+coefficient*value
    return {key:scalar(value) for key,value in result.items() if scalar(value)!=0}


def right_weight_matrix(n,beta1,beta2):
    """Right t on y1^j*y2^(n-j), with coefficients on the left in k[t]."""
    if not isinstance(n,int) or n<0:raise ValueError('n must be a nonnegative integer')
    matrix=sp.zeros(n+1)
    qint=lambda k,v:sum(v**i for i in range(k))
    for j in range(n+1):
        l=n-j
        matrix[j,j]=p**(j-l)*t
        if j:matrix[j-1,j]=-p**(1-l)*beta1*qint(j,p**2)
        if l:matrix[j+1,j]=-p**(l-2)*beta2*qint(l,p**-2)
    return sp.ImmutableMatrix(matrix)


def weight_matrix_checks(check,max_degree=5):
    B1,B2=sp.symbols('beta1 beta2')
    A=ore_algebra(B1,B2);tt,y1,y2=A.generators
    for n in range(max_degree+1):
        matrix=right_weight_matrix(n,B1,B2)
        valid=True;residuals=[]
        for j in range(n+1):
            actual=y1**j*y2**(n-j)*tt
            expected=A.element(0)
            for i in range(n+1):
                coefficient=sp.Poly(matrix[i,j],t)
                for (degree,),value in coefficient.terms():
                    expected=expected+value*tt**degree*y1**i*y2**(n-i)
            residual=actual-expected
            valid &= not residual
            residuals.append(residual.certificate())
        check(f'weights.matrix.{n}','weight matrix regressions','D/G',valid,
              {'matrix':str(matrix),'residuals':residuals},labels=('prop:weights',))
    # Characteristic polynomials over k(p,a,w), no specialization of p to numbers.
    a,w,z=sp.symbols('a w z')
    for n in range(max_degree+1):
        matrix=right_weight_matrix(n,1,-a*(p-1/p)**2).subs(t,w+a/w)
        actual=matrix.charpoly(z).as_expr()
        expected=sp.prod(z-(p**nu*w+a/(p**nu*w)) for nu in range(-n,n+1,2))
        residual=sp.cancel(actual-expected)
        check(f'weights.spectrum.{n}','weight spectrum regressions','D/G',residual==0,
              {'weights':list(range(-n,n+1,2)),'residual':str(residual)},labels=('prop:weights',))


def dg_central_elements(name,order):
    """Construct the displayed central generators for any primitive order >=3, h=1."""
    from math import lcm
    if name not in ('D','G') or not isinstance(order,int) or order<3:
        raise ValueError('Use family D/G and primitive order >=3')
    family=replace(FAMILIES[name].specialize({h:1}),relations=((sp.cyclotomic_poly(order,p),p),))
    A=Algebra(family);x1,x2,y1,y2=A.generators
    if name=='G':x2=x2+x1*(1/(p-1))
    alpha=(1 if name=='D' else -f)/(p**2-1)**2
    previous,current=sp.Integer(2),t
    for j in range(2,order+1):previous,current=current,sp.expand(t*current-alpha*previous)
    ell=lcm(2,order) if name=='D' else order
    polynomial=sp.Poly(sp.expand(current**(ell//order)),t)
    phi=A.element(0)
    for (k,),coefficient in polynomial.terms():
        sign=(-1)**(k*(k-1)//2) if name=='D' else 1
        phi=phi+family.scalar(coefficient)*sign*x1**(ell-k)*x2**k
    return A,(x1**ell,y1**ell,y2**order,phi)


def dg_central_checks(check):
    for name in ('D','G'):
        for order in (3,4,5,6):
            A,elements=dg_central_elements(name,order)
            for j,element in enumerate(elements):
                for k,gen in enumerate(A.generators):
                    residual=element.commutator(gen)
                    check(f'{name}.central_generators.{order}.{j}.{k}',
                          'DG central generator regressions',name,not residual,
                          residual.certificate(),labels=('thm:DGPI','lem:potY','lem:dickson'))


In [12]:
# Central orbit and outer-action certificates
"""Computational ingredients of the primitivity arguments (not ideal-theory formalization)."""
from dataclasses import replace
from itertools import product
from math import gcd
import sympy as sp


class WordSum:
    """Finite sums in the exact Laurent word normal forms."""
    def __init__(self,model,terms=None):
        self.model=model
        self.terms={a:c for a,b in (terms or {}).items() if (c:=model.scalar(b))!=0}

    @classmethod
    def from_word(cls,word):
        a,c=word.normal()
        return cls(word.model,{a:c})

    def as_word(self,exponents,coefficient):
        word=self.model.canonical_word(exponents)
        _,factor=word.normal()
        return word*(coefficient/factor)

    def __add__(self,other):
        result=self.terms.copy()
        for a,c in other.terms.items():result[a]=result.get(a,0)+c
        return WordSum(self.model,result)

    def __neg__(self):return WordSum(self.model,{a:-c for a,c in self.terms.items()})
    def __sub__(self,other):return self+-other

    def __mul__(self,other):
        result={}
        for a,c in self.terms.items():
            for b,d in other.terms.items():
                exponent,coefficient=(self.as_word(a,c)*self.as_word(b,d)).normal()
                result[exponent]=result.get(exponent,0)+coefficient
        return WordSum(self.model,result)

    def conjugate(self,unit):
        result=WordSum(self.model)
        for a,c in self.terms.items():
            result=result+WordSum.from_word(unit.conjugate(self.as_word(a,c)))
        return result

    def nonconstant(self):
        return any(any(a) for a in self.terms)

    def certificate(self):
        return [{'exponents':list(a),'coefficient':str(c)} for a,c in sorted(self.terms.items())]


def orbit_polynomial(model,torus,central_word):
    """Product (T-g(c)) over the four crossed-product cosets; all exact."""
    e,z=model.e1,model.z1
    units=(model.identity(),e,z,e*z)
    orbit=[w.conjugate(central_word) for w in units]
    coefficients=[WordSum.from_word(model.identity())]
    for word in orbit:
        value=WordSum.from_word(word)
        next_coefficients=[WordSum(model) for _ in range(len(coefficients)+1)]
        for j,c in enumerate(coefficients):
            next_coefficients[j]=next_coefficients[j]-c*value
            next_coefficients[j+1]=next_coefficients[j+1]+c
        coefficients=next_coefficients
    witnesses=[]
    all_central=True
    for coefficient in coefficients[:-1]:
        invariant=all(not (coefficient.conjugate(w)-coefficient).terms
                      for w in torus.generators+(e,z))
        all_central &= invariant
        if coefficient.nonconstant():witnesses.append(coefficient.certificate())
    return {'central':all_central,'has_nonconstant_coefficient':bool(witnesses),
            'central_monomial':central_word.certificate(),
            'orbit':[w.certificate() for w in orbit],
            'nonconstant_coefficients':witnesses}


def fixed_family_orbit_certificate(name):
    model=monomial_model(name,True);T=standard_torus(model)
    for gen,power in product(T.generators,(1,2,4)):
        central=gen**power
        if all(central.commutation(v)==1 for v in T.generators):
            return orbit_polynomial(model,T,central)
    raise ArithmeticError('No fixed central monomial found in the stated model')


def conditional_orbit_certificate(name,which,root_order):
    """A non-PI torsion branch with the other scalar still transcendental.

    Works for any specified root order. Reduction modulo Phi_n(U) is symbolic;
    the other free phase parameter beta is not specialized to a root of unity.
    """
    if name not in ('M','N','O','P','W','Z') or which not in ('u','v'):
        raise ValueError('Use one of the six primitive monomial families and u/v')
    if not isinstance(root_order,int) or root_order<1:raise ValueError('Positive root order required')
    if which=='u' and name!='N' and root_order<=2:
        raise ValueError('u=+/-1 is outside the admissible nondegenerate chart')
    U=sp.Symbol('U')
    old=monomial_model(name,True)
    substitution={alpha:(-U*beta if name=='Z' else U*beta)} if which=='u' else {alpha:U/beta}
    family=replace(FAMILIES[name],relations=((sp.cyclotomic_poly(root_order,U),U),))
    mixing={key:(a,b,c.subs(substitution,simultaneous=True))
            for key,(a,b,c) in old.mixing.items()}
    model=WordModel(family.scalar,old.x_kind,old.y_kind,old.x_q,old.y_q,mixing,name)
    T=standard_torus(model);r,rho,X,Y=T.generators
    m=root_order//gcd(root_order,2)
    central=r**m if which=='u' else (X*r if name in ('N','W') else X)**m
    if not all(central.commutation(v)==1 for v in T.generators):
        raise ArithmeticError('The conditional central monomial failed')
    result=orbit_polynomial(model,T,central)
    result.update(family=name,root_parameter=which,root_order=root_order,
                  remaining_parameter='beta is symbolic and unconstrained')
    return result


def dg_mixed_center_certificate(name,torsion,root_order):
    if name not in ('D','G') or torsion not in ('p','q') or root_order<1:
        raise ValueError('Use family D/G, torsion p/q, and a positive root order')
    if torsion=='p' and root_order<=2:
        raise ValueError('The D/G birational chart excludes p=+/-1')
    n=root_order;P,Q=sp.symbols('P Q')
    root=P if torsion=='p' else Q
    family=replace(FAMILIES['S'],relations=((sp.cyclotomic_poly(n,root),root),))
    if torsion=='q':
        a=n if name=='G' else 2*n
        vectors=[(a,0,0,0),(0,n,n,n)]
    else:vectors=[(0,2*n,0,0),(0,0,2*n,-2*n)]
    valid=True
    for a,b,c,d in vectors:
        signs=((-1)**(b+c),(-1)**a,(-1)**a,1) if name=='D' else (1,1,1,1)
        values=(signs[0]*Q**(-c-d),signs[1]*P**(d-c),
                signs[2]*Q**a*P**(b-d),Q**a*P**(c-b))
        valid &= all(family.scalar(v-1)==0 for v in values)
    return {'torsion_parameter':torsion,'symbolic_other_parameter':'Q' if torsion=='p' else 'P',
            'vectors':vectors,'root_order':n,'central':valid,
            'rank':sp.Matrix.hstack(*(sp.Matrix(v) for v in vectors)).rank()}


def primitivity_checks(check):
    for name in ('B','E','J','K','L','S','T','U'):
        result=fixed_family_orbit_certificate(name)
        check(f'{name}.central_orbit','nonprimitive orbit invariants',name,
              result['central'] and result['has_nonconstant_coefficient'],result,
              labels=('thm:primmono',))
    for name in ('M','N','O','P','W','Z'):
        model=monomial_model(name,True);T=standard_torus(model)
        u=-alpha/beta if name=='Z' else alpha/beta
        units=(model.e1,model.z1,model.e1*model.z1)
        witnesses=[];valid=True
        for unit in units:
            action=T.action(unit)
            if action['matrix']!=sp.eye(4):
                witnesses.append({'type':'nonidentity exponent matrix',
                                  'matrix':action['matrix'].tolist()})
                continue
            found=None
            for index in (0,1):
                scalar=action['scalars'][index]
                for odd in (-1,1):
                    sign=model.scalar(scalar/u**odd)
                    if sign not in (1,-1):continue
                    row=[T.commutations()[index,j] for j in range(4)]
                    even=[]
                    for value in row:
                        exponents=[k for k in (-4,-2,0,2,4) if model.scalar(value-u**k)==0]
                        if not exponents:break
                        even.append(exponents[0])
                    if len(even)==4:
                        found={'type':'odd/even exponent obstruction','generator':T.names[index],
                               'outer_scalar':str(scalar),'sign':int(sign),'odd_exponent':odd,
                               'inner_commutator_exponents':even,
                               'consequence_if_inner':'u^(2j-odd)=+/-1, forcing u torsion'}
                        break
                if found:break
            valid &= found is not None
            witnesses.append(found)
        check(f'{name}.outer_actions','primitive localization obstructions',name,valid,
              witnesses,labels=('thm:primmono',))
        # The general orbit construction is recorded above. These extra runs
        # certify the mixed PI/non-PI branches with one parameter still free.
        for which,n in [('u',3),('u',4),('v',1),('v',3)]:
            result=conditional_orbit_certificate(name,which,n)
            check(f'{name}.conditional_orbit.{which}.{n}','conditional orbit regressions',name,
                  result['central'] and result['has_nonconstant_coefficient'],result,
                  labels=('thm:primmono',))

    # The center equations of D/G: elimination in the non-torsion case leaves
    # the following full-rank linear system on (a,b,c,d).
    matrix=sp.Matrix([[0,0,-1,1],[0,0,1,1],[0,2,-1,-1],[1,0,0,0]])
    check('DG.center_nontorsion_elimination','DG center lattice certificates','D/G',
          matrix.rank()==4,{'matrix':matrix.tolist(),'determinant':int(matrix.det()),
                           'hypotheses':'neither p nor q=hp is torsion'},labels=('thm:DG',))
    # For the mixed branches, exhibit two independent central exponent vectors
    # and verify the scalar equations in a cyclotomic coefficient field.
    for name in ('D','G'):
        for torsion in ('p','q'):
            certificate=dg_mixed_center_certificate(name,torsion,5)
            check(f'{name}.mixed_center.{torsion}','DG center lattice certificates',name,
                  certificate['central'] and certificate['rank']==2,certificate,labels=('thm:DG',))


In [13]:
# Unbounded spectral, integer ideal and parity certificates
"""Finite algebraic proof objects for unbounded induction and root orders.

A certificate retains both sides of an identity and its hypotheses. Arithmetic
checking is distinct from the explicitly stated mathematical inference rules.
"""
import sympy as sp


def uniform_bundle():
    identities=[]; ideals=[]; lattices=[]; derivations=[]
    def identity(identifier,lhs,rhs,labels):
        identities.append({'id':identifier,'lhs':str(lhs),'rhs':str(rhs),'labels':list(labels)})
    def matrix_id(identifier,lhs,rhs,labels):
        lhs,rhs=sp.Matrix(lhs),sp.Matrix(rhs)
        for i in range(lhs.rows):
            for j in range(lhs.cols):identity(f'{identifier}.{i}.{j}',lhs[i,j],rhs[i,j],labels)
    def ideal(identifier,source,target,forward,backward,labels):
        ideals.append({'id':identifier,'source':list(map(str,source)),'target':list(map(str,target)),
                       'forward':forward,'backward':backward,'labels':list(labels)})
    # Weight induction: eta=p^nu*w is left as an independent transcendent.
    p,b,c,eta,P,Q,w,n,j,k=sp.symbols('p b c eta P0 Q0 w n j k')
    beta1=c*(p**2-1)/p**2;beta2=-b*(p**2-1);a=b*c
    lam=lambda x:x+a/x
    mu=lam(eta);lp,lm=lam(p*eta),lam(eta/p)
    B=sp.Matrix([[p*mu,-p*beta1],[-beta2/p,mu/p]])
    plus=(B-lm*sp.eye(2))/(lp-lm);minus=(lp*sp.eye(2)-B)/(lp-lm)
    matrix_id('weights.projectors_sum',plus+minus,sp.eye(2),('prop:weights','lem:lower'))
    matrix_id('weights.projector_plus',plus*plus,plus,('prop:weights',))
    matrix_id('weights.projector_minus',minus*minus,minus,('prop:weights',))
    matrix_id('weights.projectors_orthogonal',plus*minus,sp.zeros(2),('prop:weights',))
    matrix_id('weights.spectral_decomposition',lp*plus+lm*minus,B,('prop:weights',))
    identity('weights.eigenvalue_gap',lp-lm,(p-1/p)*(eta-a/eta),('lem:lower',))
    identity('weights.no_missing_lower',(p*mu-lp)/(p*beta1),b/eta,('lem:lower',))
    identity('weights.Omega_obstruction',mu*b/eta-b-c*(b/eta)**2,0,('lem:lower',))
    identity('weights.branch_involution',lam(a/eta/p),lp,('lem:lower',))
    identity('weights.distinct',lam(P*w)-lam(Q*w),(P-Q)*(w-a/(P*Q*w)),('prop:weights',))
    for step,index in [(-1,j),(1,j+1)]:
        identity('weights.index.'+str(step),-n+2*j+step,-(n+1)+2*index,('prop:weights',))
    identity('weights.index.left_endpoint',-n-1,-(n+1),('prop:weights',))
    identity('weights.index.right_endpoint',n+1,-(n+1)+2*(n+1),('prop:weights',))
    identity('weights.dimension',(n+1)+1,n+2,('prop:weights',))
    identity('weights.lowering.positive',(n+1)+(k-1),n+k,('prop:weights',))
    identity('weights.zero_degree_parity',(-2*k+2*k),0,('prop:weights',))
    induction={'id':'weights.all_n','labels':['prop:weights','thm:DGprim'],
       'base':{'degree':0,'weight_indices':[0],'dimension':1},
       'index_maps':['j','j+1'],'intervals':[[0,'n'],[1,'n+1']],
       'union':[0,'n+1'],'eigenvalue':'lambda(p**(-n+2*j)*w)',
       'assumptions':['characteristic zero','p not a root of unity','b*c != 0',
                      'w transcendental','PBW gives dim(V_n)=n+1',
                      'V_(n+1)=V_n*y1+V_n*y2','Omega central and nonzero',
                      'right multiplication by Omega injective by domain and flatness'],
       'inference_rules':['two complementary polynomial spectral projectors',
                          'nonvanishing of both branches from Omega obstruction',
                          'affine interval union induction','dimension counts distinct eigenlines'],
       'ideal_consequence':'lower |nu| to zero; clear denominators; use noetherian prime-product argument',
       'ideal_theory_dependency':'the final prime-product and torsion arguments are those in the manuscript'}
    M,theta,phi,eta,k,kp,N=sp.symbols('M theta phi eta k kp N')
    pf=lambda A:A[0,1]*A[2,3]-A[0,2]*A[1,3]+A[0,3]*A[1,2]
    for name,upper,target,fw,bw in [
      ('D',(M,-M-k-kp,-k-kp,-k,k,-k),(M,k,kp),
       [[1,0,0,0,0,0],[0,0,0,-1,0,0],[0,0,-1,1,0,0]],
       [[1,0,0],[-1,-1,-1],[0,-1,-1],[0,-1,0],[0,1,0],[0,-1,0]]),
      ('G',(0,-k-kp,-k-kp,-k,k,-k),(k,kp),
       [[0,0,0,-1,0,0],[0,0,-1,1,0,0]],
       [[0,0],[-1,-1],[-1,-1],[-1,0],[1,0],[-1,0]]),
      ('L',(-2*eta-phi,0,2*theta,M,2*eta+2*phi,0),(2*eta,phi,2*theta,M),
       [[-2,0,0,0,-1,0],[1,0,0,0,1,0],[0,0,1,0,0,0],[0,0,0,1,0,0]],
       [[-1,-1,0,0],[0,0,0,0],[0,0,1,0],[0,0,0,1],[1,2,0,0],[0,0,0,0]]),
      ('K',(M,eta,2*eta+phi,eta,2*eta+phi,-2*theta),(M,eta,phi,2*theta),
       [[1,0,0,0,0,0],[0,1,0,0,0,0],[0,-2,1,0,0,0],[0,0,0,0,0,-1]],
       [[1,0,0,0],[0,1,0,0],[0,2,1,0],[0,1,0,0],[0,2,1,0],[0,0,0,1*-1]])]:
        label='prop:DGdeg' if name in ('D','G') else 'thm:L' if name=='L' else 'thm:monodeg'
        A=alternating(upper)
        expected=2*k*(k+kp) if name in ('D','G') else 2*M*theta if name=='L' else -2*M*theta
        identity(name+'.pfaffian',pf(A),expected,(label,))
        ideal(name+'.entry_ideal',upper,target,fw,bw,(label,))
        derivations.append({'family':name,'label':label,'root_order':'N=2*M' if name!='G' else 'N',
           'generation_gcd':'gcd(M,k,kp)=1' if name=='D' else 'gcd(N,k,kp)=1' if name=='G' else 'gcd(M,theta,phi,eta)=1',
           'alternating_smith_rule':'s1=gcd(entries), s1*s2=abs(Pfaffian); torus degree=prod(N/gcd(N,si))',
           'conclusion':'N**2/gcd(N,2*k*(k+kp))' if name in ('D','G') else '2*N' if name=='K' else '2*N if N/N0 odd, N otherwise',
           'crossed_center_factor':1 if name in ('D','G') else 2})
    # L: d=gcd(theta,M,phi,2*eta) divides 2 by Bezout in the base ideal.
    ideal('L.double_base_in_target',(theta,M,phi,2*eta),(2*theta,2*M,2*phi,2*eta),
          [[2,0,0,0],[0,2,0,0],[0,0,2,0],[0,0,0,1]],
          # This is an inclusion, not equivalence: backward deliberately omitted.
          None,('thm:L',))
    identity('L.gcd_product_scaling',2*M,(2*M),('thm:L',))
    # Central congruence systems and complete integer parametrizations for all m.
    m=sp.Symbol('m');u,ss,rr,tt=sp.symbols('u ss rr tt')
    specs=[('Q',[[2,0,0,0],[-2,m,0,0],[2,0,m,2],[2,0,m,0]],
             ['a','c-d','a-d','a+b'],[2,2,'m','m']),
           ('V',[[-2,m,0,0],[2,0,0,0],[-2,0,m,2],[-2,0,m,0]],
             ['b','c-d','b+d','a+b'],[2,2,'m','m']),
           ('R',[[1,0,0,0],[0,m,0,0],[0,0,m,-2],[0,0,0,1]],
             ['a','b','c+2*d','d'],[1,'m','m',1])]
    for name,G,forms,divs in specs:
        G=sp.Matrix(G)
        lattices.append({'family':name,'basis':[[str(x) for x in row] for row in G.tolist()],
                         'forms':forms,'divisors':divs,'determinant':str(4*m**2 if name in ('Q','V') else m**2),
                         'labels':['thm:FIQRV', 'thm:QVdeg' if name in ('Q','V') else 'thm:Rdeg'],
                         'meaning':'m=ord(h^2)>0; columns parametrize all integral solutions'})
        identity(name+'.central_index',G.det()**2,(4*m**2 if name in ('Q','V') else m**2)**2,('thm:FIQRV',))
    # The finite list below is a proof by exhaustive parity classes, not by orders.
    parity={'classes':[
        {'order_form':'2*r+1','m_form':'2*r+1','m_parity':1,'root_to_m':1,'name':'odd'},
        {'order_form':'4*r+2','m_form':'2*r+1','m_parity':1,'root_to_m':-1,'name':'two_mod_four'},
        {'order_form':'4*r','m_form':'2*r','m_parity':0,'root_to_m':-1,'name':'zero_mod_four'}],
        'assumption':'root has exact positive order d; m=ord(root^2)',
        'elementary_rule':'ord(root^2)=d/gcd(d,2); exact order 2 gives -1 in characteristic zero',
        'families':['M','N','O','P','W','Z']}
    # Save real words, not their normalized forms, for a separate word checker.
    models=[]
    for name in ('B','E','J','K','L','M','N','O','P','S','T','U','W','Z','F','I','Q','R','V'):
        model,T,units,labels=symbolic_model(name)
        if name in ('B','E','J','M','N','O','P','S','T','U','W','Z'):
            r,rho,X,Y=T.generators
            xp=X*rho if name in ('B','J') else X*r if name in ('N','U','W') else X
            yp=Y*r if name in ('B','E') else Y*rho if name in ('P','T','Z') else Y
            T=TorusWords(model,(r,rho,xp,yp),('r','rho','Xp','Yp'))
        encode=lambda word:{'letters':[list(v) for v in word.letters],'coefficient':str(word.coefficient)}
        actions=[T.action(unit) for unit in units]
        models.append({'family':name,'planes':[model.x_kind,model.y_kind],
           'plane_q':[str(model.x_q),str(model.y_q)],
           'relations':[[list(ab),[bb,aa,str(cc)]] for ab,(bb,aa,cc) in model.mixing.items()],
           'scalar_relations':[[str(poly),str(v)] for poly,v in model.family.relations] if hasattr(model,'family') else [],
           'generators':[encode(v) for v in T.generators],'units':[encode(v) for v in units],
           'unit_labels':list(labels),'commutations':[[str(v) for v in row] for row in T.commutations().tolist()],
           'actions':[{'matrix':[[int(v) for v in row] for row in a['matrix'].tolist()],'scalars':list(map(str,a['scalars']))} for a in actions]})
    # A second K torus of index two gives a short uniform Smith proof.
    km=WordModel(sp.cancel,'coordinate','coordinate',q,-1,
        {(0,0):(0,0,H),(0,1):(1,1,H),(1,0):(0,1,H),(1,1):(1,0,H*F)},'K_index_two')
    KT=TorusWords(km,(km.z1,km.z2,km.e1,km.e2**2),('y1','y2','x1','x2_squared'))
    ku=(km.identity(),km.e2)
    ka=[KT.action(v) for v in ku]
    models.append({'family':'K_index_two','planes':['coordinate','coordinate'],
       'plane_q':[str(q),'-1'],'relations':[[list(ab),[bb,aa,str(cc)]] for ab,(bb,aa,cc) in km.mixing.items()],
       'scalar_relations':[],'generators':[encode(v) for v in KT.generators],
       'units':[encode(v) for v in ku],'unit_labels':['0','1'],
       'commutations':[[str(v) for v in row] for row in KT.commutations().tolist()],
       'actions':[{'matrix':[[int(v) for v in row] for row in a['matrix'].tolist()],
                   'scalars':list(map(str,a['scalars']))} for a in ka],
       'source_coordinates':True,'substitutions':{'H':'h','F':'f'},
       'expected_upper':['-1','H','F*H**2','H','F*H**2','q**(-2)']})
    lm=WordModel(sp.cancel,'coordinate','coordinate',q,-1,
        {(0,0):(0,1,H*F),(0,1):(1,1,H),(1,0):(0,0,H*F),(1,1):(1,0,H)},'L_index_two')
    LT=TorusWords(lm,(lm.e1*lm.e2,lm.z1,lm.z1**-1*lm.z2,lm.e1**2),('w','y1','rho','u'))
    lu=(lm.identity(),lm.e1);la=[LT.action(v) for v in lu]
    models.append({'family':'L_index_two','planes':['coordinate','coordinate'],
       'plane_q':[str(q),'-1'],'relations':[[list(ab),[bb,aa,str(cc)]] for ab,(bb,aa,cc) in lm.mixing.items()],
       'scalar_relations':[],'generators':[encode(v) for v in LT.generators],
       'units':[encode(v) for v in lu],'unit_labels':['0','1'],
       'commutations':[[str(v) for v in row] for row in LT.commutations().tolist()],
       'actions':[{'matrix':[[int(v) for v in row] for row in a['matrix'].tolist()],
                   'scalars':list(map(str,a['scalars']))} for a in la],
       'source_coordinates':True,'substitutions':{'H':'h','F':'f'},
       'expected_upper':['1/(F*H**2)','1','q**2','-1','F**2*H**2','1']})
    return {'schema':'ore14641.uniform.v1','identities':identities,'integer_ideals':ideals,
            'center_parametrizations':lattices,'weight_induction':induction,'root_parity':parity,
            'degree_derivations':derivations,'word_models':models,
            'theory_dependencies':['PBW independence and domain','alternating integer Smith theorem',
                'center of a quantum torus','finite crossed-product center/degree theorem',
                'Bezout identity for gcd and cyclic group exact order',
                'flatness and the ideal-theoretic arguments listed in the manuscript']}


def uniform_checks(check):
    data=uniform_bundle()
    for claim in data['identities']:
        result=sp.cancel(sp.sympify(claim['lhs'])-sp.sympify(claim['rhs']))
        check('uniform.'+claim['id'],'unbounded algebraic proof objects','general',result==0,
              claim,labels=claim['labels'])
    for claim in data['integer_ideals']:
        source=sp.Matrix(list(map(sp.sympify,claim['source'])))
        target=sp.Matrix(list(map(sp.sympify,claim['target'])))
        valid=sp.Matrix(claim['forward'])*source==target
        if claim['backward'] is not None:valid &= sp.Matrix(claim['backward'])*target==source
        check('uniform.'+claim['id'],'unbounded integer ideal certificates','general',valid,
              claim,labels=claim['labels'])


In [14]:
# Claim-level hypotheses and certificate contracts
"""Claim-level coverage, with hypotheses, inputs, certificate IDs and dependencies."""

# Labels sharing an algebraic claim point to the same exact data contract.
GROUPS = [
(('sec:prelim',),'All admissible source parameters; coefficient field characteristic zero.',
 'P,Q,Sigma,M and all critical cubic ambiguities agree with the fixed presentation.',
 ['source matrices','PBW overlaps'],['termination of the degree-lex rewrite order']),
(('tab:det',),'All admissible parameters; determinant composition convention in CONVENTIONS.md.',
 'The determinant automorphism has the displayed two by two matrix and preserves the base-plane relation.',
 ['det sigma table'],['definition of the double-extension determinant']),
(('tab:mono','lem:rescale','lem:plane','lem:A1','eq:A2','eq:A3','lem:A2','lem:A3'),
 'Fourteen monomial families; square-root charts require s^2=f and s!=0.',
 'Invertible basis changes, plane charts, signed inverse rules and rescaling invariants hold exactly.',
 ['monomial basis changes','plane charts','Laurent identities','plane lemma identities','general algebraic certificates'],
 ['PBW independence; no use of a square-root chart at f=0']),
(('prop:noPI',),'A,H,X,Y or the explicitly admissible degenerate specializations.',
 'Affine conjugation matrices and degenerate defining identities used in the non-PI obstruction.',
 ['degenerate presentations','non-PI affine matrices','additional PBW identities'],
 ['characteristic-zero affine conjugation obstruction and Jordan-plane theorem']),
(('eq:MDG','rem:DGnormal','lem:Omega','eq:Arel','eq:aident'),
 'D with p^2!=1, or G with p!=0,+1,-1 and f!=0.',
 'Shifted relations, Omega centrality, Xi normality, and the birational coefficient identities.',
 ['normalization of G','additional PBW identities','normal-generator matrices','general algebraic certificates'],
 ['denominators are nonzero on this parameter chart; PBW independence']),
(('lem:potY','lem:dickson','thm:DGPI'),
 'p^2!=1; for the PI application p has exact finite order and h is torsion.',
 'General eigenvector and geometric recurrences, Dickson root-power identity, and central-generator construction.',
 ['general Dickson certificate','general algebraic certificates','DG central generator regressions','Dickson samples n=3..8'],
 ['Dickson defining identity and induction; finite central-module criterion']),
(('lem:Fnormal','eq:Inormal','eq:Cnormal','thm:F','thm:I','thm:C'),
 'F:p^2=-1; I:q^2=-1; C:p^2+p+1=0; h!=0.',
 'Displayed normal elements have the required mixed scalar relations, including the exact factors of h.',
 ['normal elements','additional PBW identities','general algebraic certificates'],
 ['finite module/quantum plane PI criteria; PBW algebraic independence']),
(('prop:QV','lem:Theta','eq:Qmodel','eq:Vmodel','lem:Rnorm','eq:Rz','lem:Fnorm','eq:Fg','lem:Inorm','eq:Iw'),
 'The relevant Q,V,R,F,I source presentation and all its admissible parameters.',
 'Polynomial normalizers, their squares and mixed relations justify the Laurent model.',
 ['additional PBW identities','central elements at h=1','rational coordinate matrices','anti-isomorphism identities'],
 ['normalizer independence and generation in the division ring']),
(('prop:crossed','lem:cross','thm:monodeg','thm:FIQRV','eq:Ltorus','thm:L','prop:DGdeg','tab:degrees'),
 'A PI specialization; phases are exact integers in a common cyclic group; generated subgroup normalized where needed.',
 'Torus relations, coset closure, center lattices, action images and the 21 closed degree formulas.',
 ['crossed-product closure','torus action matrices','symbolic torus splitting','all-family degree certificates',
  'unbounded algebraic proof objects','unbounded integer ideal certificates'],
 ['center of a quantum torus; alternating Smith theorem; crossed-product degree theorem; docs/UNBOUNDED_PROOFS.md']),
(('prop:bounds','lem:rank4'),
 'C with p of exact order three, or the anti-commuting plane of F/I.',
 'Finite-module reduction identities, Hilbert-series rank, and exact degree-index constraints.',
 ['finite-module certificates','C rank-nine module','C module reduction regressions','C bound certificates'],
 ['PBW dimension comparison; finite division-ring dimension argument; C exact degree remains open']),
(('thm:primmono','prop:Bnotprim','prop:CFIQRV','prop:HXprim','thm:DG','eq:twoparams','lem:loc'),
 'Relevant admissible families; stated torsion/non-torsion hypotheses are retained for each branch.',
 'Central rational/orbit witnesses, localization normality and non-inner-action obstructions.',
 ['central rational certificates','central power certificates','nonprimitive orbit invariants',
  'primitive localization obstructions','conditional orbit regressions','DG center lattice certificates',
  'localization normality','symbolic torus splitting'],
 ['simplicity, localization of prime ideals, rationality and Nullstellensatz arguments in the manuscript']),
(('lem:hw','lem:lower','prop:weights'),
 'p not a root of unity; b*c!=0; w transcendental; Omega regular after flat extension.',
 'Two spectral projectors, nonvanishing branch obstruction and affine interval induction for every n>=0.',
 ['unbounded algebraic proof objects','general algebraic certificates','weight matrix regressions','weight spectrum regressions'],
 ['PBW dimension n+1; domain and flatness; ideal-theoretic consequence separately stated']),
(('thm:twist',),'Fixed mixed matrix and twist h; root-of-unity multiplier zeta.',
 'Bidegree two-cocycle and exact scalar identities supporting the finite-order twist argument.',
 ['general algebraic certificates','additional PBW identities'],
 ['finite Veronese/finite central extension argument in the manuscript']),
(('tab:main',),'All source admissibility conditions; exact algebraic inputs for instantiated examples.',
 'Exact scalar reductions and PI-table instantiations; proofs use the explicitly linked structural ingredients.',
 ['PI table examples','general algebraic certificates'],
 ['the monomial, normalizer, non-PI and C/F/I/D/G arguments; table evaluation alone is not an independent PI proof'])]

CONTRACTS={label:{'hypotheses':hyp,'checked_claim':claim,'certificate_categories':cats,
                  'mathematical_dependencies':deps}
           for labels,hyp,claim,cats,deps in GROUPS for label in labels}


def claim_contract(label,checks):
    selected=[c for c in checks if label in c.get('manuscript_labels',())]
    value=dict(CONTRACTS.get(label,{}))
    if not value:return None
    value['certificate_ids']=[c['id'] for c in selected]
    value['families_actually_checked']=sorted({c['family'] for c in selected})
    value['certificate_locations']=['reports/verification.json#checks','reports/certificates.json','reports/independent_verification.json']
    return value


def validate_contracts(checks,required):
    missing=[label for label in required if label not in CONTRACTS]
    if missing:raise ArithmeticError('Missing claim contracts: '+', '.join(missing))
    ids={c['id'] for c in checks}
    # Merely assigning one check a section label cannot certify source coverage.
    expected={f'{name}.{suffix}' for name in FAMILIES for suffix in ('M','detSigma','detsigma','det_preserves_plane')}
    missing=expected-ids
    if missing:raise ArithmeticError('Missing source obligations: '+', '.join(sorted(missing)))
    for name in FAMILIES:
        overlaps=[c for c in checks if c['category']=='PBW overlaps' and c['family']==name]
        if len(overlaps)!=4:raise ArithmeticError('Four distinct PBW overlaps required for '+name)
    for name in ('B','E','J','K','L','M','N','O','P','S','T','U','W','Z','F','I','Q','R','V','D','G'):
        if not any(c['category']=='all-family degree certificates' and c['family']==name for c in checks):
            raise ArithmeticError('Missing degree model: '+name)
    return True


In [15]:
# Manuscript coverage ledger and proof dependencies
"""Traceability to every label in the supplied manuscript snapshot.

The word 'covered' denotes verified computational ingredients, not a formal
proof of the abstract theorem. Open questions remain explicitly open.
"""
from collections import Counter

REQUIRED_COMPUTATIONS = (
    'sec:prelim','tab:det','tab:mono','tab:degrees','prop:noPI','eq:MDG',
    'lem:Fnormal','eq:Inormal','eq:Cnormal','prop:QV','lem:rescale','lem:plane',
    'lem:A1','eq:A2','eq:A3','lem:A2','lem:A3','prop:crossed','eq:twoparams',
    'thm:monodeg','lem:Theta','eq:Qmodel','eq:Vmodel','lem:Rnorm','eq:Rz',
    'lem:Fnorm','eq:Fg','lem:Inorm','eq:Iw','lem:cross','thm:FIQRV','eq:Ltorus',
    'thm:L','prop:DGdeg','prop:bounds','rem:DGnormal','lem:Omega','eq:Arel',
    'lem:hw','lem:lower','eq:aident','lem:loc','thm:primmono','prop:Bnotprim',
    'prop:CFIQRV','prop:HXprim','thm:DG','thm:twist','thm:C','thm:F','thm:I',
    'tab:main','lem:potY','lem:dickson','lem:rank4','thm:DGPI','prop:weights'
)

DEPENDENCIES = {
    'eq:A1':['lem:A1'],
    'eq:Theta':['lem:Theta'],
    'thm:A':['tab:main','tab:mono','thm:DGPI','thm:F','thm:I','thm:C','prop:QV','prop:noPI'],
    'thm:B':['thm:twist'],
    'thm:mono':['lem:rescale','lem:plane','lem:A1','lem:A3','tab:mono'],
    'cor:mono':['tab:mono'],
    'thm:FIdeg':['lem:Fnorm','lem:Inorm','thm:FIQRV'],
    'thm:QVdeg':['lem:Theta','eq:Qmodel','eq:Vmodel','thm:FIQRV'],
    'thm:Rdeg':['lem:Rnorm','eq:Rz','thm:FIQRV'],
    'cor:det':['tab:det','tab:main'],
    'eq:Cindex':['prop:bounds'],
    'prop:weights':['lem:hw','lem:lower','lem:Omega'],
    'lem:DGmodel':['rem:DGnormal','lem:Omega','eq:MDG'],
    'eq:Dtorus':['rem:DGnormal','lem:Omega','prop:DGdeg'],
    'eq:Gtorus':['rem:DGnormal','lem:Omega','prop:DGdeg'],
    'thm:DGprim':['rem:DGnormal','lem:Omega','prop:weights','lem:lower','thm:DG'],
    'cor:tfinite':['lem:hw'],
    'eq:Bx':['rem:DGnormal','lem:Omega'],
    'eq:Fdec':['lem:rank4','lem:Fnormal'],
    'eq:Fnormal':['lem:Fnormal'],
    'eq:mono':['tab:mono'],
    'eq:Mt':['eq:MDG','prop:QV'],
    'eq:twistscalar':['thm:twist'],
    'sec:normal':['eq:MDG','prop:QV','thm:L'],
    'sec:CFI':['lem:Fnormal','eq:Inormal','eq:Cnormal'],
    'sec:monodeg':['thm:monodeg','prop:crossed'],
    'sec:norm':['thm:FIQRV'],
    'sec:degrees':['prop:bounds','tab:degrees'],
    'sec:det':['tab:det'],
    'sec:noPI':['prop:noPI'],
    'sec:mono':['tab:mono','lem:rescale','lem:plane'],
    'sec:primitive':['thm:primmono','thm:DGprim','prop:CFIQRV'],
    'subsec:HXtorsion':['prop:HXprim'],
    'def:mono':['tab:mono'],
    'rem:monodeg':['thm:monodeg'],
    'rem:bounds':['thm:FIQRV','lem:rank4'],
    'rem:detsigma':['tab:main','tab:det'],
}

THEORETICAL = {
    'lem:fracind':'Clearing Ore denominators and the PBW tensor-product independence argument.',
    'lem:qplane':'Embedding of a quantum plane and its standard PI criterion.',
    'lem:finite':'Posner and a faithful left-regular representation on a finite-dimensional division-ring space.',
    'lem:shift':'Minimal-polynomial argument for affine conjugation on a finite central division algebra.',
    'lem:shiftmat':'Eigenvector reduction to the affine-conjugation lemma; its case matrices are checked.',
    'prop:jordan':'Jordan-plane inclusion for A/H and its characteristic-zero non-PI property.',
    'lem:words':'Dimension/independence from the original PBW basis and invertible normalizers.',
    'lem:centerQT':'The center of the division ring of a quantum torus; implemented lattices use this theorem.',
    'prop:notprimitive':'A primitive affine PI domain over an algebraically closed field cannot have GK dimension four.',
    'prop:rational':'Dixmier--Moeglin rationality implication using the stated Nullstellensatz hypotheses.',
    'rem:LM':'Bibliographic comparison; no separate computational assertion.',
}


def coverage_report(checks):
    validate_contracts(checks,REQUIRED_COMPUTATIONS)
    counts=Counter(label for check in checks for label in check.get('manuscript_labels',()))
    missing=[label for label in REQUIRED_COMPUTATIONS if counts[label]==0]
    if missing:raise ArithmeticError('Missing computational blocks: '+', '.join(missing))
    entries=[]
    for label in SOURCE_LABELS:
        kind=RESULT_KINDS.get(label,'equation/table/section')
        deps=DEPENDENCIES.get(label,[])
        support={name:counts[name] for name in deps if counts[name]}
        if kind=='question':status='open in manuscript; intentionally not assigned a computed answer'
        elif counts[label]:status='computational ingredients checked'
        elif label in THEORETICAL:status='mathematical/theoretical dependency; no machine-proof claim'
        elif support:status='uses checked ingredients; remaining reasoning is mathematical'
        else:status='contextual/bibliographic label; inspect source rather than infer a numerical claim'
        entries.append({'label':label,'kind':kind,'status':status,'direct_checks':counts[label],
                        'supporting_checks':support,'theoretical_dependency':THEORETICAL.get(label),'claim_contract':claim_contract(label,checks)})
    return {'manuscript_label_count':len(SOURCE_LABELS),
            'required_computational_blocks':len(REQUIRED_COMPUTATIONS),
            'missing_required_blocks':missing,'all_21_closed_degree_models_implemented':True,
            'formal_proof_assistant_verification':False,'entries':entries}

SOURCE_LABELS = ['thm:A', 'tab:main', 'thm:B', 'sec:prelim', 'lem:fracind', 'lem:qplane', 'lem:finite', 'lem:shift', 'lem:shiftmat', 'eq:twistscalar', 'thm:twist', 'sec:noPI', 'prop:jordan', 'prop:noPI', 'rem:LM', 'sec:mono', 'def:mono', 'eq:mono', 'lem:rescale', 'lem:plane', 'thm:mono', 'cor:mono', 'tab:mono', 'eq:EJ', 'sec:normal', 'eq:Mt', 'eq:MDG', 'lem:potY', 'lem:dickson', 'thm:DGPI', 'prop:QV', 'thm:L', 'eq:Ltorus', 'sec:CFI', 'lem:rank4', 'lem:Fnormal', 'eq:Fnormal', 'thm:F', 'eq:Fdec', 'thm:I', 'eq:Inormal', 'thm:C', 'eq:Cnormal', 'sec:monodeg', 'prop:crossed', 'lem:A1', 'eq:A1', 'eq:A2', 'eq:A3', 'lem:A2', 'lem:A3', 'thm:monodeg', 'rem:monodeg', 'sec:norm', 'lem:cross', 'lem:words', 'eq:Theta', 'lem:Theta', 'eq:Qmodel', 'eq:Vmodel', 'thm:QVdeg', 'eq:Rz', 'lem:Rnorm', 'thm:Rdeg', 'eq:Fg', 'lem:Fnorm', 'eq:Iw', 'lem:Inorm', 'thm:FIdeg', 'thm:FIQRV', 'rem:bounds', 'sec:degrees', 'prop:bounds', 'eq:Cindex', 'tab:degrees', 'q:degrees', 'sec:det', 'tab:det', 'cor:det', 'rem:detsigma', 'sec:primitive', 'prop:notprimitive', 'prop:rational', 'prop:Bnotprim', 'eq:Bswap', 'prop:CFIQRV', 'eq:Dtorus', 'eq:Gtorus', 'lem:centerQT', 'thm:DG', 'eq:Arel', 'eq:aident', 'eq:Bx', 'lem:Omega', 'lem:DGmodel', 'prop:DGdeg', 'lem:hw', 'cor:tfinite', 'lem:lower', 'prop:weights', 'thm:DGprim', 'rem:DGnormal', 'eq:twoparams', 'lem:loc', 'thm:primmono', 'subsec:HXtorsion', 'prop:HXprim', 'q:remainingprim']

RESULT_KINDS = {'thm:A': 'theorem', 'thm:B': 'theorem', 'lem:fracind': 'lemma', 'lem:qplane': 'lemma', 'lem:finite': 'lemma', 'lem:shift': 'lemma', 'lem:shiftmat': 'lemma', 'thm:twist': 'theorem', 'prop:jordan': 'proposition', 'prop:noPI': 'proposition', 'rem:LM': 'remark', 'lem:rescale': 'lemma', 'lem:plane': 'lemma', 'thm:mono': 'theorem', 'cor:mono': 'corollary', 'lem:potY': 'lemma', 'lem:dickson': 'lemma', 'thm:DGPI': 'theorem', 'prop:QV': 'proposition', 'thm:L': 'theorem', 'lem:rank4': 'lemma', 'lem:Fnormal': 'lemma', 'thm:F': 'theorem', 'thm:I': 'theorem', 'thm:C': 'theorem', 'prop:crossed': 'proposition', 'lem:A1': 'lemma', 'lem:A2': 'lemma', 'lem:A3': 'lemma', 'thm:monodeg': 'theorem', 'rem:monodeg': 'remark', 'lem:cross': 'lemma', 'lem:words': 'lemma', 'lem:Theta': 'lemma', 'thm:QVdeg': 'theorem', 'lem:Rnorm': 'lemma', 'thm:Rdeg': 'theorem', 'lem:Fnorm': 'lemma', 'lem:Inorm': 'lemma', 'thm:FIdeg': 'theorem', 'thm:FIQRV': 'theorem', 'rem:bounds': 'remark', 'prop:bounds': 'proposition', 'q:degrees': 'question', 'cor:det': 'corollary', 'rem:detsigma': 'remark', 'prop:notprimitive': 'proposition', 'prop:rational': 'proposition', 'prop:Bnotprim': 'proposition', 'prop:CFIQRV': 'proposition', 'lem:centerQT': 'lemma', 'thm:DG': 'theorem', 'lem:Omega': 'lemma', 'lem:DGmodel': 'lemma', 'prop:DGdeg': 'proposition', 'lem:hw': 'lemma', 'cor:tfinite': 'corollary', 'lem:lower': 'lemma', 'prop:weights': 'proposition', 'thm:DGprim': 'theorem', 'rem:DGnormal': 'remark', 'lem:loc': 'lemma', 'thm:primmono': 'theorem', 'prop:HXprim': 'proposition', 'q:remainingprim': 'question'}


In [16]:
# Auditable verification suite
"""Run the exact checks and write a human-readable and machine-readable audit."""
from collections import Counter
from dataclasses import replace
from math import gcd
from pathlib import Path
import argparse
import json
import platform
import sympy as sp


def run_all(include_degree_samples=True, include_dickson_samples=True):
    checks = []
    def check(identifier,category,family,passed,detail=None,labels=()):
        if not labels:
            mapping={'source matrices':('sec:prelim',),'PBW overlaps':('sec:prelim',),
                     'det sigma table':('tab:det',),'monomial basis changes':('tab:mono',),
                     'central elements at h=1':('prop:QV',),'normalization of G':('eq:MDG',),
                     'rational coordinate matrices':('prop:QV',),
                     'degenerate presentations':('prop:noPI',),'PI table examples':('tab:main',),
                     'Dickson samples n=3..8':('lem:dickson',),
                     'DG degree samples N<=10':('prop:DGdeg',),'L degree samples N<=8':('thm:L',)}
            labels=mapping.get(category,())
            if category=='normal elements':
                labels=({'F':('lem:Fnormal',),'I':('eq:Inormal',),'C':('eq:Cnormal',)}[family])
        checks.append({'id':identifier,'category':category,'family':family,
                       'passed':bool(passed),'detail':detail,'manuscript_labels':list(labels)})
    for name,family in FAMILIES.items():
        algebra = Algebra(family)
        x1,x2,y1,y2 = algebra.generators
        residual = family.matrix(family.reshuffled_m()-family.printed_m)
        check(f'{name}.M','source matrices',name,not any(residual),str(residual))
        det = family.scalar(family.sigma.det())
        check(f'{name}.detSigma','source matrices',name,det != 0,
              {'determinant':str(sp.factor(det)),
               'nonzero_parameters':[str(v) for v in family.nonzero],
               'meaning':'nonzero in the generic admissible parameter field'})
        residual = family.matrix(family.detsigma()-family.expected_detsigma)
        check(f'{name}.detsigma','det sigma table',name,not any(residual),str(residual))
        det_matrix = family.detsigma()
        u = det_matrix[0,0]*x1+det_matrix[0,1]*x2
        v = det_matrix[1,0]*x1+det_matrix[1,1]*x2
        q12,q11 = family.Q
        residual = v*u-q12*u*v-q11*u**2
        check(f'{name}.det_preserves_plane','det sigma table',name,not residual,residual.certificate())
        for word,residual in algebra.overlaps().items():
            check(f'{name}.overlap.{word}','PBW overlaps',name,not residual,residual.certificate())

    for name,datum in MONOMIAL.items():
        family = datum.family
        transformed = datum.transformed_sigma()
        target = datum.target_sigma()
        for row in range(4):
            residual = family.matrix(transformed[row,:]-target[row,:])
            check(f'{name}.monomial.{row+1}','monomial basis changes',name,not any(residual),str(residual))
        check(f'{name}.basis_invertible','monomial basis changes',name,
              datum.x_basis.det() != 0 and datum.y_basis.det() != 0,
              {'det_x':str(datum.x_basis.det()),'det_y':str(datum.y_basis.det()),
               'invariants':[str(v) for v in datum.invariants()],
               'additional_restriction':'s != 0' if datum.square_root else None})

    # Identities used in the PI proofs, at a symbolic twist h.
    for name in ('F','I','C'):
        algebra = Algebra(FAMILIES[name])
        x1,x2,y1,y2 = algebra.generators
        if name == 'F':
            omega = [x1*x2,x1**2+x2**2]
            residuals = [y*w-(-2*p*h**2)*w*y for y in (y1,y2) for w in omega]
        elif name == 'I':
            omega = [y1*y2,y1**2+y2**2]
            residuals = [w*x-(-2*q*h**2)*x*w for x in (x1,x2) for w in omega]
        else:
            a,b = y1**2*y2,y1**3-y2**3
            residuals = [a*x-3*(1-p)*h**3*x*a for x in (x1,x2)]
            residuals += [b*x-3*(2*p+1)*h**3*x*b for x in (x1,x2)]
        for j,residual in enumerate(residuals,1):
            check(f'{name}.normal.{j}','normal elements',name,not residual,residual.certificate())

    for name in ('Q','V'):
        algebra = Algebra(FAMILIES[name].specialize({h:1}))
        x1,x2,y1,y2 = algebra.generators
        elements = [y1**2+y2**2,y1**2*y2**2]
        elements += ([x1**2+x2**2,x1**2*x2**2+sp.Rational(1,2)*x2**4]
                     if name=='Q' else [x2,x1**2*(x1-x2)**2])
        for j,element in enumerate(elements,1):
            for k,gen in enumerate(algebra.generators,1):
                residual = element.commutator(gen)
                check(f'{name}.central.{j}.{k}','central elements at h=1',name,
                      not residual,residual.certificate())

    algebra = Algebra(FAMILIES['G'])
    x1,x2,y1,y2 = algebra.generators
    e2 = x2+(1/(p-1))*x1
    residuals = [y1*x1-h*p*x1*y1,
                 y1*e2-h*(p**2*e2*y1+x1*y2),
                 y2*x1-h*p*x1*y2,
                 y2*e2-h*(f*x1*y1+e2*y2)]
    for j,residual in enumerate(residuals,1):
        check(f'G.coordinate_shift.{j}','normalization of G','G',not residual,residual.certificate())

    matrices = {'Q':sp.Matrix([[-1,1-t],[1+t,1]]),
                'V':sp.Matrix([[1,t],[1-t,0]])}
    for name,matrix in matrices.items():
        image = (matrix**2*(2*sp.eye(2)-matrix**2) if name=='Q'
                 else (matrix**2-matrix)**2)
        polynomial = t**2*(2-t**2) if name=='Q' else (t**2-t)**2
        residual = sp.simplify(image-polynomial*sp.eye(2))
        check(f'{name}.invariant_matrix','rational coordinate matrices',name,not any(residual),str(residual))

    # Degenerate algebras remain admissible; monomial bases must not be used there.
    for name,value in [('D',1),('D',-1),('M',0),('O',0),('P',0),('W',0)]:
        family = FAMILIES[name].specialize({p:value} if name=='D' else {f:value})
        for word,residual in Algebra(family).overlaps().items():
            check(f'{name}.degenerate.{value}.{word}','degenerate presentations',name,
                  not residual,residual.certificate())
    o0,dminus = FAMILIES['O'].specialize({f:0}),FAMILIES['D'].specialize({p:-1})
    check('O0.equals.Dminus','degenerate presentations','O/D',
          o0.sigma==dminus.sigma and o0.P==dminus.P and o0.Q==dminus.Q)

    # Negative controls: exact, nonzero residuals must be detected.
    bad = FAMILIES['Z'].sigma.as_mutable()
    bad[0,0],bad[0,2],bad[3,0],bad[3,2] = 0,h,0,f*h
    wrong = Algebra(replace(FAMILIES['Z'],sigma=bad.as_immutable()))
    for word,residual in wrong.overlaps().items():
        check(f'Z.printed_relations_rejected.{word}','negative controls','Z',bool(residual),
              {'expected':'nonzero','residual':residual.certificate()})
    for name,subs in [('G',{p:1}),('Z',{f:0}),('K',{f:0}),('B',{p:2}),('C',{p:1}),('E',{h:0})]:
        try:
            FAMILIES[name].specialize(subs)
            rejected = False
        except ValueError:
            rejected = True
        check(f'{name}.invalid.{subs}','negative controls',name,rejected)

    # Exact root-of-unity decisions test the table evaluator; they are not proofs
    # of the theorem assigning the table to all double Ore algebras.
    for name,scale in [('C',1/sp.sqrt(3)),('E',1/sp.sqrt(2)),('F',1/sp.sqrt(2)),
                       ('I',1/sp.sqrt(2)),('J',1/sp.sqrt(2)),('S',sp.Rational(1,2)),
                       ('T',sp.Rational(1,2)),('U',sp.Rational(1,2))]:
        unscaled,scaled = pi_decision(name,{h:1}),pi_decision(name,{h:scale})
        check(f'{name}.twist_example','PI table examples',name,
              unscaled['status'] is False and scaled['status'] is True,
              {'h=1':unscaled,'PI_specialization':scaled,'h':str(scale)})
    for name,subs in [('O',{f:9,h:1/(2*sp.sqrt(2))}),('N',{f:2,g:sp.sqrt(3)})]:
        family = FAMILIES[name].specialize(subs)
        decision = pi_decision(name,subs)
        check(f'{name}.detfinite_not_PI','PI table examples',name,
              family.detsigma()==sp.eye(2) and decision['status'] is False,decision)
    for value,expected in [(1,1),(-1,2),(sp.I,4),(-sp.I,4),
                           ((1+sp.I)/sp.sqrt(2),8),(sp.sqrt(2),0),(sp.Rational(3,2),0)]:
        actual = exact_root_order(value)
        check(f'root_order.{value}','root-of-unity arithmetic','scalar',actual==expected,
              {'value':str(value),'order':actual})

    extended_polynomial_checks(check)
    rational_center_checks(check)
    affine_nonpi_checks(check)
    word_checks(check)
    algebraic_checks(check)
    primitivity_checks(check)
    uniform_checks(check)
    # These APIs take arbitrary orders; the listed calls are regression inputs.
    all_degree_checks(check)
    if include_degree_samples:
        weight_matrix_checks(check)
        dg_central_checks(check)

    if include_dickson_samples:
        for order in range(3,9):
            for name in ('D','G'):
                residual = dickson_matrix_residual(order,name)
                check(f'{name}.Dickson.order{order}','Dickson samples n=3..8',name,
                      not any(residual),str(residual))

    if include_degree_samples:
        for name in ('D','G'):
            for order in range(3,11):
                for k in range(order):
                    for kp in range(order):
                        try:
                            matrix = dg_matrix(name,order,k,kp)
                        except ValueError:
                            continue
                        certificate = torus_certificate(matrix,order)
                        predicted = dg_closed_formula(order,k,kp)
                        check(f'{name}.SNF.{order}.{k}.{kp}','DG degree samples N<=10',name,
                              certificate['pi_degree']==predicted,
                              {'N':order,'k':k,'h_exponent':kp,
                               'smith_diagonal':certificate['smith_diagonal'],
                               'computed':certificate['pi_degree'],'formula':predicted})
        for order in range(2,9,2):
            for th in range(order):
                for ph in range(order):
                    for eta in range(order):
                        try:
                            matrix = l_matrix(order,th,ph,eta)
                        except ValueError:
                            continue
                        certificate = torus_certificate(matrix,order)
                        computed = 2*certificate['pi_degree']
                        predicted = l_closed_formula(order,th,ph,eta)
                        check(f'L.SNF.{order}.{th}.{ph}.{eta}','L degree samples N<=8','L',
                              computed==predicted,
                              {'N':order,'theta':th,'phi':ph,'eta':eta,
                               'computed_algebra_degree':computed,'formula':predicted,
                               'torus_certificate':certificate})
        for order in range(2,6):
            matrix = alternating((1,2,0,1,1,2))
            certificate = torus_certificate(matrix,order)
            actual = brute_radical_size(matrix,order)
            check(f'SNF.independent_enumeration.{order}','independent Smith controls','torus',
                  actual==certificate['radical_size_mod_N'],certificate)

    totals = Counter(check['category'] for check in checks)
    failed = [check['id'] for check in checks if not check['passed']]
    coverage=coverage_report(checks)
    return {'project':'Double Ore 14641 PI verification','version':'0.3.0',
            'environment':{'python':platform.python_version(),'sympy':sp.__version__},
            'summary':{'total':len(checks),'passed':len(checks)-len(failed),'failed':len(failed),
                       'categories':dict(totals),'families':26,'monomial_families':14},
            'failed_checks':failed,
            'scope':{'symbolic':'26 presentations; 14 monomial data; all torus/action models; normalizers; central rational and orbit certificates; localization elements; module and recurrence certificates',
                     'arbitrary_order_apis':'21 closed PI degree computations, C bounds, center lattices, conditional central orbits, central generators and weight matrices',
                     'sampled':'Finite-order regressions are identified by category; they are not the sole basis for symbolic identities',
                     'not_machine_formalized':['abstract division-ring and ideal-theoretic arguments',
                                               'a proof-assistant formalization of the manuscript'],
                     'open_in_manuscript':['exact PI degree of C','the remaining primitivity question']},
            'coverage':coverage,
            'checks':checks}


def write_report(report,directory):
    directory = Path(directory)
    directory.mkdir(parents=True,exist_ok=True)
    (directory/'verification.json').write_text(json.dumps(report,indent=2,ensure_ascii=False,default=lambda a:int(a) if getattr(a,'is_Integer',False) else str(a))+'\n')
    (directory/'coverage.json').write_text(json.dumps(report['coverage'],indent=2,ensure_ascii=False)+'\n')
    lines = ['# Exact verification report','',
             f"Checks passed: **{report['summary']['passed']}/{report['summary']['total']}**.",
             f"Failed checks: **{report['summary']['failed']}**.",'',
             '| Category | Checks |','|---|---:|']
    lines += [f'| {name} | {count} |' for name,count in report['summary']['categories'].items()]
    lines += ['', 'The parameter equations are reduced symbolically; no floating point is used.',
              'For the 26 presentations, all four cubic ambiguities are resolved in the admissible parameter field.',
              'The 26 Sigma matrices and independently transcribed M matrices agree after tensor reshuffling.',
              'The overall factor h missing in the printed M for J is restored explicitly.',
              'All 26 det(sigma) matrices agree with the manuscript table.',
              'All 14 monomial basis changes are checked at symbolic parameters.',
              '', '## Scope', '',
              'All 21 closed degree models are implemented with exact torus matrices, center lattices and crossed-product action images, without a root-order cutoff.',
              'The derived degree is compared independently with the manuscript formula. General recurrence and module identities have separate algebraic certificates.',
              'Finite regression inputs remain explicitly labeled. Abstract division-ring, ideal-theoretic and independence proofs are documented mathematical dependencies, not proof-assistant formalizations.',
              'The PI table evaluator still instantiates Table tab:main; it is not presented as an independent proof.',
              'The exact degree of C and remaining primitivity cases stay open, as in the manuscript.',
              '', '## Manuscript correction found by the completed checks','',
              'In the L torus ordered as w,y1,y1^-1*y2,u, the first upper exponent must be -(2*eta+phi).',
              'The manuscript prints +(2*eta+phi). The relation w*y1=(h^2*f)^(-1)*y1*w verifies the correction.',
              'Its Pfaffian and alternating Smith invariants are unchanged, so the stated degree formula remains unchanged.',
              'See docs/MANUSCRIPT_CORRECTIONS.md for the exact replacement.',
              '', '## Negative control for the printed Z relations', '',
              'Replacing x1*y1 by x1*y2 in the two indicated positions produces nonzero cubic overlap residuals.',
              'The four explicit residuals are retained in verification.json.',
              '',f"Python {report['environment']['python']}; SymPy {report['environment']['sympy']}."]
    if report['failed_checks']:
        lines += ['', '## Failed checks','']+report['failed_checks']
    (directory/'verification.md').write_text('\n'.join(lines)+'\n')


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--output',default='reports')
    parser.add_argument('--quick',action='store_true',help='omit finite order degree and Dickson samples')
    args = parser.parse_args()
    report = run_all(not args.quick,not args.quick)
    write_report(report,args.output)
    print(json.dumps(report['summary'],indent=2))
    if report['failed_checks']:
        raise SystemExit('FAILED: '+', '.join(report['failed_checks']))


In [17]:
# Independent certificate bundle exporter
"""Export inspectable inputs and witnesses for the independent checker."""
from pathlib import Path
import argparse
import json
import sympy as sp
from sympy.matrices.normalforms import smith_normal_decomp


def export_bundle(report):
    sources=[]
    for name,family in FAMILIES.items():
        sources.append({'family':name,'P':list(map(str,family.P)),'Q':list(map(str,family.Q)),
           'sigma':[[str(v) for v in row] for row in family.sigma.tolist()],
           'M':[[str(v) for v in row] for row in family.printed_m.tolist()],
           'expected_detsigma':[[str(v) for v in row] for row in family.expected_detsigma.tolist()],
           'relations':[[str(poly),str(var)] for poly,var in family.relations],
           'admissibility':list(map(str,family.nonzero)),'source':{'section':family.section,'page':family.page}})
    cases=[]
    for check in report['checks']:
        c=check.get('detail')
        if check['category'] not in ('all-family degree certificates','center-action parity branches',
                                     'fourth-root degree branches','K/L parity certificates'):continue
        A=sp.Matrix(c['commutation_matrix']);N=c['root_order']
        D,U,V=smith_normal_decomp(A,domain=sp.ZZ)
        Gamma=sp.Matrix(c['center_lattice']);factors=[N//sp.gcd(N,D[i,i]) for i in range(4)]
        kernel=V*sp.diag(*factors);W=kernel.inv()*Gamma
        if not all(v.is_Integer for v in W) or abs(W.det())!=1:raise ArithmeticError('Kernel equivalence must be unimodular')
        integer=lambda mat:[[int(v) for v in row] for row in mat.tolist()]
        cases.append({'id':check['id'],'family':c['family'],'N':N,'A':integer(A),
             'D':integer(D),'U':integer(U),'V':integer(V),'Gamma':integer(Gamma),
             'kernel_change':integer(W),'actions':[
                 {'coset':a['coset'],'exponent_matrix':integer(sp.Matrix(a['exponent_matrix'])),
                  'phases':list(map(int,a['phases']))} for a in c.get('actions',[])],
             'torus_degree':c['pi_degree'],'center_action_order':c['center_action_order'],
             'algebra_degree':c['algebra_pi_degree'],'formula':c['closed_formula']})
    basis_changes=[]
    for name,datum in MONOMIAL.items():
        basis_changes.append({'family':name,'square_root':bool(datum.square_root),
            'E':[[str(v) for v in row] for row in datum.x_basis.tolist()],
            'Z':[[str(v) for v in row] for row in datum.y_basis.tolist()],
            'target':[[str(v) for v in row] for row in datum.target_sigma().tolist()]})
    return {'schema':'ore14641.certificate_bundle.v1','sources':sources,'basis_changes':basis_changes,
            'uniform':uniform_bundle(),'integer_cases':cases,
            'scope':'Inputs and witnesses, never only pass flags or simplified zero residuals.'}


def main():
    parser=argparse.ArgumentParser();parser.add_argument('--reports',default='reports');args=parser.parse_args()
    directory=Path(args.reports)
    report=json.loads((directory/'verification.json').read_text())
    bundle=export_bundle(report)
    (directory/'certificates.json').write_text(json.dumps(bundle,indent=2,ensure_ascii=False)+'\n')
    print('Exported',len(bundle['integer_cases']),'integer certificates and',len(bundle['uniform']['identities']),'generic identities.')


## 2. Inspect a family

Change `family_name` to any letter A--Z. The displayed matrices are in the
original Section 4 coordinates. In particular, G has not silently been replaced
by its shifted presentation. The x-plane and y-plane relations are determined
by Q and P. The four displayed mixed rules follow directly from the matrix.


In [18]:
family_name = 'Z'
family = FAMILIES[family_name]
print('Family', family_name, '| ZZ09 subcase', family.section, '| printed page', family.page)
print('P =', family.P, '| Q =', family.Q)
print('Nonzero restrictions:', ', '.join(map(str,family.nonzero)))
print('Parameter equations:', [(str(a), str(v)) for a,v in family.relations])
display(Math(r'\Sigma = '+sp.latex(family.sigma)))
display(Math(r'M = '+sp.latex(family.printed_m)))
display(Math(r'\det(\sigma)|_{A_1} = '+sp.latex(family.detsigma())))
print('Ordinary det(Sigma) =', sp.factor(family.scalar(family.sigma.det())))
algebra = Algebra(family)
for lhs,rhs in algebra.rules.items():
    if lhs[0] >= 2 and lhs[1] < 2:
        print(' '.join(('x1','x2','y1','y2')[i] for i in lhs),'=',Element(algebra,rhs))


Family Z | ZZ09 subcase 4.4.4 | printed page 400
P = (1, 0) | Q = (-1, 0)
Nonzero restrictions: h, f, f + 1
Parameter equations: []


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Ordinary det(Sigma) = h**4*(f + 1)**2
y1 x1 = (h)*x1*y1 + (h)*x2*y2
y1 x2 = (h)*x1*y2 + (h)*x2*y1
y2 x1 = (-h)*x1*y2 + (f*h)*x2*y1
y2 x2 = (f*h)*x1*y1 + (-h)*x2*y2


## 3. Run all exact checks

For every family there are four critical cubic ambiguities. All reductions
decrease degree-lex order, so their resolution checks confluence of the six
quadratic rules. Parameter equations are reduced symbolically, covering both
allowed fourth-root or cube-root branches. No random floating point test is used.

The generic Dickson eigenvector and root-power identities are certified
symbolically for the general argument. Additional explicit order checks are
regressions. The degree APIs accept arbitrary common root orders, and derive
center-action images from exact modular lattices rather than a bounded search.
Finite lists of input orders are still labeled as regression checks.


In [19]:
report = run_all()
display(Markdown('| Category | Checks |\n|---|---:|\n'+'\n'.join(
    f'| {name} | {count} |' for name,count in report['summary']['categories'].items())))
print('PASS:',report['summary']['passed'],'/',report['summary']['total'])
print('FAILED:',report['failed_checks'])
if report['failed_checks']:
    raise RuntimeError('Verification failed: '+str(report['failed_checks']))


| Category | Checks |
|---|---:|
| source matrices | 52 |
| det sigma table | 52 |
| PBW overlaps | 104 |
| monomial basis changes | 70 |
| normal elements | 12 |
| central elements at h=1 | 32 |
| normalization of G | 4 |
| rational coordinate matrices | 2 |
| degenerate presentations | 25 |
| negative controls | 10 |
| PI table examples | 10 |
| root-of-unity arithmetic | 7 |
| additional PBW identities | 61 |
| manuscript discrepancy controls | 1 |
| localization normality | 48 |
| anti-isomorphism identities | 12 |
| central rational certificates | 102 |
| non-PI affine matrices | 14 |
| central power certificates | 9 |
| normal-generator matrices | 4 |
| Laurent identities | 490 |
| plane charts | 28 |
| formal parameter charts | 42 |
| symbolic torus splitting | 12 |
| Laurent inverse controls | 19 |
| crossed-product closure | 38 |
| torus action matrices | 19 |
| general algebraic certificates | 47 |
| plane lemma identities | 20 |
| finite-module certificates | 7 |
| general Dickson certificate | 2 |
| C rank-nine module | 6 |
| C module reduction regressions | 55 |
| nonprimitive orbit invariants | 8 |
| primitive localization obstructions | 6 |
| conditional orbit regressions | 24 |
| DG center lattice certificates | 5 |
| unbounded algebraic proof objects | 40 |
| unbounded integer ideal certificates | 5 |
| all-family degree certificates | 84 |
| center-action parity branches | 9 |
| fourth-root degree branches | 6 |
| K/L parity certificates | 6 |
| C bound certificates | 3 |
| weight matrix regressions | 6 |
| weight spectrum regressions | 6 |
| DG central generator regressions | 128 |
| Dickson samples n=3..8 | 12 |
| DG degree samples N<=10 | 412 |
| L degree samples N<=8 | 720 |
| independent Smith controls | 4 |

PASS: 2900 / 2900
FAILED: []


## 3b. Independent checker, induction and parity certificates

This checker uses only exact integers and Python's `Fraction` arithmetic.
The proof objects contain both sides of each identity, signed source words,
and unimodular Smith/kernel witnesses, rather than only zero residuals.
The uniform argument and its mathematical assumptions are documented in
`UNBOUNDED_PROOFS.md` in the companion package.


In [20]:
from pathlib import Path
Path("independent_check.py").write_text('#!/usr/bin/env python3\n"""Independent certificate checker. Python standard library ONLY.\n\nNo imports from ore14641, SymPy, NumPy, Sage, or Jupyter. Polynomials use sparse\nFraction arithmetic; PBW uses a rightmost worklist; balanced Laurent planes use\ncentral squares plus alternating words, not the producer\'s ratio coordinates.\n"""\nimport ast\nfrom fractions import Fraction as F\nfrom itertools import combinations, product\nimport json\nimport re\nfrom math import gcd, isqrt, prod\nfrom pathlib import Path\nimport sys\n\n\nclass Ring:\n    def __init__(self,expressions=(),relations=()):\n        names={\'I\'}\n        for s in tuple(expressions)+tuple(v[0] for v in relations):\n            names.update(x.id for x in ast.walk(ast.parse(str(s),mode=\'eval\')) if isinstance(x,ast.Name))\n        self.names=tuple(sorted(names));self.index={v:i for i,v in enumerate(self.names)}\n        self.zero=(0,)*len(self.names);self.relations={};self.cache={}\n        # Quotient relations are monic univariate polynomials.\n        self.relations[self.index[\'I\']]=(F(1),F(0),F(1))\n        for expression,var in relations:\n            polynomial=self.parse(expression).num\n            pos=self.index[var];coeff={m[pos]:c for m,c in polynomial.items()}\n            if any(any(e for j,e in enumerate(m) if j!=pos) for m in polynomial):\n                raise ValueError(\'Only univariate coefficient constraints allowed\')\n            degree=max(coeff);lead=coeff[degree]\n            self.relations[pos]=tuple(coeff.get(i,F(0))/lead for i in range(degree+1))\n    def clean(self,p):\n        result={};pending=list(p.items())\n        while pending:\n            m,c=pending.pop()\n            if not c:continue\n            for j,coeff in self.relations.items():\n                d=len(coeff)-1\n                if m[j]>=d:\n                    for i,a in enumerate(coeff[:-1]):\n                        if a:\n                            mm=list(m);mm[j]+=i-d;pending.append((tuple(mm),-c*a))\n                    break\n            else:result[m]=result.get(m,F(0))+c\n        return {m:c for m,c in result.items() if c}\n    def add(self,a,b):\n        r=a.copy()\n        for m,c in b.items():r[m]=r.get(m,F(0))+c\n        return self.clean(r)\n    def mul(self,a,b):\n        r={}\n        for m,c in a.items():\n            for n,d in b.items():\n                k=tuple(x+y for x,y in zip(m,n));r[k]=r.get(k,F(0))+c*d\n        return self.clean(r)\n    def const(self,v):return Rat(self,{self.zero:F(v)},{self.zero:F(1)})\n    def parse(self,source):\n        source=str(source)\n        if source in self.cache:return self.cache[source]\n        def visit(x):\n            if isinstance(x,ast.Constant) and isinstance(x.value,int):return self.const(x.value)\n            if isinstance(x,ast.Name):\n                m=list(self.zero);m[self.index[x.id]]=1\n                return Rat(self,self.clean({tuple(m):F(1)}),{self.zero:F(1)})\n            if isinstance(x,ast.UnaryOp):\n                a=visit(x.operand)\n                if isinstance(x.op,ast.USub):return -a\n                if isinstance(x.op,ast.UAdd):return a\n            if isinstance(x,ast.BinOp):\n                a=visit(x.left)\n                if isinstance(x.op,ast.Pow):\n                    e=visit(x.right).integer();return a**e\n                b=visit(x.right)\n                if isinstance(x.op,ast.Add):return a+b\n                if isinstance(x.op,ast.Sub):return a-b\n                if isinstance(x.op,ast.Mult):return a*b\n                if isinstance(x.op,ast.Div):return a/b\n            raise ValueError(\'Unsupported expression node \'+ast.dump(x))\n        value=visit(ast.parse(source,mode=\'eval\').body);self.cache[source]=value\n        return value\n\n\nclass Rat:\n    def __init__(self,ring,num,den):\n        self.ring=ring;num=ring.clean(num);den=ring.clean(den)\n        if not den:raise ZeroDivisionError(\'Zero certificate denominator\')\n        if not num:self.num={};self.den={ring.zero:F(1)};return\n        # Cancel common monomial factors (no external polynomial gcd).\n        mn=[min(m[j] for m in num) for j in range(len(ring.names))]\n        md=[min(m[j] for m in den) for j in range(len(ring.names))]\n        cancel=tuple(min(a,b) for a,b in zip(mn,md))\n        if any(cancel):\n            num={tuple(a-b for a,b in zip(m,cancel)):c for m,c in num.items()}\n            den={tuple(a-b for a,b in zip(m,cancel)):c for m,c in den.items()}\n        lead=den[max(den)]\n        self.num={m:c/lead for m,c in num.items()};self.den={m:c/lead for m,c in den.items()}\n    def asrat(self,b):return b if isinstance(b,Rat) else self.ring.const(b)\n    def __neg__(self):return Rat(self.ring,{m:-c for m,c in self.num.items()},self.den)\n    def __add__(self,b):\n        b=self.asrat(b);r=self.ring\n        if self.den==b.den:return Rat(r,r.add(self.num,b.num),self.den)\n        return Rat(r,r.add(r.mul(self.num,b.den),r.mul(b.num,self.den)),r.mul(self.den,b.den))\n    __radd__=__add__\n    def __sub__(self,b):return self+-self.asrat(b)\n    def __rsub__(self,b):return self.asrat(b)+-self\n    def __mul__(self,b):\n        b=self.asrat(b);r=self.ring\n        return Rat(r,r.mul(self.num,b.num),r.mul(self.den,b.den))\n    __rmul__=__mul__\n    def __truediv__(self,b):\n        b=self.asrat(b);r=self.ring\n        return Rat(r,r.mul(self.num,b.den),r.mul(self.den,b.num))\n    def __pow__(self,n):\n        if n<0:return Rat(self.ring,self.den,self.num)**(-n)\n        r=self.ring.const(1);a=self\n        while n:\n            if n&1:r=r*a\n            a=a*a;n//=2\n        return r\n    def __bool__(self):return bool(self.num)\n    def __eq__(self,b):\n        b=self.asrat(b);r=self.ring\n        return r.mul(self.num,b.den)==r.mul(b.num,self.den)\n    def integer(self):\n        if set(self.num)-{self.ring.zero} or set(self.den)-{self.ring.zero}:raise ValueError(\'Not constant\')\n        a=self.num.get(self.ring.zero,F(0))/self.den[self.ring.zero]\n        if a.denominator!=1:raise ValueError(\'Not integral\')\n        return int(a)\n\n\ndef transpose(A):return list(map(list,zip(*A)))\ndef mm(A,B):return [[sum(a*b for a,b in zip(row,col)) for col in zip(*B)] for row in A]\ndef eye(n):return [[int(i==j) for j in range(n)] for i in range(n)]\ndef det(A):\n    A=[list(map(F,row)) for row in A];n=len(A);ans=F(1)\n    for j in range(n):\n        pivot=next((i for i in range(j,n) if A[i][j]),None)\n        if pivot is None:return 0\n        if pivot!=j:A[j],A[pivot]=A[pivot],A[j];ans=-ans\n        value=A[j][j];ans*=value\n        for i in range(j+1,n):\n            factor=A[i][j]/value\n            for k in range(j+1,n):A[i][k]-=factor*A[j][k]\n    return ans\n\n\ndef source_checks(sources,record):\n    for s in sources:\n        expressions=sum(s[\'sigma\'],[])+sum(s[\'M\'],[])+list(s[\'P\'])+list(s[\'Q\'])\n        expressions+=sum(s[\'expected_detsigma\'],[])+[p for p,v in s[\'relations\']]\n        ring=Ring(expressions,s[\'relations\']);parse=ring.parse\n        sigma=[[parse(v) for v in row] for row in s[\'sigma\']]\n        M=[[parse(v) for v in row] for row in s[\'M\']];order=(0,2,1,3)\n        record(s[\'family\']+\'.reshuffle\',all(M[i][j]==sigma[order[i]][order[j]] for i in range(4) for j in range(4)))\n        p12,p11=map(parse,s[\'P\']);q12,q11=map(parse,s[\'Q\'])\n        blocks=[[[sigma[2*a+i][2*b+j] for j in range(2)] for i in range(2)] for a in range(2) for b in range(2)]\n        B11,B12,B21,B22=blocks\n        C1,C2,C3=mm(B11,B12),mm(B11,B22),mm(B21,B12)\n        D=[[-p11*C1[i][j]+C2[i][j]-p12*C3[i][j] for j in range(2)] for i in range(2)]\n        record(s[\'family\']+\'.detsigma\',all(D[i][j]==parse(s[\'expected_detsigma\'][i][j]) for i in range(2) for j in range(2)))\n        rules={(1,0):{(0,1):q12,(0,0):q11},(3,2):{(2,3):p12,(2,2):p11}}\n        targets=((0,2),(1,2),(0,3),(1,3))\n        for i,key in enumerate(((2,0),(2,1),(3,0),(3,1))):rules[key]={w:sigma[i][j] for j,w in enumerate(targets) if sigma[i][j]}\n        rules={key:{w:c for w,c in value.items() if c} for key,value in rules.items()}\n        record(s[\'family\']+\'.decreasing\',all(v<k for k,ts in rules.items() for v in ts))\n        def reduce_terms(initial):\n            pending=list(initial);out={}\n            while pending:\n                word,c=pending.pop()\n                pos=next((j for j in range(len(word)-2,-1,-1) if word[j:j+2] in rules),None)\n                if pos is None:out[word]=out.get(word,ring.const(0))+c\n                else:\n                    for v,d in rules[word[pos:pos+2]].items():pending.append((word[:pos]+v+word[pos+2:],c*d))\n            return {w:c for w,c in out.items() if c}\n        for abc in combinations(range(4),3):\n            word=tuple(reversed(abc));branches=[]\n            for pos in (0,1):branches.append(reduce_terms([(word[:pos]+v+word[pos+2:],d) for v,d in rules[word[pos:pos+2]].items()]))\n            keys=set(branches[0])|set(branches[1])\n            record(s[\'family\']+\'.PBW.\'+str(word),all(branches[0].get(w,ring.const(0))==branches[1].get(w,ring.const(0)) for w in keys))\n\n\ndef basis_checks(sources,changes,record):\n    sources={s[\'family\']:s for s in sources}\n    for change in changes:\n        source=sources[change[\'family\']]\n        convert=lambda expr:re.sub(r\'\\bf\\b\',\'(s**2)\',expr) if change[\'square_root\'] else expr\n        strings=sum(source[\'sigma\'],[])+sum(change[\'E\'],[])+sum(change[\'Z\'],[])+sum(change[\'target\'],[])\n        strings=list(map(convert,strings))\n        r=Ring(strings,source[\'relations\'])\n        matrix=lambda A:[[r.parse(convert(x)) for x in row] for row in A]\n        S,E,Z,T=map(matrix,(source[\'sigma\'],change[\'E\'],change[\'Z\'],change[\'target\']))\n        C=[[Z[i//2][j//2]*E[i%2][j%2] for j in range(4)] for i in range(4)]\n        record(change[\'family\']+\'.independent_basis_change\',mm(C,S)==mm(T,C))\n        record(change[\'family\']+\'.independent_basis_invertible\',bool(E[0][0]*E[1][1]-E[0][1]*E[1][0]) and bool(Z[0][0]*Z[1][1]-Z[0][1]*Z[1][0]))\n\n\ndef word_checks(models,record):\n    for model in models:\n        expressions=model[\'plane_q\']+sum(model[\'commutations\'],[])+[v[1][2] for v in model[\'relations\']]\n        expressions+=[s for a in model[\'actions\'] for s in a[\'scalars\']]\n        expressions+=[p for p,v in model[\'scalar_relations\']]\n        ring=Ring(expressions,model[\'scalar_relations\']);one=ring.const(1)\n        rules={}\n        for (a,b),(j,i,c) in model[\'relations\']:\n            c=ring.parse(c)\n            for key,val in [((a,b,1,1),(j,i,c)),((i,b,1,-1),(j,a,one/c)),\n                            ((a,j,-1,1),(b,i,one/c)),((i,j,-1,-1),(b,a,c))]:\n                if key in rules and rules[key]!=val:raise ValueError(\'Inconsistent inverse relations\')\n                rules[key]=val\n        def multiply(u,v):return (u[0]+v[0],u[1]*v[1])\n        def power(u,n):\n            if n<0:return power((tuple((a,b,-sg) for a,b,sg in reversed(u[0])),one/u[1]),-n)\n            return u[0]*n,u[1]**n\n        def decode(u):return tuple(tuple(x) for x in u[\'letters\']),ring.parse(u[\'coefficient\'])\n        def normal(word):\n            letters,c=word;letters=list(letters)\n            # Rightmost mixed inversion, versus the producer\'s leftmost pass.\n            while True:\n                at=next((i for i in range(len(letters)-2,-1,-1) if letters[i][0]==1 and letters[i+1][0]==0),None)\n                if at is None:break\n                _,a,sy=letters[at];_,b,sx=letters[at+1]\n                j,i,factor=rules[a,b,sy,sx];c=c*factor\n                letters[at:at+2]=[(0,j,sx),(1,i,sy)]\n            results=[]\n            for plane,kind in enumerate(model[\'planes\']):\n                current=[(i,s) for pl,i,s in letters if pl==plane]\n                if kind==\'coordinate\':\n                    x,y=0,0;q=ring.parse(model[\'plane_q\'][plane])\n                    for index,sg in current:\n                        if index==0:c=c*q**(y*sg);x+=sg\n                        else:y+=sg\n                    results.append((x,y))\n                else:\n                    square=0;stack=[]\n                    for index,sg in current:\n                        if sg<0:square-=1\n                        if stack and stack[-1]==index:stack.pop();square+=1\n                        else:stack.append(index)\n                    results.append((square,tuple(stack)))\n            return tuple(results),c\n        def equal(u,v):\n            a,c=normal(u);b,d=normal(v);return a==b and c==d\n        gens=list(map(decode,model[\'generators\']));units=list(map(decode,model[\'units\']))\n        if \'expected_upper\' in model:\n            record(model[\'family\']+\'.uniform_smith_input\',all(ring.parse(model[\'commutations\'][i][j])==ring.parse(v)\n                for (i,j),v in zip(combinations(range(4),2),model[\'expected_upper\'])))\n        for i,j in combinations(range(4),2):\n            factor=ring.parse(model[\'commutations\'][i][j])\n            right=multiply(gens[j],gens[i]);right=(right[0],right[1]*factor)\n            record(model[\'family\']+f\'.word.commutation.{i}.{j}\',equal(multiply(gens[i],gens[j]),right))\n        for u,label,action in zip(units,model[\'unit_labels\'],model[\'actions\']):\n            record(model[\'family\']+\'.word.action_det.\'+label,abs(det(action[\'matrix\']))==1)\n            for j,g in enumerate(gens):\n                rhs=((),ring.parse(action[\'scalars\'][j]))\n                for i in range(4):rhs=multiply(rhs,power(gens[i],action[\'matrix\'][i][j]))\n                lhs=multiply(multiply(u,g),power(u,-1))\n                record(model[\'family\']+f\'.word.action.{label}.{j}\',equal(lhs,rhs))\n        # Universal faithfulness / kernel, using the full-rank central lattice.\n        name=model[\'family\'];acts=model[\'actions\']\n        if name in (\'B\',\'E\',\'J\',\'S\',\'T\',\'U\'):\n            record(name+\'.universal_center_faithful\',all(a[\'matrix\']!=eye(4) for a in acts[1:]))\n        elif name in (\'Q\',\'V\',\'K_index_two\',\'L_index_two\'):\n            record(name+\'.universal_center_faithful\',acts[1][\'matrix\']!=eye(4))\n        elif name==\'R\':\n            signatures=[(a[\'matrix\'][0][0],a[\'scalars\'][0]) for a in acts]\n            record(name+\'.universal_center_faithful\',len(set(signatures))==4 and all(model[\'commutations\'][0][j]==\'1\' for j in range(4)))\n        elif name in (\'F\',\'I\'):\n            signatures=[(a[\'matrix\'][0][0],a[\'scalars\'][0],a[\'matrix\'][3][3],a[\'scalars\'][3]) for a in acts]\n            record(name+\'.universal_center_faithful\',len(set(signatures))==8 and all(model[\'commutations\'][i][j]==\'1\' for i in (0,3) for j in range(4)))\n        elif name in (\'M\',\'N\',\'O\',\'P\',\'W\',\'Z\'):\n            exceptional=[a for a in acts[1:] if a[\'matrix\']==eye(4)]\n            record(name+\'.one_possible_kernel_element\',len(exceptional)==1)\n            action=exceptional[0]\n            # u=alpha/beta except Z, where u=-alpha/beta; v=alpha*beta.\n            chars=[]\n            for expression in action[\'scalars\']:\n                z=ring.parse(expression)\n                if len(z.num)!=1 or len(z.den)!=1:raise ValueError(\'Expected Laurent scalar character\')\n                (nm,nc),=z.num.items();(dm,dc),=z.den.items()\n                exps={v:nm[i]-dm[i] for i,v in enumerate(ring.names)}\n                aa,bb=exps.get(\'alpha\',0),exps.get(\'beta\',0)\n                if (aa-bb)%2 or (aa+bb)%2:raise ValueError(\'Not a character in u,v\')\n                up,vp=(aa-bb)//2,(aa+bb)//2;constant=nc/dc\n                if name==\'Z\':constant*=(-1)**up\n                if any(e for v,e in exps.items() if v not in (\'alpha\',\'beta\')):raise ValueError(\'Unexpected character\')\n                chars.append((constant,up,vp))\n            parity=((1,1),(1,-1),(0,-1))\n            for ui,(um,uv) in enumerate(parity):\n                for vi,(vm,vv) in enumerate(parity):\n                    values=[]\n                    for idx,(constant,up,vp) in enumerate(chars):\n                        if idx<2:\n                            if vp:raise ValueError(\'Cross character in first pair\')\n                            values.append(constant**um*F(uv)**up)\n                        else:\n                            if up:raise ValueError(\'Cross character in second pair\')\n                            values.append(constant**vm*F(vv)**vp)\n                    trivial=all(v==1 for v in values)\n                    expected=name in (\'O\',\'W\',\'Z\') and ui==vi==0\n                    record(name+f\'.universal_parity.{ui}.{vi}\',trivial==expected)\n\n\ndef arithmetic_checks(data,record):\n    for claim in data[\'identities\']:\n        r=Ring((claim[\'lhs\'],claim[\'rhs\']))\n        record(claim[\'id\'],r.parse(claim[\'lhs\'])==r.parse(claim[\'rhs\']))\n    for claim in data[\'integer_ideals\']:\n        r=Ring(claim[\'source\']+claim[\'target\']);src=list(map(r.parse,claim[\'source\']));tar=list(map(r.parse,claim[\'target\']))\n        forward=mm(claim[\'forward\'],[[x] for x in src])\n        record(claim[\'id\']+\'.forward\',all(a[0]==b for a,b in zip(forward,tar)))\n        if claim[\'backward\'] is not None:\n            back=mm(claim[\'backward\'],[[x] for x in tar])\n            record(claim[\'id\']+\'.backward\',all(a[0]==b for a,b in zip(back,src)))\n    for cert in data[\'center_parametrizations\']:\n        # W=D^(-1)*C*Gamma is an integer unimodular matrix independent of m.\n        # This proves necessity, sufficiency and index for ALL positive m.\n        strings=sum(cert[\'basis\'],[])+cert[\'forms\']+[\'a\',\'b\',\'c\',\'d\',\'m\']\n        r=Ring(strings);G=[[r.parse(v) for v in row] for row in cert[\'basis\']]\n        C=[]\n        for form in cert[\'forms\']:\n            f=r.parse(form);row=[]\n            for var in (\'a\',\'b\',\'c\',\'d\'):\n                mon=r.parse(var);value=0\n                for degree,coef in f.num.items():\n                    if degree in mon.num:value=coef\n                row.append(value)\n            C.append(row)\n        W=mm(C,G)\n        W=[[ (value/r.parse(str(divisor))).integer() for value in row] for row,divisor in zip(W,cert[\'divisors\'])]\n        record(cert[\'family\']+\'.general_center_parametrization\',abs(det(W))==1)\n    ind=data[\'weight_induction\']\n    record(\'weights.induction_base\',ind[\'base\']=={\'degree\':0,\'weight_indices\':[0],\'dimension\':1})\n    record(\'weights.interval_cover\',ind[\'index_maps\']==[\'j\',\'j+1\'] and ind[\'intervals\']==[[0,\'n\'],[1,\'n+1\']] and ind[\'union\']==[0,\'n+1\'])\n    expected=[(\'2*r+1\',\'2*r+1\',1,1),(\'4*r+2\',\'2*r+1\',1,-1),(\'4*r\',\'2*r\',0,-1)]\n    record(\'roots.exhaustive_parity_partition\',[(v[\'order_form\'],v[\'m_form\'],v[\'m_parity\'],v[\'root_to_m\']) for v in data[\'root_parity\'][\'classes\']]==expected)\n    record(\'degree.general_rules_covered\',set(v[\'family\'] for v in data[\'degree_derivations\'])=={\'D\',\'G\',\'K\',\'L\'})\n\n\ndef integer_checks(cases,record):\n    for i,c in enumerate(cases):\n        A,N,U,V,D=c[\'A\'],c[\'N\'],c[\'U\'],c[\'V\'],c[\'D\'];size=len(A)\n        record(f\'integer.{i}.alternating\',all(A[j][k]==-A[k][j] for j in range(size) for k in range(size)))\n        record(f\'integer.{i}.smith_identity\',mm(mm(U,A),V)==D)\n        record(f\'integer.{i}.unimodular\',abs(det(U))==abs(det(V))==1)\n        diag=[abs(D[j][j]) for j in range(size)]\n        record(f\'integer.{i}.smith_diagonal\',all(D[j][k]==0 for j in range(size) for k in range(size) if j!=k) and all(not b if not a else b%a==0 for a,b in zip(diag,diag[1:])))\n        kernel=[[V[j][k]*(N//gcd(N,diag[k])) for k in range(size)] for j in range(size)]\n        Gamma=c[\'Gamma\'];W=c[\'kernel_change\']\n        record(f\'integer.{i}.complete_center\',mm(kernel,W)==Gamma and abs(det(W))==1)\n        record(f\'integer.{i}.central\',all(v%N==0 for row in mm(A,Gamma) for v in row))\n        image=prod(N//gcd(N,d) for d in diag);td=isqrt(image)\n        record(f\'integer.{i}.torus_degree\',td*td==image==abs(det(Gamma)) and td==c[\'torus_degree\'])\n        def multiply(u,v):\n            phase,a=u;more,b=v\n            # Independently count crossings in the ordered product.\n            for hi in range(size):\n                for lo in range(hi):phase-=a[hi]*b[lo]*A[lo][hi]\n            return (phase+more)%N,tuple(a[j]+b[j] for j in range(size))\n        def power(u,n):\n            # Inversion + binary powering, rather than the producer\'s n(n-1)/2 formula.\n            if n<0:\n                ph,a=u;opposite=tuple(-v for v in a)\n                crossed=multiply((ph,a),(0,opposite))[0]\n                u=((-crossed)%N,opposite);n=-n\n            result=(0,(0,)*size)\n            while n:\n                if n&1:result=multiply(result,u)\n                u=multiply(u,u);n//=2\n            return result\n        signatures=[]\n        for action in c[\'actions\']:\n            signature=[]\n            for column in transpose(Gamma):\n                value=(0,(0,)*size)\n                for j,n in enumerate(column):\n                    mon=(action[\'phases\'][j],tuple(row[j] for row in action[\'exponent_matrix\']))\n                    value=multiply(value,power(mon,n))\n                signature.append(value)\n            signatures.append(tuple(signature))\n        factor=len(set(signatures)) if signatures else 1\n        record(f\'integer.{i}.crossed_center\',factor==c[\'center_action_order\'])\n        record(f\'integer.{i}.algebra_degree\',td*factor==c[\'algebra_degree\']==c[\'formula\'])\n\n\ndef verify(bundle):\n    passed=[]\n    def record(name,ok):\n        if not ok:raise ArithmeticError(\'Independent certificate failed: \'+name)\n        passed.append(name)\n    source_checks(bundle[\'sources\'],record)\n    basis_checks(bundle[\'sources\'],bundle[\'basis_changes\'],record)\n    for model in bundle[\'uniform\'][\'word_models\']:\n        if not model.get(\'source_coordinates\'):continue\n        source=next(s for s in bundle[\'sources\'] if s[\'family\']==model[\'family\'][0])\n        r=Ring(sum(source[\'sigma\'],[])+[\'h\',\'f\',\'q\'])\n        targets=((0,0),(1,0),(0,1),(1,1))\n        for (a,b),(j,i,coefficient) in model[\'relations\']:\n            for key,value in model[\'substitutions\'].items():\n                coefficient=re.sub(r\'\\b\'+key+r\'\\b\',\'(\'+value+\')\',coefficient)\n            record(model[\'family\']+f\'.source_mixed.{a}.{b}\',all(r.parse(source[\'sigma\'][2*a+b][c])==\n                   (r.parse(coefficient) if v==(j,i) else 0) for c,v in enumerate(targets)))\n    arithmetic_checks(bundle[\'uniform\'],record)\n    word_checks(bundle[\'uniform\'][\'word_models\'],record)\n    integer_checks(bundle[\'integer_cases\'],record)\n    return {\'checker\':\'standard-library independent checker v1\',\'passed\':len(passed),\'failed\':0,\n            \'source_families\':len(bundle[\'sources\']),\'laurent_models\':len(bundle[\'uniform\'][\'word_models\']),\n            \'integer_certificates\':len(bundle[\'integer_cases\']),\n            \'unbounded_claims\':[\'weight induction algebra and interval cover\',\n                \'generic integer ideal identities\',\'complete Q/V/R center parametrizations\',\n                \'source-derived torus splits and actions\',\'all parity classes for M/N/O/P/W/Z\'],\n            \'assumptions\':bundle[\'uniform\'][\'theory_dependencies\'],\'checks\':passed}\n\n\nif __name__==\'__main__\':\n    bundle=json.loads(Path(sys.argv[1]).read_text());report=verify(bundle)\n    if len(sys.argv)>2:Path(sys.argv[2]).write_text(json.dumps(report,indent=2)+\'\\n\')\n    print(json.dumps({k:v for k,v in report.items() if k not in (\'checks\',\'assumptions\')},indent=2))\n')
import json, subprocess, sys
bundle=export_bundle(report)
Path('independent_certificates.json').write_text(json.dumps(bundle,indent=2))
result=subprocess.run([sys.executable,'-I','independent_check.py',
                'independent_certificates.json','independent_report.json'],
                check=True,capture_output=True,text=True)
print(result.stdout)
independent_report=json.loads(Path('independent_report.json').read_text())
assert independent_report['failed']==0
assert 'ore14641' not in sys.modules
print('Primary and independent verification completed without repository imports.')


{
  "checker": "standard-library independent checker v1",
  "passed": 1840,
  "failed": 0,
  "source_families": 26,
  "laurent_models": 21,
  "integer_certificates": 105,
  "unbounded_claims": [
    "weight induction algebra and interval cover",
    "generic integer ideal identities",
    "complete Q/V/R center parametrizations",
    "source-derived torus splits and actions",
    "all parity classes for M/N/O/P/W/Z"
  ]
}

Primary and independent verification completed without repository imports.


## 4. Inspect a monomial basis change

Rows of E and Z give the new x- and y-bases. The code uses
$(Z\otimes E)\Sigma(Z\otimes E)^{-1}$ and compares all four rows with the
claimed monomial relations. For square-root bases, $f=s^2$ and $s\ne0$.
Do not use these bases at the degenerate $f=0$ endpoints.


In [21]:
monomial_family = 'O'
datum = MONOMIAL[monomial_family]
print('Family',monomial_family,'| type',datum.kind)
display(Math('E = '+sp.latex(datum.x_basis)))
display(Math('Z = '+sp.latex(datum.y_basis)))
display(Math(r"\Sigma' = "+sp.latex(datum.transformed_sigma())))
print('Full lambdas:',datum.lambdas)
print('Invariants:',datum.invariants())
assert not any(datum.family.matrix(datum.transformed_sigma()-datum.target_sigma()))


Family O | type I


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Full lambdas: (h*(1 - s), h*(s + 1), h*(s + 1), h*(1 - s))
Invariants: (h**2*s**2 - 2*h**2*s + h**2, h**2*s**2 + 2*h**2*s + h**2)


## 5. A normal-element identity in F

With $\omega_1=x_1x_2$, $\omega_2=x_1^2+x_2^2$, the manuscript uses
$y_j\omega_k=(-2ph^2)\omega_k y_j$. The products below are computed in the
noncommutative algebra, rather than with commuting SymPy generator symbols.


In [22]:
A = Algebra(FAMILIES['F'])
x1,x2,y1,y2 = A.generators
omega = (x1*x2,x1**2+x2**2)
for j,y in enumerate((y1,y2),1):
    for k,w in enumerate(omega,1):
        residual = y*w-(-2*p*h**2)*w*y
        print(f'y{j} omega{k} - (-2 p h^2) omega{k} y{j} =',residual)
        assert not residual


y1 omega1 - (-2 p h^2) omega1 y1 = 0
y1 omega2 - (-2 p h^2) omega2 y1 = 0
y2 omega1 - (-2 p h^2) omega1 y2 = 0
y2 omega2 - (-2 p h^2) omega2 y2 = 0


## 6. Smith normal form and PI degrees

For an integer alternating exponent matrix A and a primitive Nth root,
$|\operatorname{im}(A\bmod N)|=\prod_i N/\gcd(N,d_i)$, where the $d_i$ are
the integer Smith entries. The torus PI degree is the square root of this index.

The first two examples use the manuscript's birational torus identifications
for D and G. The L algebra degree needs a **factor 2** from its crossed product;
the torus degree alone is not the answer for L.


In [23]:
for family_name,N,k,kp in [('G',5,1,2),('D',8,1,3)]:
    matrix = dg_matrix(family_name,N,k,kp)
    certificate = torus_certificate(matrix,N)
    display(Math('A_{'+family_name+'} = '+sp.latex(matrix)))
    print(family_name,certificate)
    assert certificate['pi_degree']==dg_closed_formula(N,k,kp)
N,theta,phi,eta = 4,0,0,1
certificate = torus_certificate(l_matrix(N,theta,phi,eta),N)
print('L torus PI degree:',certificate['pi_degree'])
print('L algebra PI degree:',2*certificate['pi_degree'])
assert 2*certificate['pi_degree']==l_closed_formula(N,theta,phi,eta)


<IPython.core.display.Math object>

G {'root_order': 5, 'commutation_matrix': [[0, 0, -3, -3], [0, 0, -1, 1], [3, 1, 0, -1], [3, -1, 1, 0]], 'smith_diagonal': [1, 1, 6, 6], 'image_size': 625, 'radical_size_mod_N': 1, 'pi_degree': 25}


<IPython.core.display.Math object>

D {'root_order': 8, 'commutation_matrix': [[0, 4, -8, -4], [-4, 0, -1, 1], [8, 1, 0, -1], [4, -1, 1, 0]], 'smith_diagonal': [1, 1, 8, 8], 'image_size': 64, 'radical_size_mod_N': 64, 'pi_degree': 8}
L torus PI degree: 2
L algebra PI degree: 4


## 7. Why the twist cannot be normalized to 1

The table evaluator below applies the manuscript's PI criterion to exact
parameters. Root-of-unity status is determined from cyclotomic minimal
polynomials; this is not a truncated search for an integer power close to 1.


In [24]:
examples = [('C',1/sp.sqrt(3)),('E',1/sp.sqrt(2)),('F',1/sp.sqrt(2)),
            ('I',1/sp.sqrt(2)),('J',1/sp.sqrt(2)),('S',sp.Rational(1,2)),
            ('T',sp.Rational(1,2)),('U',sp.Rational(1,2))]
rows = []
for family_name,scale in examples:
    rows.append(f"| {family_name} | {pi_decision(family_name,{h:1})['status']} | {scale} | {pi_decision(family_name,{h:scale})['status']} |")
display(Markdown('| Family | PI at h=1 | Another h | PI there |\n|---|---|---|---|\n'+'\n'.join(rows)))


| Family | PI at h=1 | Another h | PI there |
|---|---|---|---|
| C | False | sqrt(3)/3 | True |
| E | False | sqrt(2)/2 | True |
| F | False | sqrt(2)/2 | True |
| I | False | sqrt(2)/2 | True |
| J | False | sqrt(2)/2 | True |
| S | False | 1/2 | True |
| T | False | 1/2 | True |
| U | False | 1/2 | True |

## 8. Finite-order determinant does not suffice for PI

These exact specializations from the manuscript have $\det\sigma=I$ but
fail its PI criterion. They illustrate why a matrix order check is not itself
a replacement for the general PI argument.


In [25]:
for family_name,params in [('O',{f:9,h:1/(2*sp.sqrt(2))}),('N',{f:2,g:sp.sqrt(3)})]:
    specialized = FAMILIES[family_name].specialize(params)
    print(family_name,params,'det(sigma) =',specialized.detsigma())
    print(pi_decision(family_name,params))
    assert specialized.detsigma()==sp.eye(2)
    assert pi_decision(family_name,params)['status'] is False


O {f: 9, h: sqrt(2)/4} det(sigma) = Matrix([[1, 0], [0, 1]])
{'family': 'O', 'method': 'Instantiation of manuscript Table tab:main', 'status': False, 'required_scalars': [{'scalar': 'sqrt(2)', 'root_order': 0}, {'scalar': '-sqrt(2)/2', 'root_order': 0}]}


N {f: 2, g: sqrt(3)} det(sigma) = Matrix([[1, 0], [0, 1]])


{'family': 'N', 'method': 'Instantiation of manuscript Table tab:main', 'status': False, 'required_scalars': [{'scalar': 'sqrt(3) + 2', 'root_order': 0}, {'scalar': '2 - sqrt(3)', 'root_order': 0}]}


## 9. Source correction in Z: a negative control

The expanded relations on printed p. 400 replace $x_1y_1$ by $x_1y_2$
in two positions, while the Sigma matrix has $x_1y_1$. The source-based
presentation passes the four cubic checks. The two replacements make the
following overlap residuals nonzero. The wrong presentation is only a negative
control and is never substituted into the family registry.


In [26]:
for item in report['checks']:
    if item['id'].startswith('Z.printed_relations_rejected.'):
        print(item['id'].split('.')[-1])
        print(item['detail']['residual'])
        assert item['passed']


y1 x2 x1
{'x1 x1 y2': 'f*h**2 - h**2', 'x1 x2 y1': 'f*h**2', 'x1 x2 y2': '-f*h**2 - 2*h**2'}
y2 x2 x1
{'x1 x1 y2': '-2*f*h**2', 'x1 x2 y1': 'f**2*h**2', 'x1 x2 y2': '-f*h**2'}
y2 y1 x1
{'x1 y1 y2': 'f*h**2', 'x1 y2 y2': '-f*h**2', 'x2 y1 y1': 'f*h**2', 'x2 y1 y2': '-f*h**2'}
y2 y1 x2
{'x1 y1 y2': '-f*h**2', 'x1 y2 y2': 'f*h**2', 'x2 y1 y2': '-f*h**2', 'x2 y2 y2': 'f*h**2'}


## 10. All 21 closed degree models

Each row below is computed from the source-derived torus and its actions on
the center lattice. The comparison with the manuscript's formula is independent
of that computation. The sample order 12 is just a compact display; the API
also runs at arbitrarily specified common orders (including 3988 in the suite).

The exact-parameter interface accepts algebraic $p,q,f,g,h$, with no numerical
angles or tolerances. It records a default algebraic branch if p or q is omitted.


In [27]:
rows = []
for item in report['checks']:
    if item['category']=='all-family degree certificates' and item['id'].endswith('.12'):
        c = item['detail']
        rows.append(f"| {c['family']} | {c['pi_degree']} | {c['center_action_order']} | {c['algebra_pi_degree']} | {c['closed_formula']} |")
display(Markdown('| Family | Torus degree | Center action order | Algebra degree | Formula |\n|---|---:|---:|---:|---:|\n'+'\n'.join(rows)))
exact = degree_from_parameters('Z',{f:3,h:-sp.Rational(1,2)})
print('Z(3,-1/2): PI degree =',exact['algebra_pi_degree'])
display(Math('A = '+sp.latex(sp.Matrix(exact['commutation_matrix']))))
display(Math(r'\Gamma = '+sp.latex(sp.Matrix(exact['center_lattice']))))
print('Center-action image order:',exact['center_action_order'])


| Family | Torus degree | Center action order | Algebra degree | Formula |
|---|---:|---:|---:|---:|
| B | 6 | 4 | 24 | 24 |
| E | 12 | 4 | 48 | 48 |
| J | 12 | 4 | 48 | 48 |
| K | 6 | 4 | 24 | 24 |
| L | 6 | 4 | 24 | 24 |
| M | 12 | 4 | 48 | 48 |
| N | 12 | 4 | 48 | 48 |
| O | 12 | 4 | 48 | 48 |
| P | 12 | 4 | 48 | 48 |
| S | 6 | 4 | 24 | 24 |
| T | 6 | 4 | 24 | 24 |
| U | 6 | 4 | 24 | 24 |
| W | 12 | 4 | 48 | 48 |
| Z | 12 | 4 | 48 | 48 |
| F | 3 | 8 | 24 | 24 |
| I | 3 | 8 | 24 | 24 |
| Q | 12 | 2 | 24 | 24 |
| R | 6 | 4 | 24 | 24 |
| V | 12 | 2 | 24 | 24 |
| D | 36 | 1 | 36 | 36 |
| G | 36 | 1 | 36 | 36 |

Z(3,-1/2): PI degree = 6


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Center-action image order: 2


## 11. C: a rank-nine module and an open exact degree

The code gives an unbounded reduction of $y_1^iy_2^j$ in a nine-element left
$\Bbbk[a,b]$ basis, where $a=y_1^2y_2$, $b=y_1^3-y_2^3$. Its recurrence
identities and Hilbert-series equality are checked separately.
The subtorus degree and the exhaustive integer constraints yield the three
possibilities in the manuscript; the code does not silently choose one.


In [28]:
print('C module basis exponents:',C_BASIS)
print('Reduction of y1^8 y2^5:',c_module_reduce(8,5))
c_certificate = degree_from_parameters('C',{h:1/sp.sqrt(3)})
print('Possible PI degrees:',c_certificate['possible_pi_degrees'])
assert c_certificate['exact_degree_is_open']


C module basis exponents: ((0, 0), (1, 0), (0, 1), (2, 0), (1, 1), (0, 2), (3, 0), (1, 2), (4, 0))
Reduction of y1^8 y2^5: {(4, 0, 2): 1}


Possible PI degrees: [12, 36, 108]


## 12. Primitivity: algebraic witnesses and the remaining proof dependencies

Normality of the localization elements is checked in the original PBW algebra.
Central orbit polynomials are checked by exact Laurent reduction. In the next
example u has order 3 while the other parameter remains symbolic, so this is
not merely a PI example with both parameters specialized to roots of unity.

For the primitive branch, the code certifies the nonidentity exponent actions
and the odd/even obstruction to an inner monomial action. The simplicity,
local-closedness and Nullstellensatz arguments remain mathematical proofs.


In [29]:
witness = conditional_orbit_certificate('O','u',3)
print('Orbit coefficients central:',witness['central'])
print('Nonconstant central coefficient exists:',witness['has_nonconstant_coefficient'])
print(witness['remaining_parameter'])
for item in report['checks']:
    if item['id']=='O.outer_actions':
        print('Outer-action certificates:',item['detail'])


Orbit coefficients central: True
Nonconstant central coefficient exists: True
beta is symbolic and unconstrained
Outer-action certificates: [{'type': 'nonidentity exponent matrix', 'matrix': [[-1, 0, 0, 0], [0, -1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]}, {'type': 'nonidentity exponent matrix', 'matrix': [[-1, 0, 0, 0], [0, -1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]}, {'type': 'odd/even exponent obstruction', 'generator': 'r', 'outer_scalar': 'beta/alpha', 'sign': 1, 'odd_exponent': -1, 'inner_commutator_exponents': [0, -2, 0, 0], 'consequence_if_inner': 'u^(2j-odd)=+/-1, forcing u torsion'}]


## 13. Coverage ledger and the manuscript's L sign correction

Every required computational block must have a labeled check, or the verifier
fails. The ledger also records the theoretical dependencies and open questions.

**Correction:** in the L torus ordered as $(w,y_1,y_1^{-1}y_2,u)$, the first
upper exponent is $-(2\eta+\varphi)$, since
$w y_1=(h^2f)^{-1}y_1w$. The supplied manuscript prints the opposite sign.
The Pfaffian and alternating Smith invariants are unchanged, so the closed
degree formula is unaffected. The program retains an explicit negative control
for the old sign. See the accompanying manuscript-correction note.


In [30]:
print('Manuscript labels:',report['coverage']['manuscript_label_count'])
print('Required computation blocks:',report['coverage']['required_computational_blocks'])
print('Missing blocks:',report['coverage']['missing_required_blocks'])
assert not report['coverage']['missing_required_blocks']
inspect_label = 'lem:Fnorm'  # change to any computational label
for item in report['checks']:
    if inspect_label in item['manuscript_labels']:
        print(item['id'],item['passed'],item['detail'])


Manuscript labels: 108
Required computation blocks: 57
Missing blocks: []
F.normalizers.0 True {}
F.normalizers.1 True {}
F.normalizers.2 True {}
F.normalizers.3 True {}
F.normalizers.4 True {}


## 14. Export a verification record

The JSON file contains each check identifier, its status, and the zero residual
or exact certificate. The original source files are not needed to run this
notebook and are not redistributed with it.

For future extensions, add further identities or degree models to the readable
Python source, then regenerate this notebook. Mathematical/theoretical
dependencies and open questions are retained in the coverage record.


In [31]:
from pathlib import Path
write_report(report,Path('notebook_reports'))
Path('notebook_verification.json').write_text(Path('notebook_reports/verification.json').read_text())
print('Saved notebook_verification.json')
print('All',report['summary']['total'],'checks passed.')


Saved notebook_verification.json
All 2900 checks passed.
